In [ ]:
# ============================================================
# 00 — ONE-TIME UPLOAD: laptop ZIP -> Google Drive (permanent)
# Run this notebook ONCE. After it finishes, never run it again
# unless you intentionally want to re-upload.
# ============================================================

# ---------- 1. Mount Google Drive ----------
from google.colab import drive
drive.mount('/content/drive')
# A browser popup will ask you to allow access. Click Allow.

import os, zipfile, shutil

# ---------- 2. Permanent project folder on Drive ----------
PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
RAW_DIR      = os.path.join(PROJECT_ROOT, 'data', 'raw')

# Create the folder structure if it doesn't exist yet.
for sub in ['data/raw', 'data/interim', 'data/processed',
            'docs', 'notebooks', 'models', 'results']:
    os.makedirs(os.path.join(PROJECT_ROOT, sub), exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR     :", RAW_DIR)


# ---------- 3. Upload your ZIP from your laptop ----------
from google.colab import files
print("\nPlease select your ZIP file (e.g. seasons.zip).")
uploaded = files.upload()   # a file picker will appear
# After upload, `uploaded` is a dict: {filename: bytes}


# ---------- 4. Extract the ZIP into RAW_DIR ----------
for fname in uploaded.keys():
    zip_path = os.path.join('/content', fname)
    print(f"\nExtracting {fname} -> {RAW_DIR}")

    with zipfile.ZipFile(zip_path, 'r') as z:
        # Some ZIPs contain a top folder; flatten so CSVs land directly in RAW_DIR
        for member in z.namelist():
            # Skip directories inside the zip
            if member.endswith('/'):
                continue
            # Take only the basename (drop any inner folder path)
            base = os.path.basename(member)
            if not base:
                continue
            with z.open(member) as src, open(os.path.join(RAW_DIR, base), 'wb') as dst:
                shutil.copyfileobj(src, dst)

    # Optional: delete the uploaded zip from /content (it's gone anyway on disconnect)
    os.remove(zip_path)


# ---------- 5. Verify what landed on Drive ----------
print("\n=== Files now in RAW_DIR (permanent) ===")
files_on_drive = sorted(os.listdir(RAW_DIR))
for f in files_on_drive:
    size_kb = os.path.getsize(os.path.join(RAW_DIR, f)) / 1024
    print(f"  {f:30s}  {size_kb:8.1f} KB")

print(f"\nTotal files: {len(files_on_drive)}")
print("\n✅ Done. These files are now permanently on your Google Drive.")
print("   You will NEVER need to upload them again.")
print("   Next session: just run notebook 02_data_loading.ipynb.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/epl_match_outcome
RAW_DIR     : /content/drive/MyDrive/epl_match_outcome/data/raw

Please select your ZIP file (e.g. seasons.zip).


In [ ]:
# ============================================================
# 02 — DATA LOADING (Step 02 of the pipeline)
# Reads the 11 raw CSVs from permanent Google Drive storage,
# adds season columns, and logs shape/column audit.
# READ-ONLY on data/raw/ — nothing is modified.
# ============================================================

# ---------- 1. Mount Google Drive (needed every session) ----------
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

# ---------- 2. Permanent paths ----------
PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
RAW_DIR      = os.path.join(PROJECT_ROOT, 'data', 'raw')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(DOCS_DIR, exist_ok=True)

print("RAW_DIR contents:")
for f in sorted(os.listdir(RAW_DIR)):
    print("   ", f)


# ---------- 3. Season mapping (filename -> start year, label) ----------
SEASON_MAP = {
    'season-1516.csv': (2015, '2015/16'),
    'season-1617.csv': (2016, '2016/17'),
    'season-1718.csv': (2017, '2017/18'),
    'season-1819.csv': (2018, '2018/19'),
    'season-1920.csv': (2019, '2019/20'),
    'season-2021.csv': (2020, '2020/21'),
    'season-2122.csv': (2021, '2021/22'),
    'season-2223.csv': (2022, '2022/23'),
    'season-2324.csv': (2023, '2023/24'),
    'season-2425.csv': (2024, '2024/25'),
    'season-2526.csv': (2025, '2025/26'),
}


# ---------- 4. Load each file and tag with season columns ----------
frames = []
audit_rows = []

for fname, (start_year, label) in SEASON_MAP.items():
    fpath = os.path.join(RAW_DIR, fname)
    if not os.path.exists(fpath):
        print(f"⚠️  MISSING: {fname}")
        continue

    df = pd.read_csv(fpath)

    # Option C: integer season start_year + human-readable label
    df['season']       = start_year
    df['season_label'] = label

    frames.append(df)

    audit_rows.append({
        'file':         fname,
        'season_label': label,
        'rows':         df.shape[0],
        'columns':      df.shape[1] - 2,   # subtract the 2 season cols we added
    })

audit = pd.DataFrame(audit_rows).sort_values('season_label').reset_index(drop=True)
print("\n=== ROW / COLUMN AUDIT ===")
print(audit.to_string(index=False))


# ---------- 5. Combine (in memory only) ----------
all_matches = pd.concat(frames, ignore_index=True)
print("\nCombined shape:", all_matches.shape)
print("Columns:", list(all_matches.columns))


# ---------- 6. Schema consistency check ----------
base_cols = set(pd.read_csv(os.path.join(RAW_DIR, list(SEASON_MAP)[0]),
                            nrows=0).columns)
for fname in SEASON_MAP:
    fpath = os.path.join(RAW_DIR, fname)
    if not os.path.exists(fpath):
        continue
    cols = set(pd.read_csv(fpath, nrows=0).columns)
    if cols != base_cols:
        print(f"⚠️  SCHEMA DIFFERS in {fname}")
        print("   missing vs base:", base_cols - cols)
        print("   extra   vs base:", cols - base_cols)
print("Schema check done.")


# ---------- 7. Preview ----------
print("\n=== head ===")
print(all_matches.head(3).to_string())
print("\n=== tail ===")
print(all_matches.tail(3).to_string())

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(DOCS_DIR, exist_ok=True)

problem_statement = """# Problem Statement — EPL Match Outcome Prediction

**Project:** Predicting English Premier League match outcomes from pre-match information.
**Status:** LOCKED — every later decision is checked against this page.

---

## 1. Task
Multiclass classification with 3 classes.
- Target column: `FTR`
- Classes: `H` = Home win, `D` = Draw, `A` = Away win
- The classes are mutually exclusive. We predict a joint probability
  distribution over {H, D, A} that sums to 1, via a softmax / multinomial model.
  We do NOT use three independent binary classifiers (they can disagree and
  produce incoherent probabilities).

## 2. Prediction moment
Before kickoff. Every feature uses only matches played strictly earlier.
No current-match stats. Enforced by rolling-feature shifts + assert tests (Step 08I).

## 3. Output
Three probabilities [P(H), P(D), P(A)] per match, summing to 1.

## 4. Primary success metric
Log loss (a proper scoring rule).
Secondary: accuracy, per-class precision/recall/F1, macro F1, confusion matrix
(watch the Draw column), calibration curves. Optional: Brier, RPS.

## 5. Success bar
1. Beat DummyClassifier(most_frequent) on validation log loss.
2. Beat class-prior baseline on validation log loss.
3. If odds source available: approach fair (margin-removed) implied probs.
   (Raw CSVs contain no odds; source will be added when we reach the odds section.)

## 6. Scope
- League: English Premier League only
- Development seasons: 2015/16 – 2024/25
- Locked test set: 2025/26 (opened once, Step 17) 🔒
- Live set: 2026/27 (arrives at Step 22; not part of this project now)
- Data: 11 season CSVs, identical 22-column schema, 380 rows each

## 7. Splits (locked at Step 06)
| Split | Seasons |
| --- | --- |
| Train | 2015/16 – 2022/23 |
| Validation | 2023/24 – 2024/25 |
| Test | 2025/26 🔒 |

Rules: split by season/date (never random rows);
assert last-train-date < first-val-date < first-test-date;
test file saved separately and not opened until Step 17.

## 8. Known limits (accepted, honest)
1. No injuries, suspensions, or lineups
2. No cup or European fixtures
3. No in-play / live data
4. No weather, pitch, or stadium condition
5. No managerial-change flags
6. No transfer-window timing features
7. Referee not used as a predictive feature (available, excluded by design)
8. Odds not present in raw CSVs (source to be added later)
9. No external Elo / club-coefficient data (built from these matches only)
10. Promoted/relegated teams create structural missingness across seasons

## 9. What this page forbids
- Using 2025/26 data for any development decision
- Using any current-match statistic as a feature
- Reporting accuracy alone as evidence the model is good
- Silently changing scope, metric, or success bar (all changes go to decision_log.md)
"""

path = os.path.join(DOCS_DIR, '01_problem_statement.md')
with open(path, 'w') as f:
    f.write(problem_statement)

print("Written:", path)
print("Size   :", os.path.getsize(path), "bytes")

In [ ]:
decision_log_path = os.path.join(DOCS_DIR, 'decision_log.md')

initial_log = """# Decision Log — EPL Match Outcome Project

Every meaningful decision is recorded here with its reason, so we never
lose track of "why we did X instead of Y."

---

## Step 01 — Problem Definition
**Decision:** Multiclass classification with softmax over {H, D, A}.
**Reason:** Classes are mutually exclusive; we need a joint probability
distribution that sums to 1.
**Decision:** Primary metric is log loss (proper scoring rule).
**Reason:** We output probabilities, not labels; accuracy can be gamed by
the majority class and ignores confidence.

## Step 02 — Data Collection
**Decision:** Store season as both integer start-year (`season`) and
human-readable label (`season_label`).
**Reason:** Integer sorts and filters cleanly; string is readable.
**Decision:** Raw CSVs live permanently on Google Drive at
`MyDrive/epl_match_outcome/data/raw/`. Never edited.
**Reason:** Colab wipes `/content/` on disconnect; Drive persists.
**Finding:** All 11 files have 380 rows and 22 columns; schema is
identical; combined shape (4180, 24).
**Odds:** Raw CSVs contain no odds. Source to be provided when we reach
the odds-column section (Step 03 audit / Step 9.5C benchmark).
"""

with open(decision_log_path, 'w') as f:
    f.write(initial_log)

print("Written:", decision_log_path)

In [ ]:
# ============================================================
# STEP 03 — INITIAL DATA CHECK
# Read-only audit of the 11 raw season CSVs.
# No cleaning, no imputation, no dropping. Observation only.
# Outputs: docs/leakage_map.md + docs/data_quality_report.md
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, re, json
from collections import Counter
import numpy as np
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
RAW_DIR      = os.path.join(PROJECT_ROOT, 'data', 'raw')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(DOCS_DIR, exist_ok=True)

SEASON_MAP = {
    'season-1516.csv': (2015, '2015/16'),
    'season-1617.csv': (2016, '2016/17'),
    'season-1718.csv': (2017, '2017/18'),
    'season-1819.csv': (2018, '2018/19'),
    'season-1920.csv': (2019, '2019/20'),
    'season-2021.csv': (2020, '2020/21'),
    'season-2122.csv': (2021, '2021/22'),
    'season-2223.csv': (2022, '2022/23'),
    'season-2324.csv': (2023, '2023/24'),
    'season-2425.csv': (2024, '2024/25'),
    'season-2526.csv': (2025, '2025/26'),
}

print("RAW_DIR:", RAW_DIR)
print("Files found:", sorted(os.listdir(RAW_DIR)))

In [ ]:
# ---------- Load every file as-is (no edits) ----------
frames = []
for fname, (start_year, label) in SEASON_MAP.items():
    fpath = os.path.join(RAW_DIR, fname)
    df = pd.read_csv(fpath)
    df['season']       = start_year
    df['season_label'] = label
    frames.append(df)

all_matches = pd.concat(frames, ignore_index=True)

print("Combined shape:", all_matches.shape)
print("Number of files loaded:", len(frames))

In [ ]:
# ---------- Assertion 5 (fail-loud): no odds-like columns ----------
# Patterns used by football-data.co.uk for bookmaker columns:
#   B365H/D/A, BWH/D/A, IWH/D/A, PSH/D/A, WHH/D/A, VCH/D/A,
#   MaxH/D/A, AvgH/D/A, BFEH/D/A, and any column containing "odds".
ODDS_PATTERN = re.compile(
    r'^(B365|BW|IW|PS|WH|VC|Max|Avg|BFE|BF|LB|SJ|SB|GB|BS)[HDA]$'
    r'|odds', re.IGNORECASE
)

base_cols = [c for c in all_matches.columns if c not in ('season', 'season_label')]
odds_like = [c for c in base_cols if ODDS_PATTERN.match(c)]

assert len(odds_like) == 0, (
    f"STOP — odds-like columns detected: {odds_like}. "
    "This project excludes odds by design. Remove them before proceeding."
)

print("✅ No odds-like columns detected. Odds exclusion confirmed.")
print("   Base columns audited:", base_cols)

In [ ]:
# ---------- Assertion 2 (fail-loud): every file has 380 rows ----------
per_file_rows = (
    all_matches.groupby('season_label', sort=True)
               .size()
               .rename('rows')
               .to_frame()
)
per_file_rows['ok_380'] = per_file_rows['rows'].eq(380)

bad = per_file_rows[~per_file_rows['ok_380']]
assert bad.empty, f"STOP — these seasons are not 380 rows:\n{bad}"

print("✅ Every season has exactly 380 rows.")
print(per_file_rows.to_string())

In [ ]:
# ---------- Assertion 4 (fail-loud): schema identical ----------
base_cols_set = set(base_cols)
for fname in SEASON_MAP:
    fpath = os.path.join(RAW_DIR, fname)
    cols = set(pd.read_csv(fpath, nrows=0).columns)
    assert cols == base_cols_set, (
        f"STOP — schema differs in {fname}\n"
        f"  missing: {base_cols_set - cols}\n"
        f"  extra  : {cols - base_cols_set}"
    )
print("✅ Schema identical across all 11 files. Base columns:", base_cols)

In [ ]:
# ---------- Shape / dtypes / describe ----------
print("=== dtypes ===")
print(all_matches.dtypes)

print("\n=== numeric describe ===")
num_cols = all_matches.select_dtypes(include=[np.number]).columns.tolist()
print(all_matches[num_cols].describe().T.to_string())

In [ ]:
# ---------- Missingness ----------
miss_overall = all_matches[base_cols].isna().sum().rename('missing_overall').to_frame()
miss_overall['pct_overall'] = (miss_overall['missing_overall'] / len(all_matches) * 100).round(3)

miss_by_season = (
    all_matches.groupby('season_label')[base_cols].apply(lambda g: g.isna().sum())
)

print("=== Missing values — overall ===")
print(miss_overall.to_string())
print("\n=== Missing values — by season (nonzero columns only) ===")
nz = miss_by_season.loc[:, miss_by_season.sum(axis=0) > 0]
print("(empty table means no missing values anywhere)" if nz.empty else nz.to_string())

In [ ]:
# ---------- Missingness ----------
miss_overall = all_matches[base_cols].isna().sum().rename('missing_overall').to_frame()
miss_overall['pct_overall'] = (miss_overall['missing_overall'] / len(all_matches) * 100).round(3)

miss_by_season = (
    all_matches.groupby('season_label')[base_cols].apply(lambda g: g.isna().sum())
)

print("=== Missing values — overall ===")
print(miss_overall.to_string())
print("\n=== Missing values — by season (nonzero columns only) ===")
nz = miss_by_season.loc[:, miss_by_season.sum(axis=0) > 0]
print("(empty table means no missing values anywhere)" if nz.empty else nz.to_string())

In [ ]:
# ---------- Exact duplicates ----------
dupe_mask = all_matches.duplicated(subset=['Date', 'HomeTeam', 'AwayTeam'], keep=False)
dupe_rows = all_matches[dupe_mask].sort_values(['Date', 'HomeTeam', 'AwayTeam'])

print(f"Rows involved in duplicate (Date+Home+Away) groups: {dupe_mask.sum()}")
if not dupe_rows.empty:
    print("\nDuplicate rows preview:")
    print(dupe_rows[['Date','HomeTeam','AwayTeam','FTHG','FTAG','FTR','season_label']]
          .head(20).to_string(index=False))

In [ ]:
# ---------- Same-day reversed fixtures ----------
# If (A vs B) and (B vs A) both appear on the SAME date → impossible.
# (They can legitimately appear on different dates — that's just the reverse fixture.)
key_fwd = all_matches[['Date','HomeTeam','AwayTeam']].apply(tuple, axis=1)
key_rev = all_matches[['Date','AwayTeam','HomeTeam']].apply(tuple, axis=1)

same_day_reverse = key_fwd.isin(set(key_rev))
print(f"Same-day reversed fixtures found: {same_day_reverse.sum()}")
if same_day_reverse.any():
    print(all_matches[same_day_reverse][['Date','HomeTeam','AwayTeam','season_label']]
          .head(20).to_string(index=False))

In [ ]:
# ---------- Same-day reversed fixtures ----------
# If (A vs B) and (B vs A) both appear on the SAME date → impossible.
# (They can legitimately appear on different dates — that's just the reverse fixture.)
key_fwd = all_matches[['Date','HomeTeam','AwayTeam']].apply(tuple, axis=1)
key_rev = all_matches[['Date','AwayTeam','HomeTeam']].apply(tuple, axis=1)

same_day_reverse = key_fwd.isin(set(key_rev))
print(f"Same-day reversed fixtures found: {same_day_reverse.sum()}")
if same_day_reverse.any():
    print(all_matches[same_day_reverse][['Date','HomeTeam','AwayTeam','season_label']]
          .head(20).to_string(index=False))

In [ ]:
# ---------- Date parse (fail-loud) ----------
dates = pd.to_datetime(all_matches['Date'], errors='coerce', dayfirst=False)
assert dates.notna().all(), (
    f"STOP — {dates.isna().sum()} rows failed date parsing. "
    "Inspect the raw Date column before proceeding."
)
all_matches['Date_parsed'] = dates

# Range and monotonicity per season
print("=== Date range per season ===")
date_range = (
    all_matches.groupby('season_label')['Date_parsed']
               .agg(['min', 'max', 'count'])
)
print(date_range.to_string())

# Monotonic check: within each season, sorted rows should already be in date order
violations = []
for lbl, g in all_matches.groupby('season_label'):
    if not g['Date_parsed'].is_monotonic_increasing:
        violations.append(lbl)
assert not violations, f"STOP — dates not monotonic within: {violations}"
print("\n✅ Dates parsed cleanly; each season is in monotonic date order.")

In [ ]:
# ---------- Teams per season ----------
teams_by_season = {
    lbl: sorted(set(g['HomeTeam']).union(g['AwayTeam']))
    for lbl, g in all_matches.groupby('season_label')
}
for lbl in sorted(teams_by_season):
    print(f"{lbl}: {len(teams_by_season[lbl])} teams")

# Promoted / relegated: entered or left the league between consecutive seasons
seasons_sorted = sorted(teams_by_season)
promoted_relegated = []
for i in range(1, len(seasons_sorted)):
    prev, cur = seasons_sorted[i-1], seasons_sorted[i]
    prev_set, cur_set = set(teams_by_season[prev]), set(teams_by_season[cur])
    entered = sorted(cur_set - prev_set)
    left    = sorted(prev_set - cur_set)
    promoted_relegated.append({
        'season': cur,
        'promoted_in':   entered,
        'relegated_out': left,
    })

pr_df = pd.DataFrame(promoted_relegated)
print("\n=== Promoted / relegated between consecutive seasons ===")
print(pr_df.to_string(index=False))

In [ ]:
# ---------- Referee column audit ----------
print("Referee — missing:", all_matches['Referee'].isna().sum())
print("Referee — unique names:", all_matches['Referee'].nunique())
print("\nReferee counts (top 15):")
print(all_matches['Referee'].value_counts().head(15).to_string())

# Names present in some seasons but not others
refs_by_season = {
    lbl: set(g['Referee'].dropna())
    for lbl, g in all_matches.groupby('season_label')
}
all_refs = set().union(*refs_by_season.values())
print("\nTotal distinct referee names across all seasons:", len(all_refs))

In [ ]:
# ---------- FTR assertion + distribution ----------
valid_labels = {'H', 'D', 'A'}
present = set(all_matches['FTR'].dropna().unique())
assert present.issubset(valid_labels), (
    f"STOP — unexpected FTR values: {present - valid_labels}"
)

print("=== Overall FTR distribution ===")
overall = all_matches['FTR'].value_counts(normalize=True).round(4)
print(overall.to_string())

print("\n=== FTR distribution per season ===")
per_season = (
    all_matches.groupby('season_label')['FTR']
               .value_counts(normalize=True)
               .unstack()
               .round(4)
)
print(per_season.to_string())

In [ ]:
# ---------- Logic checks ----------
problems = {}

# 1. FTR agrees with FTHG vs FTAG
def ftr_from_goals(h, a):
    if h > a: return 'H'
    if h < a: return 'A'
    return 'D'
computed = all_matches.apply(lambda r: ftr_from_goals(r['FTHG'], r['FTAG']), axis=1)
problems['FTR_mismatch_vs_FTHG_FTAG'] = int((computed != all_matches['FTR']).sum())

# 2. Half-time goals ≤ full-time goals
problems['HT_goals_gt_FT_goals'] = int(
    ((all_matches['HTHG'] > all_matches['FTHG']) |
     (all_matches['HTAG'] > all_matches['FTAG'])).sum()
)

# 3. No negative stats
stat_cols = ['FTHG','FTAG','HTHG','HTAG','HS','AS','HST','AST','HF','AF','HC','AC','HY','AY','HR','AR']
neg = 0
for c in stat_cols:
    neg += int((all_matches[c] < 0).sum())
problems['negative_stat_values'] = neg

# 4. HTR agrees with HTHG vs HTAG
computed_htr = all_matches.apply(lambda r: ftr_from_goals(r['HTHG'], r['HTAG']), axis=1)
problems['HTR_mismatch_vs_HTHG_HTAG'] = int((computed_htr != all_matches['HTR']).sum())

print("=== Logic check results (0 = clean) ===")
for k, v in problems.items():
    print(f"  {k:35s}: {v}")

In [ ]:
# ---------- Leakage map ----------
# Tier definitions:
#   pre-match          → safe to use, known before kickoff
#   post-match, BANNED → contains information from the match itself
#   target             → the label
leakage_map = {
    'Date':      ('pre-match',  'Kickoff date. Known before the match.'),
    'HomeTeam':  ('pre-match',  'Home team name. Known before the match.'),
    'AwayTeam':  ('pre-match',  'Away team name. Known before the match.'),

    'FTHG':  ('post-match, BANNED', 'Full-time home goals — result of the match.'),
    'FTAG':  ('post-match, BANNED', 'Full-time away goals — result of the match.'),
    'FTR':   ('target',             'Full-time result. The label we predict.'),
    'HTHG':  ('post-match, BANNED', 'Half-time home goals.'),
    'HTAG':  ('post-match, BANNED', 'Half-time away goals.'),
    'HTR':   ('post-match, BANNED', 'Half-time result.'),

    'Referee': ('pre-match',  'Match official. Available pre-kickoff. EXCLUDED by design from modelling; audited for completeness only.'),

    'HS':  ('post-match, BANNED', 'Home shots.'),
    'AS':  ('post-match, BANNED', 'Away shots.'),
    'HST': ('post-match, BANNED', 'Home shots on target.'),
    'AST': ('post-match, BANNED', 'Away shots on target.'),
    'HF':  ('post-match, BANNED', 'Home fouls.'),
    'AF':  ('post-match, BANNED', 'Away fouls.'),
    'HC':  ('post-match, BANNED', 'Home corners.'),
    'AC':  ('post-match, BANNED', 'Away corners.'),
    'HY':  ('post-match, BANNED', 'Home yellow cards.'),
    'AY':  ('post-match, BANNED', 'Away yellow cards.'),
    'HR':  ('post-match, BANNED', 'Home red cards.'),
    'AR':  ('post-match, BANNED', 'Away red cards.'),
}

lm_df = pd.DataFrame(
    [(c, tier, note) for c, (tier, note) in leakage_map.items()],
    columns=['column', 'tier', 'note']
)
print(lm_df.to_string(index=False))

# Write markdown
leakage_md_lines = [
    "# Leakage Map — EPL Match Outcome Project",
    "",
    "Every raw column is tagged by its information tier. Modelling may only use",
    "columns in the `pre-match` tier. Post-match columns may be used **only** as",
    "the source of lagged rolling features, where the value is taken from a match",
    "**earlier in time** than the one being predicted (see Step 08 leakage guard).",
    "",
    "| Column | Tier | Note |",
    "| --- | --- | --- |",
]
for _, row in lm_df.iterrows():
    leakage_md_lines.append(f"| `{row['column']}` | {row['tier']} | {row['note']} |")
leakage_md_lines += [
    "",
    "## Rules derived from this map",
    "1. `FTR` is the target — never a feature.",
    "2. All `post-match, BANNED` columns are excluded as direct features.",
    "3. Derived features (rolling form, Elo, etc.) must be computed from",
    "   **strictly earlier** matches and shifted by one match (Step 08I).",
    "4. `Referee` is pre-match but excluded by design from modelling.",
    "5. No odds column exists in this dataset by design.",
]

lm_path = os.path.join(DOCS_DIR, 'leakage_map.md')
with open(lm_path, 'w') as f:
    f.write("\n".join(leakage_md_lines))
print("\nWrote:", lm_path)

In [ ]:
# ---------- Data quality report ----------
def fmt(x):
    return "0" if x == 0 else str(x)

report_lines = [
    "# Data Quality Report — Step 03",
    "",
    "Observation only. No cleaning performed at this step.",
    "",
    "## Scope",
    f"- Files audited: {len(SEASON_MAP)}",
    f"- Combined shape: {all_matches.shape[0]} rows × {len(base_cols)} base columns (+2 season columns)",
    "- Source: local CSVs in data/raw/ (permanent Google Drive)",
    "",
    "## Passed fail-loud assertions ✅",
    "- No odds-like columns detected (odds excluded by design).",
    "- Every season has exactly 380 rows.",
    "- Schema is identical across all 11 files.",
    "- All Date values parse as valid datetimes; each season is monotonic in date order.",
    "- All FTR values ⊆ {H, D, A}.",
    "",
    "## Missing values",
    f"- Overall missing cells in base columns: {int(miss_overall['missing_overall'].sum())}",
    "",
    "## Duplicates",
    f"- Rows involved in duplicate (Date+Home+Away) groups: {int(dupe_mask.sum())}",
    "",
    "## Logic checks (0 = clean)",
]
for k, v in problems.items():
    report_lines.append(f"- {k}: {v}")

report_lines += [
    "",
    "## Teams per season",
    f"- Minimum teams in a season: {min(len(v) for v in teams_by_season.values())}",
    f"- Maximum teams in a season: {max(len(v) for v in teams_by_season.values())}",
    "",
    "## Promoted / relegated (season-by-season)",
    "See notebook output for the full table. Summary:",
    f"- Transitions audited: {len(promoted_relegated)}",
    "",
    "## Referee",
    f"- Missing: {int(all_matches['Referee'].isna().sum())}",
    f"- Distinct names across all seasons: {len(all_refs)}",
    "",
    "## Recommended actions (do NOT execute here — these are Step 05/07 items)",
    "1. If duplicates or same-day reversed fixtures exist, remove them in Step 05",
    "   with a written change-log entry.",
    "2. If any team-name variants exist across seasons, normalise in Step 05.",
    "3. If any missing values exist, defer imputation to Step 07 (fit on train only).",
    "4. Promoted/relegated table feeds Step 07B (promoted-team handling).",
    "",
    "## Confirmed non-issues",
    "- Row count uniform (380 per season) — no missing fixtures.",
    "- Column set uniform — no schema drift across seasons.",
    "- No odds columns — the odds-exclusion decision is enforced by assertion.",
]

report_path = os.path.join(DOCS_DIR, 'data_quality_report.md')
with open(report_path, 'w') as f:
    f.write("\n".join(report_lines))
print("Wrote:", report_path)

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
log_path     = os.path.join(DOCS_DIR, 'decision_log.md')

append_block = """

## Step 03 — Initial Data Check
**Findings (all assertions passed):**
- 11 files, 380 rows each, identical 22-column schema.
- Zero missing values anywhere, all 11 seasons.
- Zero duplicates (Date+Home+Away); zero same-day reversed fixtures.
- All Dates parse cleanly; each season monotonic.
- FTR distribution overall: H 44.3% / A 32.0% / D 23.7%.
- Referee column: 51 distinct names, zero missing.
- Logic checks clean (FTR agrees with goals; HT ≤ FT; no negatives).

**Note (risk to flag, no action):**
- Draw rate varies by season (2018/19: 18.7%, 2015/16: 28.2%).
- Home-win rate varies by season (2020/21: 37.9%, 2016/17: 49.2%).
- This means validation (2023/24–24/25) may differ from test (2025/26)
  on base rates. Keep in mind for calibration and final report.

**Note (COVID seasons kept, flagged):**
- 2019/20 and 2020/21 have compressed calendars and (2020/21) elevated
  away-win rate due to empty stadiums. Kept in training as real football,
  but flagged in case future ablations want to exclude them.

**Artifacts written:**
- docs/leakage_map.md
- docs/data_quality_report.md
- docs/decision_log.md (this append)
"""

with open(log_path, 'a') as f:
    f.write(append_block)

print("Appended Step 03 findings to:", log_path)
print("Current size:", os.path.getsize(log_path), "bytes")

In [ ]:
# ============================================================
# STEP 04 — EXPLORATORY DATA ANALYSIS
# Scope: 2015/16 – 2024/25 (10 seasons, 3800 matches).
# 2025/26 is loaded but explicitly REMOVED from any EDA.
# Every chart ends with a "what decision does this change?" prompt.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
RAW_DIR      = os.path.join(PROJECT_ROOT, 'data', 'raw')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results', 'eda')
os.makedirs(RESULTS_DIR, exist_ok=True)

SEASON_MAP = {
    'season-1516.csv': (2015, '2015/16'),
    'season-1617.csv': (2016, '2016/17'),
    'season-1718.csv': (2017, '2017/18'),
    'season-1819.csv': (2018, '2018/19'),
    'season-1920.csv': (2019, '2019/20'),
    'season-2021.csv': (2020, '2020/21'),
    'season-2122.csv': (2021, '2021/22'),
    'season-2223.csv': (2022, '2022/23'),
    'season-2324.csv': (2023, '2023/24'),
    'season-2425.csv': (2024, '2024/25'),
    'season-2526.csv': (2025, '2025/26'),
}

DEV_SEASONS = ['2015/16','2016/17','2017/18','2018/19','2019/20',
               '2020/21','2021/22','2022/23','2023/24','2024/25']
TEST_SEASON = '2025/26'   # locked; never touched in this notebook

COVID_SEASONS = ['2019/20', '2020/21']

# ---------- Load ----------
frames = []
for fname, (sy, label) in SEASON_MAP.items():
    df = pd.read_csv(os.path.join(RAW_DIR, fname))
    df['season']       = sy
    df['season_label'] = label
    frames.append(df)

full = pd.concat(frames, ignore_index=True)
full['Date'] = pd.to_datetime(full['Date'])

# ---------- HARD SCOPE LOCK: drop 2025/26 ----------
dev = full[full['season_label'].isin(DEV_SEASONS)].copy()
assert TEST_SEASON not in dev['season_label'].unique(), \
    "STOP — 2025/26 leaked into EDA scope."

print("Full dataset shape      :", full.shape)
print("Development shape (EDA) :", dev.shape)
print("Seasons in EDA          :", sorted(dev['season_label'].unique()))
print("✅ 2025/26 excluded from all EDA below.")

In [ ]:
def season_palette(seasons):
    return ['#d62728' if s in COVID_SEASONS else '#1f77b4' for s in seasons]

def save_fig(name):
    path = os.path.join(RESULTS_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches='tight')
    plt.show()
    print("Saved:", path)

In [ ]:
# ---------- A. Target: H/D/A base rates + draw rate over time ----------
overall = dev['FTR'].value_counts(normalize=True).reindex(['H','D','A'])
print("=== Overall FTR distribution (2015/16–2024/25) ===")
print((overall * 100).round(2).to_string())

per_season = (dev.groupby('season_label')['FTR']
                .value_counts(normalize=True)
                .unstack()
                .reindex(columns=['H','D','A'])
                .loc[DEV_SEASONS])

fig, ax = plt.subplots()
for col, color in zip(['H','D','A'], ['#2ca02c','#7f7f7f','#d62728']):
    ax.plot(per_season.index, per_season[col]*100, marker='o', label=col, color=color)
for s in COVID_SEASONS:
    ax.axvspan(DEV_SEASONS.index(s)-0.5, DEV_SEASONS.index(s)+0.5,
               color='red', alpha=0.08)
ax.set_title("A. Outcome rates per season (COVID seasons shaded)")
ax.set_ylabel("% of matches")
ax.set_ylim(0, 55)
ax.legend(title='Outcome')
plt.xticks(rotation=45)
save_fig("A_outcome_rates_per_season.png")

print("\n=== Per-season FTR rates (%) ===")
print((per_season*100).round(2).to_string())

print("""
WHAT THIS SHOWS: base rates of H/D/A per season; draw rate drift.
WHY WE MADE IT: draw class is minority; drift affects calibration.
WHAT TO LOOK FOR: whether draw rate or home rate swings meaningfully.
WHAT WE FOUND: (fill after viewing)
DECISION CHANGED: class_weight strategy; note drift in decision log.
""")

In [ ]:
# ---------- B. Home advantage ----------
per_season_goals = (dev.groupby('season_label')
                      .agg(home_goals=('FTHG','mean'),
                           away_goals=('FTAG','mean'))
                      .loc[DEV_SEASONS])
per_season_goals['gap'] = per_season_goals['home_goals'] - per_season_goals['away_goals']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for s in COVID_SEASONS:
    idx = DEV_SEASONS.index(s)
    axes[0].axvspan(idx-0.5, idx+0.5, color='red', alpha=0.08)
    axes[1].axvspan(idx-0.5, idx+0.5, color='red', alpha=0.08)

axes[0].plot(per_season_goals.index, per_season_goals['home_goals'],
             marker='o', label='Home goals / match', color='#2ca02c')
axes[0].plot(per_season_goals.index, per_season_goals['away_goals'],
             marker='o', label='Away goals / match', color='#d62728')
axes[0].set_title("B1. Avg goals per match (COVID shaded)")
axes[0].set_ylabel("Goals per match")
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(per_season_goals.index, per_season_goals['gap'],
            color=season_palette(per_season_goals.index))
axes[1].set_title("B2. Home minus away goals")
axes[1].set_ylabel("Goal gap")
axes[1].tick_params(axis='x', rotation=45)

save_fig("B_home_advantage_over_time.png")

print("=== Per-season goal averages ===")
print(per_season_goals.round(3).to_string())

print("""
WHAT THIS SHOWS: home-scoring vs away-scoring averages, gap over time.
WHY WE MADE IT: quantify how big home advantage is and whether it's stable.
WHAT TO LOOK FOR: 2020/21 (COVID empty stadiums) gap collapsing toward 0.
WHAT WE FOUND: (fill)
DECISION CHANGED: whether to add explicit home/away form features.
""")

In [ ]:
# ---------- C. Goals distribution ----------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

goals_per_match = dev['FTHG'] + dev['FTAG']
axes[0].hist(goals_per_match, bins=range(0, 13), edgecolor='white', color='#1f77b4')
axes[0].set_title("C1. Goals per match (2015/16–2024/25)")
axes[0].set_xlabel("Total goals"); axes[0].set_ylabel("Count")

gm_season = (dev.assign(total=dev['FTHG']+dev['FTAG'])
               .groupby('season_label')['total'].mean()
               .loc[DEV_SEASONS])
axes[1].plot(gm_season.index, gm_season.values, marker='o',
             color='#1f77b4')
for s in COVID_SEASONS:
    idx = DEV_SEASONS.index(s)
    axes[1].axvspan(idx-0.5, idx+0.5, color='red', alpha=0.08)
axes[1].set_title("C2. Avg goals per match by season (COVID shaded)")
axes[1].set_ylabel("Goals per match")
axes[1].tick_params(axis='x', rotation=45)

save_fig("C_goals_distribution.png")

print("Avg goals per match overall:", round(goals_per_match.mean(), 3))
print("Max in a single match      :", int(goals_per_match.max()))

print("""
WHAT THIS SHOWS: distribution shape and season-to-season scoring trend.
WHY WE MADE IT: confirms football-typical scoring; supports later Poisson thinking.
WHAT TO LOOK FOR: no bizarre spikes; a mild trend if any.
WHAT WE FOUND: (fill)
DECISION CHANGED: whether goal-based features (rolling goals) are meaningful.
""")

In [ ]:
# ---------- D. Points per team-season (top vs rest) ----------
# Compute end-of-season points per team.
def season_table(group):
    rows = []
    for team in pd.unique(pd.concat([group['HomeTeam'], group['AwayTeam']])):
        pts = 0
        for _, m in group.iterrows():
            if m['HomeTeam'] == team:
                if m['FTHG'] > m['FTAG']: pts += 3
                elif m['FTHG'] == m['FTAG']: pts += 1
            elif m['AwayTeam'] == team:
                if m['FTAG'] > m['FTHG']: pts += 3
                elif m['FTAG'] == m['FTHG']: pts += 1
        rows.append({'team': team, 'points': pts})
    df = pd.DataFrame(rows).sort_values('points', ascending=False).reset_index(drop=True)
    df['rank'] = df.index + 1
    df['ppG']  = df['points'] / 38
    return df

season_points = {}
for lbl, g in dev.groupby('season_label'):
    season_points[lbl] = season_table(g)

# Determine "top" team-season = top-6 by points AND PPG > season median
top_labels = {}
for lbl, tbl in season_points.items():
    med = tbl['ppG'].median()
    top = tbl[(tbl['rank'] <= 6) & (tbl['ppG'] > med)]['team'].tolist()
    top_labels[lbl] = top

# Plot: distribution of points per season, top-6 highlighted
fig, ax = plt.subplots(figsize=(13, 5))
positions = np.arange(len(DEV_SEASONS))
for i, lbl in enumerate(DEV_SEASONS):
    tbl = season_points[lbl]
    colors = ['#ff7f0e' if t in top_labels[lbl] else '#1f77b4' for t in tbl['team']]
    ax.scatter([i]*len(tbl), tbl['points'], c=colors, alpha=0.7, s=55)
ax.set_xticks(positions); ax.set_xticklabels(DEV_SEASONS, rotation=45)
ax.set_ylabel("Final points")
ax.set_title("D. Final points per team-season (orange = our 'top' definition)")
save_fig("D_team_strength_points.png")

print("=== 'Top' team-seasons (top-6 AND PPG > median) ===")
for lbl in DEV_SEASONS:
    print(f"{lbl}: {top_labels[lbl]}")

print("""
WHAT THIS SHOWS: spread of team quality per season, and how we'll define 'top'.
WHY WE MADE IT:  we need a data-driven definition of strong teams for later
                analysis, without hand-picking the 'big six'.
WHAT TO LOOK FOR: whether the spread is stable across seasons, or widening.
WHAT WE FOUND: (fill)
DECISION CHANGED: definition of 'top' team for future charts;
                  informs whether team-strength features (Elo, PPG) will help.
""")

In [ ]:
# ---------- E. Position-based favourite win rate ----------
# Build a rank per team per season (already have) and join onto matches.
rank_lookup = {}
for lbl, tbl in season_points.items():
    rank_lookup[lbl] = dict(zip(tbl['team'], tbl['rank']))

dev2 = dev.copy()
dev2['home_rank_end'] = dev2.apply(lambda r: rank_lookup[r['season_label']][r['HomeTeam']], axis=1)
dev2['away_rank_end'] = dev2.apply(lambda r: rank_lookup[r['season_label']][r['AwayTeam']], axis=1)
dev2['home_fav']      = dev2['home_rank_end'] < dev2['away_rank_end']

fav_home = dev2[dev2['home_fav']]
fav_away = dev2[~dev2['home_fav']]

stats = pd.DataFrame({
    'higher-placed team is HOME': fav_home['FTR'].value_counts(normalize=True),
    'higher-placed team is AWAY': fav_away['FTR'].value_counts(normalize=True),
}).reindex(['H','D','A'])

# For "favourite wins" = higher-placed team wins:
fav_win_rate = (
    (fav_home['FTR'] == 'H').mean() * len(fav_home) / len(dev2)
    + (fav_away['FTR'] == 'A').mean() * len(fav_away) / len(dev2)
)
print(f"Higher-placed team wins: {fav_win_rate:.3f} of all matches")

fig, ax = plt.subplots()
stats.plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e'])
ax.set_title("E. Outcome distribution when the higher-placed team is home vs away")
ax.set_ylabel("Proportion of matches")
ax.tick_params(axis='x', rotation=0)
save_fig("E_position_vs_outcome.png")

print(stats.round(3).to_string())
print("""
WHAT THIS SHOWS: strength-vs-outcome relationship (position as strength proxy).
WHY WE MADE IT:  validate that strength should matter; forecasts Elo/PPG value.
WHAT TO LOOK FOR: higher-placed team wins clearly more often than base rate.
WHAT WE FOUND: (fill)
DECISION CHANGED: confidence that team-strength features are worth engineering.
""")

In [ ]:
# ---------- F. Rest days between matches per team ----------
dev2 = dev.sort_values('Date').copy()

def days_between_matches(g):
    g = g.sort_values('Date')
    return g['Date'].diff().dt.days.dropna()

rest = pd.concat([
    dev2.groupby(['season_label','HomeTeam']).apply(days_between_matches),
    dev2.groupby(['season_label','AwayTeam']).apply(days_between_matches),
])
rest = rest.dropna()

fig, ax = plt.subplots()
ax.hist(rest, bins=range(1, 25), edgecolor='white', color='#2ca02c')
ax.set_title("F. Rest days between consecutive matches per team")
ax.set_xlabel("Days since last match"); ax.set_ylabel("Count")
save_fig("F_rest_days.png")

print("Rest days describe:")
print(rest.describe().round(2).to_string())
print("\n% of matches with <= 3 rest days:", round((rest <= 3).mean() * 100, 2))
print("""
WHAT THIS SHOWS: how much rest varies; how often teams play on short rest.
WHY WE MADE IT:  decide if a rest-days feature is worth building (Step 08E).
WHAT TO LOOK FOR: meaningful share of short-rest (<4 days) matches.
WHAT WE FOUND: (fill)
DECISION CHANGED: include or exclude rest-days feature in Step 08.
""")

In [ ]:
# ---------- G. Goals per match by month ----------
dev2['month'] = dev2['Date'].dt.month
goals_by_month = (dev2.assign(total=dev2['FTHG']+dev2['FTAG'])
                       .groupby('month')['total'].agg(['mean','count']))

fig, ax = plt.subplots()
ax.bar(goals_by_month.index, goals_by_month['mean'],
       color='#1f77b4', edgecolor='white')
ax.set_title("G. Goals per match by calendar month (all seasons pooled)")
ax.set_xlabel("Month"); ax.set_ylabel("Avg goals per match")
ax.set_xticks(range(1,13))
save_fig("G_goals_by_month.png")

print(goals_by_month.round(2).to_string())
print("""
WHAT THIS SHOWS: whether scoring patterns vary by season stage.
WHY WE MADE IT:  decide if 'season stage' should be a feature (Step 08).
WHAT TO LOOK FOR: noticeable month-to-month swings (fatigue, weather).
WHAT WE FOUND: (fill)
DECISION CHANGED: include or exclude a season-stage feature.
""")

In [ ]:
# ---------- H. Promoted vs non-promoted teams ----------
# Using the promoted/relegated table from Step 03.
promoted = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}

rows = []
for lbl, tbl in season_points.items():
    for _, r in tbl.iterrows():
        rows.append({
            'season_label': lbl,
            'team': r['team'],
            'ppG': r['ppG'],
            'promoted': (lbl in promoted and r['team'] in promoted[lbl]),
        })
ppdf = pd.DataFrame(rows)

fig, ax = plt.subplots()
sns.boxplot(data=ppdf, x='promoted', y='ppG', ax=ax,
            palette=['#1f77b4', '#d62728'])
ax.set_xticklabels(['Not promoted', 'Promoted'])
ax.set_title("H. Season-to-date PPG — promoted vs non-promoted teams")
ax.set_ylabel("Points per game (season)")
save_fig("H_promoted_vs_rest.png")

print(ppdf.groupby('promoted')['ppG'].describe().round(3).to_string())
print("""
WHAT THIS SHOWS: whether promoted teams systematically underperform.
WHY WE MADE IT:  informs Step 07B (promoted-team handling with no history).
WHAT TO LOOK FOR: promoted teams visibly lower PPG distribution.
WHAT WE FOUND: (fill)
DECISION CHANGED: baseline adjustment for promoted teams in feature build.
""")

In [ ]:
# ---------- Findings table ----------
findings = pd.DataFrame([
    {'chart': 'A. Outcome rates',
     'decision': 'Class weight strategy; drift note'},
    {'chart': 'B. Home advantage',
     'decision': 'Whether home/away form features are needed'},
    {'chart': 'C. Goals distribution',
     'decision': 'Whether rolling-goal features make sense'},
    {'chart': 'D. Team strength spread',
     'decision': "Definition of 'top' used later"},
    {'chart': 'E. Position vs outcome',
     'decision': 'Confidence in team-strength features'},
    {'chart': 'F. Rest days',
     'decision': 'Include/exclude rest-days feature'},
    {'chart': 'G. Goals by month',
     'decision': 'Include/exclude season-stage feature'},
    {'chart': 'H. Promoted teams',
     'decision': 'Step 07B adjusted baseline'},
])

print(findings.to_string(index=False))
print("\nAll EDA charts saved to:", RESULTS_DIR)

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
log_path     = os.path.join(DOCS_DIR, 'decision_log.md')

append_block = """

## Step 04 — EDA Findings (2015/16–2024/25 only, 2025/26 untouched)

### A. Outcome rates
- Overall: H 44.50%, D 23.32%, A 32.18%.
- Per-season H rate swings 37.9% (2020/21) to 49.2% (2016/17).
- Per-season D rate swings 18.7% (2018/19) to 28.2% (2015/16).
- Decision: class weights needed. Calibration to be checked carefully.
- Note: 2020/21 COVID anomaly — home rate collapsed.

### B. Home advantage
- Typical home-away goal gap: ~0.30–0.40 goals/match.
- 2020/21 gap: 0.011 (COVID).
- 2024/25 gap: 0.092 (notably low — flag for validation scrutiny).
- Decision: build home/away form features (Step 08B).

### C. Goals distribution
- Avg goals/match: 2.83. Max: 9.
- Distribution is football-typical, right-skewed, long tail.
- Decision: rolling-goal features valid. Poisson baseline valid.
- Keep outliers (real football).

### D. Team strength / "top" definition
- Definition: top-6 by points AND PPG above season median.
- Captures big-six plus honest anomalies (Leicester 15/16, Newcastle 22/23, Brighton 22/23, Villa 23/24 and 24/25).
- Decision: use this definition for any "top team" analysis.
- Team strength is not fixed → use Elo / rolling PPG, not static labels.

### E. Position vs outcome
- Higher-placed team wins 56.2% overall.
- When home: 63.4% win rate.
- When away: 49.1% win rate.
- Decision: team-strength features are justified.
- PPG baseline (Step 9.5B) is meaningful and non-trivial.

### F. Rest days
- Mean rest: 15.4 days. Median: 14.
- Only 4.57% of matches have ≤3 days rest.
- Decision: include a rest-days feature (Step 08E). Modest signal.
- Plan: cap at ~14 days in feature engineering.

### G. Goals by month
- Stable 2.72–3.03 goals/match for normal months.
- May bumps to 3.03 (end-of-season effect), small.
- No meaningful season-stage signal.
- Decision: SKIP a "season stage" feature. Document as deliberate exclusion.

### H. Promoted vs non-promoted
- Promoted teams: mean 0.885 PPG. Non-promoted: 1.461 PPG.
- Even best promoted seasons ~ mid-table established teams.
- Decision: Step 07B promoted-team handling is critical.
- Plan: fill missing history with average of past promoted teams (train only) + is_promoted flag.

### Artifacts written
- results/eda/A..H (8 charts)
- docs/decision_log.md (this append)

### Key strategic takeaways for later steps
1. Class weights matter (draw class ~23%).
2. Home/away form features justified.
3. Team-strength features (Elo, rolling PPG) justified.
4. Rest-days feature = yes, modest.
5. Season-stage feature = no.
6. Promoted-team handling = critical (Step 07B).
7. Calibration will be checked because base rates drift across seasons.
"""

with open(log_path, 'a') as f:
    f.write(append_block)

print("Appended Step 04 findings to:", log_path)
print("Size:", os.path.getsize(log_path), "bytes")

In [ ]:
# ============================================================
# STEP 05 — INITIAL DATA CLEANING
# Produces data/interim/matches_clean.csv
# Light step: parse dates, tag seasons, sort, save. No filtering,
# no dropping, no imputation.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
RAW_DIR      = os.path.join(PROJECT_ROOT, 'data', 'raw')
INTERIM_DIR  = os.path.join(PROJECT_ROOT, 'data', 'interim')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

SEASON_MAP = {
    'season-1516.csv': (2015, '2015/16'),
    'season-1617.csv': (2016, '2016/17'),
    'season-1718.csv': (2017, '2017/18'),
    'season-1819.csv': (2018, '2018/19'),
    'season-1920.csv': (2019, '2019/20'),
    'season-2021.csv': (2020, '2020/21'),
    'season-2122.csv': (2021, '2021/22'),
    'season-2223.csv': (2022, '2022/23'),
    'season-2324.csv': (2023, '2023/24'),
    'season-2425.csv': (2024, '2024/25'),
    'season-2526.csv': (2025, '2025/26'),
}

frames = []
for fname, (sy, label) in SEASON_MAP.items():
    df = pd.read_csv(os.path.join(RAW_DIR, fname))
    df['season']       = sy
    df['season_label'] = label
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
print("Loaded shape:", raw.shape)
print("Columns:", list(raw.columns))

In [ ]:
# ---------- Parse Date ----------
raw['Date'] = pd.to_datetime(raw['Date'], errors='raise')

assert raw['Date'].notna().all(), "Date parse failed on at least one row."

# ---------- Sort chronologically ----------
# Stable sort by Date only (matches on the same date retain their original order).
raw = raw.sort_values('Date', kind='mergesort').reset_index(drop=True)

print("Sorted by Date. New index range:", raw.index.min(), "→", raw.index.max())
print("First date:", raw['Date'].min())
print("Last date :", raw['Date'].max())

In [ ]:
# ---------- Final assertions before writing ----------
# 1. Every season is present with 380 rows.
row_counts = raw.groupby('season_label').size()
assert set(row_counts.index) == set(SEASON_MAP.keys() or [v[1] for v in SEASON_MAP.values()]) \
       or row_counts.shape[0] == len(SEASON_MAP), "Season set mismatch."
assert (row_counts == 380).all(), f"Non-380 season found:\n{row_counts[row_counts != 380]}"

# 2. Schema intact — all 22 raw columns + 2 season columns.
expected_cols = ['Date','HomeTeam','AwayTeam',
                 'FTHG','FTAG','FTR','HTHG','HTAG','HTR','Referee',
                 'HS','AS','HST','AST','HF','AF','HC','AC','HY','AY','HR','AR',
                 'season','season_label']
assert list(raw.columns) == expected_cols, \
    f"Column mismatch.\nExpected: {expected_cols}\nGot: {list(raw.columns)}"

# 3. Zero missing values anywhere.
assert raw.isna().sum().sum() == 0, "Unexpected missing values in clean file."

# 4. Target column contains only {H, D, A}.
assert set(raw['FTR'].unique()).issubset({'H','D','A'}), "Unexpected FTR values."

print("✅ All pre-save assertions passed.")
print("   Rows:", len(raw))
print("   Columns:", len(raw.columns))

In [ ]:
# ---------- Save to data/interim ----------
clean_path = os.path.join(INTERIM_DIR, 'matches_clean.csv')
raw.to_csv(clean_path, index=False)

print("Saved:", clean_path)
print("Size (bytes):", os.path.getsize(clean_path))

# Confirm reload gives the same thing
reloaded = pd.read_csv(clean_path, parse_dates=['Date'])
assert reloaded.shape == raw.shape, "Reload shape mismatch."
assert (reloaded['FTR'] == raw['FTR']).all(), "FTR mismatch on reload."
print("✅ Reload check passed — file is deterministic.")

In [ ]:
# ---------- Change log ----------
changes = [
    ("Date",         "Converted from string to datetime64[ns].",
                     "Needed for time-based sorting and rolling features."),
    ("season",       "Added as int (start-year of the season).",
                     "Sorting and filtering downstream."),
    ("season_label", "Added as string (e.g. '2015/16').",
                     "Human-readable season column."),
]

not_changed = [
    "No rows removed.",
    "No columns removed (Referee retained per Step 03 decision).",
    "No missing values imputed (there were none).",
    "No duplicates removed (there were none).",
    "No outliers removed or capped.",
    "No features engineered (that is Step 08).",
    "No train/val/test split (that is Step 06).",
]

lines = [
    "# Cleaning Change Log — Step 05",
    "",
    "Raw files in `data/raw/` are NEVER edited. This log documents",
    "what was changed to produce `data/interim/matches_clean.csv`.",
    "",
    "## Inputs",
    f"- Source folder: `data/raw/`",
    f"- Files: {len(SEASON_MAP)} season CSVs",
    f"- Combined rows: {len(raw)}",
    f"- Combined columns (raw): 22",
    "",
    "## Changes applied",
    "",
    "| Item | Change | Reason |",
    "| --- | --- | --- |",
]
for item, change, reason in changes:
    lines.append(f"| `{item}` | {change} | {reason} |")

lines += [
    "",
    "## Deliberately NOT changed",
]
for item in not_changed:
    lines.append(f"- {item}")

lines += [
    "",
    "## Outputs",
    "- `data/interim/matches_clean.csv` (4180 rows × 24 columns)",
    "",
    "## Assertions passed before save",
    "- Every season has exactly 380 rows.",
    "- Column order and names match expectation.",
    "- Zero missing values across all columns.",
    "- FTR values are a subset of {H, D, A}.",
    "- File reload check confirms determinism.",
]

log_path = os.path.join(DOCS_DIR, 'cleaning_change_log.md')
with open(log_path, 'w') as f:
    f.write("\n".join(lines))

print("Wrote:", log_path)
print("Size (bytes):", os.path.getsize(log_path))

In [ ]:
# ---------- Quick snapshot of what we saved ----------
print("Shape          :", raw.shape)
print("First match    :", raw.iloc[0][['Date','HomeTeam','AwayTeam','FTR','season_label']].to_dict())
print("Last match     :", raw.iloc[-1][['Date','HomeTeam','AwayTeam','FTR','season_label']].to_dict())
print("\nSeason distribution:")
print(raw['season_label'].value_counts().sort_index().to_string())
print("\nDtypes:")
print(raw.dtypes.to_string())

In [ ]:
# ============================================================
# STEP 06 — DATASET SPLITTING 🔒
# Splits matches_clean.csv into train / val / test by season.
# Writes 3 files + lock marker + manifest.
# After this notebook, test.csv is NEVER loaded until Step 17.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import hashlib
import pandas as pd
from datetime import datetime

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
INTERIM_DIR  = os.path.join(PROJECT_ROOT, 'data', 'interim')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

# ---------- Load the frozen clean dataset ----------
clean_path = os.path.join(INTERIM_DIR, 'matches_clean.csv')
df = pd.read_csv(clean_path, parse_dates=['Date'])

print("Loaded clean dataset:", df.shape)
print("Date range:", df['Date'].min(), "→", df['Date'].max())
print("Seasons:", sorted(df['season_label'].unique()))

In [ ]:
# ---------- Define splits by season ----------
TRAIN_SEASONS = ['2015/16','2016/17','2017/18','2018/19',
                 '2019/20','2020/21','2021/22','2022/23']
VAL_SEASONS   = ['2023/24','2024/25']
TEST_SEASONS  = ['2025/26']

train = df[df['season_label'].isin(TRAIN_SEASONS)].sort_values('Date').reset_index(drop=True)
val   = df[df['season_label'].isin(VAL_SEASONS)].sort_values('Date').reset_index(drop=True)
test  = df[df['season_label'].isin(TEST_SEASONS)].sort_values('Date').reset_index(drop=True)

print(f"Train rows: {len(train)}  ({train['Date'].min().date()} → {train['Date'].max().date()})")
print(f"Val rows  : {len(val)}  ({val['Date'].min().date()} → {val['Date'].max().date()})")
print(f"Test rows : {len(test)}  ({test['Date'].min().date()} → {test['Date'].max().date()})")
print(f"Total     : {len(train) + len(val) + len(test)}")

In [ ]:
# ---------- Assertions ----------
assert len(train) == 3040, f"Train rows expected 3040, got {len(train)}"
assert len(val)   == 760,  f"Val rows expected 760, got {len(val)}"
assert len(test)  == 380,  f"Test rows expected 380, got {len(test)}"
assert len(train) + len(val) + len(test) == len(df), "Split does not partition the dataset."

# Chronological order: last train < first val < first test
assert train['Date'].max() < val['Date'].min(),  "Train/Val date overlap!"
assert val['Date'].max()   < test['Date'].min(), "Val/Test date overlap!"

# All test teams must have appeared in train ∪ val
dev_teams  = set(train['HomeTeam']) | set(train['AwayTeam']) | \
             set(val['HomeTeam'])   | set(val['AwayTeam'])
test_teams = set(test['HomeTeam'])  | set(test['AwayTeam'])
missing    = test_teams - dev_teams
assert not missing, f"Test contains unseen teams: {missing}"

# Schema consistency
assert list(train.columns) == list(val.columns) == list(test.columns), "Column mismatch."

print("✅ All assertions passed.")
print(f"   Train: {len(train)} rows")
print(f"   Val  : {len(val)} rows")
print(f"   Test : {len(test)} rows")
print(f"   Test teams all appeared in dev data: {len(test_teams)} teams")

In [ ]:
# ---------- Save splits ----------
train_path = os.path.join(PROC_DIR, 'train.csv')
val_path   = os.path.join(PROC_DIR, 'val.csv')
test_path  = os.path.join(PROC_DIR, 'test.csv')

train.to_csv(train_path, index=False)
val.to_csv(val_path,     index=False)
test.to_csv(test_path,   index=False)

print("Saved:")
for p in (train_path, val_path, test_path):
    print(f"  {p}  ({os.path.getsize(p)} bytes)")

In [ ]:
# ---------- TEST_IS_LOCKED marker ----------
lock_path = os.path.join(PROC_DIR, 'TEST_IS_LOCKED.txt')
lock_contents = """============================================================
TEST DATA IS LOCKED 🔒
============================================================

File: test.csv
Contains: English Premier League 2025/26 season (380 matches).

This file is NOT used for:
  - feature engineering
  - preprocessing fitting
  - model selection
  - hyperparameter tuning
  - threshold tuning
  - calibration
  - any "quick peek" to check how the model is doing

It is opened EXACTLY ONCE — at Step 17 (Final Model Testing) —
using a frozen model trained on train + val only.

If you find yourself tempted to read it "just to check", stop and
re-read docs/split_manifest.md. That temptation is the exact
mechanism that turns a good project into a meaningless one.

============================================================
"""
with open(lock_path, 'w') as f:
    f.write(lock_contents)
print("Wrote:", lock_path)

In [ ]:
# ---------- SHA-256 hashes for tamper detection ----------
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

hashes = {
    'train.csv': sha256(train_path),
    'val.csv'  : sha256(val_path),
    'test.csv' : sha256(test_path),
}

manifest_lines = [
    "# Split Manifest — EPL Match Outcome Project 🔒",
    "",
    "This file describes the train / validation / test split and records",
    "file hashes for tamper detection. Every modelling decision reads from",
    "train.csv and val.csv only. test.csv is opened once, at Step 17.",
    "",
    "## Split boundaries",
    "",
    "| Split | Seasons | Rows | First date | Last date |",
    "| --- | --- | --- | --- | --- |",
    f"| Train      | 2015/16 – 2022/23 | {len(train)} | {train['Date'].min().date()} | {train['Date'].max().date()} |",
    f"| Validation | 2023/24 – 2024/25 | {len(val)}   | {val['Date'].min().date()}   | {val['Date'].max().date()}   |",
    f"| Test 🔒    | 2025/26           | {len(test)}  | {test['Date'].min().date()}  | {test['Date'].max().date()}  |",
    "",
    "## Rules (non-negotiable)",
    "",
    "1. Test data is used exactly once, at Step 17, for final evaluation.",
    "2. No preprocessing is fitted on test data.",
    "3. No feature engineering decisions are based on test data.",
    "4. No hyperparameter tuning is done on test data.",
    "5. No 'quick peeks' at test performance.",
    "6. The final model is frozen before Step 17 — no changes after seeing test results.",
    "",
    "## File hashes (SHA-256)",
    "",
    "These hashes fingerprint the split files. If a hash no longer matches,",
    "the file has been modified. Rebuild all three files from",
    "`data/interim/matches_clean.csv` if this happens.",
    "",
    "```",
    f"{hashes['train.csv']}  train.csv",
    f"{hashes['val.csv']}    val.csv",
    f"{hashes['test.csv']}   test.csv",
    "```",
    "",
    f"*Manifest written: {datetime.utcnow().isoformat(timespec='seconds')}Z*",
]

manifest_path = os.path.join(DOCS_DIR, 'split_manifest.md')
with open(manifest_path, 'w') as f:
    f.write("\n".join(manifest_lines))

print("Wrote:", manifest_path)
print()
print("=== SHA-256 hashes ===")
for name, h in hashes.items():
    print(f"{name:10s}  {h}")

In [ ]:
# ---------- Live 2026/27 placeholder ----------
live_dir = os.path.join(PROC_DIR, 'live_202627')
os.makedirs(live_dir, exist_ok=True)

readme = """# Live 2026/27 — Reserved

This folder is reserved for the 2026/27 live prediction set.

It will be populated at Step 22 (Deployment / Prediction), when
2026/27 fixtures and results become available.

Until then, this folder stays empty.
"""
with open(os.path.join(live_dir, 'README.md'), 'w') as f:
    f.write(readme)

print("Created:", live_dir)

In [ ]:
# ---------- Final verification snapshot ----------
print("=== Files in data/processed/ ===")
for f in sorted(os.listdir(PROC_DIR)):
    p = os.path.join(PROC_DIR, f)
    if os.path.isfile(p):
        print(f"  {f:25s}  {os.path.getsize(p):>10d} bytes")
    else:
        print(f"  {f}/  (directory)")

print()
print("=== Split summary ===")
print(f"Train : {len(train):5d} rows  {train['season_label'].value_counts().sort_index().to_dict()}")
print(f"Val   : {len(val):5d} rows  {val['season_label'].value_counts().sort_index().to_dict()}")
print(f"Test  : {len(test):5d} rows  {test['season_label'].value_counts().sort_index().to_dict()}")

print()
print("🔒 TEST IS NOW LOCKED.")
print("   From this point on, test.csv is not loaded in any notebook")
print("   until Step 17 — Final Model Testing.")

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
log_path     = os.path.join(DOCS_DIR, 'decision_log.md')

append_block = """

## Step 06 — Dataset Splitting 🔒
**Split locked:**
- Train:      2015/16 – 2022/23 (3040 rows, 2015-08-08 → 2023-05-28)
- Validation: 2023/24 – 2024/25 (760 rows, 2023-08-11 → 2025-05-25)
- Test:       2025/26           (380 rows, 2025-08-15 → 2026-05-24) 🔒
- Live:       2026/27 reserved for Step 22.

**Artifacts written:**
- data/processed/train.csv
- data/processed/val.csv
- data/processed/test.csv (LOCKED)
- data/processed/TEST_IS_LOCKED.txt
- data/processed/live_202627/README.md
- docs/split_manifest.md (contains SHA-256 fingerprints)

**Assertions passed:**
- Row counts exact (3040 / 760 / 380 = 4180).
- Chronological order: max(train) < min(val) < min(test).
- All 20 test teams appeared in train ∪ val — no unseen teams.
- Schema identical across splits.

**SHA-256 hashes:**
- train.csv  b12ad8ddcc2a17da67b26a2cd6e76a319aa58dd94a7a211c4416d542fa1359ca
- val.csv    f13980a180db50177e6354a8d591e1746c74d5be38ae397e31de96a69f5f783d
- test.csv   8521098e5deb6d62b6f4d3165f6338c0828575d8da89ce024e48daf8c32a8d89

**Lock rules (non-negotiable, from this point until Step 17):**
- test.csv is not loaded in any notebook.
- No preprocessing is fitted on test.
- No feature engineering decisions based on test.
- No hyperparameter tuning on test.
- No "quick peeks" at test performance.
- Frozen final model before Step 17.
"""

with open(log_path, 'a') as f:
    f.write(append_block)

print("Appended Step 06 record to:", log_path)

In [ ]:
# ============================================================
# STEP 07A — PREPROCESSING INFRASTRUCTURE
# Writes src/features.py (shared module) and verifies it imports.
# No features built yet — that is Step 08.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

print("SRC_DIR :", SRC_DIR)
print("Docs    :", DOCS_DIR)

In [ ]:
features_py = '''"""
features.py — shared preprocessing and feature-engineering utilities.

Used by:
  - 07A_preprocessing.ipynb  (this notebook, written here)
  - 08_feature_engineering.ipynb  (imports and uses)
  - 07H final pipeline notebook
  - Step 17 final testing notebook
  - Step 22 live prediction notebook

Rules:
  - No global state. Every function takes its inputs explicitly.
  - Every rolling feature is shifted by one match (no current-match leakage).
  - Learned statistics are computed on train only, then applied.
  - Assertions fail loudly if leakage is detected.
"""

from __future__ import annotations
import numpy as np
import pandas as pd


# ============================================================
# 1. TARGET ENCODING
# ============================================================

TARGET_MAP = {"A": 0, "D": 1, "H": 2}
TARGET_INV = {v: k for k, v in TARGET_MAP.items()}


def encode_target(series: pd.Series) -> pd.Series:
    """Map FTR string labels to integers. A=0, D=1, H=2."""
    return series.map(TARGET_MAP).astype(int)


def decode_target(series: pd.Series) -> pd.Series:
    """Reverse mapping. Useful for reporting."""
    return series.map(TARGET_INV)


# ============================================================
# 2. LEAKAGE GUARDS
# ============================================================

def assert_no_current_match_leakage(features_df: pd.DataFrame,
                                    match_dates: pd.Series,
                                    match_home: pd.Series,
                                    match_away: pd.Series) -> None:
    """
    Placeholder that verifies shape alignment. Full check happens
    in Step 08I by verifying that rolling windows never include the
    current row. This function ensures the features frame has the same
    length as the input.
    """
    assert len(features_df) == len(match_dates), \\
        f"Features rows ({len(features_df)}) != matches rows ({len(match_dates)})"


def assert_rolling_shifted(features_df: pd.DataFrame,
                           raw_stats_df: pd.DataFrame,
                           stat_cols: list,
                           team_col: str,
                           date_col: str) -> None:
    """
    For each team, verify that the FIRST match in a season has NaN in
    rolling features (no history available). This is a light proxy
    check that rolling is behaving.

    A more thorough check is in Step 08I.
    """
    # Light check only: we verify the first row per team per season has
    # NaN for rolling columns that depend on history.
    pass


# ============================================================
# 3. PROMOTED-TEAM HANDLING
# ============================================================

def get_promoted_teams(promoted_table: dict, season_label: str) -> list:
    """Return list of teams promoted into the given season."""
    return list(promoted_table.get(season_label, []))


def compute_promoted_baseline(train_df: pd.DataFrame,
                              promoted_table: dict,
                              early_gameweeks: int = 5) -> dict:
    """
    Compute, from TRAIN ONLY, a baseline profile for promoted teams
    during their first `early_gameweeks` matches of a season.

    Returns a dict with mean values for the rolling features the
    model will use. These will be used to fill missing values for
    promoted teams' early-season rows.
    """
    # Identify matches where either team is a promoted team in their
    # first N matches of the season.
    train = train_df.copy().sort_values("Date").reset_index(drop=True)

    rows_for_baseline = []
    for season_label, group in train.groupby("season_label"):
        promoted = get_promoted_teams(promoted_table, season_label)
        if not promoted:
            continue
        for team in promoted:
            team_matches = train[
                (train["season_label"] == season_label) &
                ((train["HomeTeam"] == team) | (train["AwayTeam"] == team))
            ].sort_values("Date").head(early_gameweeks)
            rows_for_baseline.append(team_matches)

    if not rows_for_baseline:
        return {}

    baseline_df = pd.concat(rows_for_baseline, ignore_index=True)

    # Simple baseline: mean goals for/against when the promoted team plays,
    # expressed from the promoted team's perspective.
    def promoted_goals(m):
        return m["FTHG"] if m["HomeTeam"] in get_promoted_teams(promoted_table, m["season_label"]) else m["FTAG"]
    def promoted_conceded(m):
        return m["FTAG"] if m["HomeTeam"] in get_promoted_teams(promoted_table, m["season_label"]) else m["FTHG"]

    gf = baseline_df.apply(promoted_goals, axis=1).mean()
    ga = baseline_df.apply(promoted_conceded, axis=1).mean()

    return {
        "promoted_baseline_goals_for":     float(gf),
        "promoted_baseline_goals_against": float(ga),
    }


# ============================================================
# 4. EARLY-SEASON FORM HANDLING
# ============================================================

def compute_prev_season_form(train_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each team and each season, compute the team's final form in the
    PREVIOUS season (mean goals for/against, points per game).

    Used as a fallback for the first few gameweeks of a new season
    (carry-over approach for 07A).

    Returns a dataframe with columns:
      team, season_label, prev_season_ppg, prev_season_gf, prev_season_ga
    """
    # End-of-season points per team per season
    rows = []
    for (season_label, team), group in _iter_team_seasons(train_df):
        pts, gf, ga = 0, 0, 0
        for _, m in group.iterrows():
            if m["HomeTeam"] == team:
                gf += m["FTHG"]; ga += m["FTAG"]
                if m["FTHG"] > m["FTAG"]: pts += 3
                elif m["FTHG"] == m["FTAG"]: pts += 1
            else:
                gf += m["FTAG"]; ga += m["FTHG"]
                if m["FTAG"] > m["FTHG"]: pts += 3
                elif m["FTAG"] == m["FTHG"]: pts += 1
        matches = len(group)
        rows.append({
            "team": team,
            "season_label": season_label,
            "points": pts,
            "goals_for": gf,
            "goals_against": ga,
            "matches": matches,
            "ppg": pts / max(matches, 1),
            "gf_per_match": gf / max(matches, 1),
            "ga_per_match": ga / max(matches, 1),
        })
    end_of_season = pd.DataFrame(rows)

    # Shift to next season -> previous-season form for the new season
    season_order = sorted(end_of_season["season_label"].unique())
    next_season = {s: season_order[i+1] for i, s in enumerate(season_order) if i+1 < len(season_order)}
    end_of_season["next_season"] = end_of_season["season_label"].map(next_season)

    prev_lookup = end_of_season.dropna(subset=["next_season"]).rename(columns={
        "next_season": "season_label",
        "ppg": "prev_season_ppg",
        "gf_per_match": "prev_season_gf_per_match",
        "ga_per_match": "prev_season_ga_per_match",
    })[["team", "season_label", "prev_season_ppg",
        "prev_season_gf_per_match", "prev_season_ga_per_match"]]

    return prev_lookup


def _iter_team_seasons(df):
    """Helper: iterate over (season, team) groups."""
    for season_label, season_group in df.groupby("season_label"):
        teams = sorted(set(season_group["HomeTeam"]).union(season_group["AwayTeam"]))
        for team in teams:
            mask = (season_group["HomeTeam"] == team) | (season_group["AwayTeam"] == team)
            yield (season_label, team), season_group[mask]


# ============================================================
# 5. CLASS WEIGHTS
# ============================================================

def compute_class_weights(y_int: pd.Series) -> dict:
    """
    Compute class weights inversely proportional to class frequency.
    Returns dict {class_index: weight} suitable for class_weight= arg
    in sklearn LogisticRegression.
    """
    counts = y_int.value_counts().sort_index()
    total  = counts.sum()
    n_cls  = len(counts)
    return {int(cls): float(total / (n_cls * cnt)) for cls, cnt in counts.items()}


def compute_sample_weights(y_int: pd.Series) -> pd.Series:
    """
    Convert class weights into per-row sample weights for XGBoost.
    """
    cw = compute_class_weights(y_int)
    return y_int.map(cw).astype(float)


# ============================================================
# 6. TESTING UTILITY
# ============================================================

def test_feature_builder(df: pd.DataFrame) -> None:
    """
    Basic sanity test for the feature table.
    Runs after Step 08 builds features. Should pass on any split.
    """
    assert "Date" in df.columns, "Missing Date column."
    assert df["Date"].is_monotonic_increasing, "Rows are not sorted by Date."


print("features.py module loaded.")
'''

module_path = os.path.join(SRC_DIR, 'features.py')
with open(module_path, 'w') as f:
    f.write(features_py)

print("Wrote:", module_path)
print("Size (bytes):", os.path.getsize(module_path))

In [ ]:
import sys
sys.path.append(SRC_DIR)
import importlib
import features
importlib.reload(features)

print("Module import: OK")
print()
print("Public functions:")
for name in dir(features):
    if not name.startswith("_") and callable(getattr(features, name)):
        print("  -", name)

In [ ]:
import pandas as pd

s = pd.Series(['H', 'D', 'A', 'H', 'D'])
enc = features.encode_target(s)
dec = features.decode_target(enc)

print("Original :", s.tolist())
print("Encoded  :", enc.tolist())
print("Decoded  :", dec.tolist())
assert enc.tolist() == [2, 1, 0, 2, 1], "encode_target broken."
assert dec.tolist() == ['H', 'D', 'A', 'H', 'D'], "decode_target broken."
print("✅ target encoding round-trips correctly.")

In [ ]:
y = pd.Series([2, 2, 2, 1, 1, 0])  # H H H D D A
cw = features.compute_class_weights(y)
sw = features.compute_sample_weights(y)

print("Class counts :", y.value_counts().sort_index().to_dict())
print("Class weights:", cw)
print("Sample weights (first 10):", sw.head(10).tolist())

# Higher weight for smaller class
assert cw[0] > cw[2], "Class with fewer samples should have higher weight."
print("✅ class-weight logic correct.")

In [ ]:
features_py = '''"""
features.py — shared preprocessing and feature-engineering utilities.

Used by:
  - 07A_preprocessing.ipynb  (this notebook, written here)
  - 08_feature_engineering.ipynb  (imports and uses)
  - 07H final pipeline notebook
  - Step 17 final testing notebook
  - Step 22 live prediction notebook

Rules:
  - No global state. Every function takes its inputs explicitly.
  - Every rolling feature is shifted by one match (no current-match leakage).
  - Learned statistics are computed on train only, then applied.
  - Assertions fail loudly if leakage is detected.
"""

from __future__ import annotations
import numpy as np
import pandas as pd


# ============================================================
# 1. TARGET ENCODING
# ============================================================

TARGET_MAP = {"A": 0, "D": 1, "H": 2}
TARGET_INV = {v: k for k, v in TARGET_MAP.items()}


def encode_target(series: pd.Series) -> pd.Series:
    """Map FTR string labels to integers. A=0, D=1, H=2."""
    return series.map(TARGET_MAP).astype(int)


def decode_target(series: pd.Series) -> pd.Series:
    """Reverse mapping. Useful for reporting."""
    return series.map(TARGET_INV)


# ============================================================
# 2. LEAKAGE GUARDS
# ============================================================

def assert_no_current_match_leakage(features_df: pd.DataFrame,
                                    match_dates: pd.Series,
                                    match_home: pd.Series,
                                    match_away: pd.Series) -> None:
    """Placeholder — full check happens in Step 08I."""
    assert len(features_df) == len(match_dates), \\
        f"Features rows ({len(features_df)}) != matches rows ({len(match_dates)})"


def assert_rolling_shifted(features_df: pd.DataFrame,
                           raw_stats_df: pd.DataFrame,
                           stat_cols: list,
                           team_col: str,
                           date_col: str) -> None:
    """Placeholder — full check in Step 08I."""
    pass


# ============================================================
# 3. PROMOTED-TEAM HANDLING
# ============================================================

def get_promoted_teams(promoted_table: dict, season_label: str) -> list:
    """Return list of teams promoted into the given season."""
    return list(promoted_table.get(season_label, []))


def compute_promoted_baseline(train_df: pd.DataFrame,
                              promoted_table: dict,
                              early_gameweeks: int = 5) -> dict:
    """
    Compute, from TRAIN ONLY, a baseline profile for promoted teams
    during their first `early_gameweeks` matches of a season.
    """
    train = train_df.copy().sort_values("Date").reset_index(drop=True)

    rows_for_baseline = []
    for season_label, group in train.groupby("season_label"):
        promoted = get_promoted_teams(promoted_table, season_label)
        if not promoted:
            continue
        for team in promoted:
            team_matches = train[
                (train["season_label"] == season_label) &
                ((train["HomeTeam"] == team) | (train["AwayTeam"] == team))
            ].sort_values("Date").head(early_gameweeks)
            rows_for_baseline.append(team_matches)

    if not rows_for_baseline:
        return {}

    baseline_df = pd.concat(rows_for_baseline, ignore_index=True)

    def promoted_goals(m):
        return m["FTHG"] if m["HomeTeam"] in get_promoted_teams(promoted_table, m["season_label"]) else m["FTAG"]
    def promoted_conceded(m):
        return m["FTAG"] if m["HomeTeam"] in get_promoted_teams(promoted_table, m["season_label"]) else m["FTHG"]

    gf = baseline_df.apply(promoted_goals, axis=1).mean()
    ga = baseline_df.apply(promoted_conceded, axis=1).mean()

    return {
        "promoted_baseline_goals_for":     float(gf),
        "promoted_baseline_goals_against": float(ga),
    }


# ============================================================
# 4. EARLY-SEASON FORM HANDLING
# ============================================================

def compute_prev_season_form(train_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each team and each TARGET season, compute the team's end-of-season
    stats in the PREVIOUS season.

    Output columns:
      team, season_label (target), prev_season_label (source),
      prev_season_ppg, prev_season_gf_per_match, prev_season_ga_per_match
    """
    rows = []
    for (season_label, team), group in _iter_team_seasons(train_df):
        pts, gf, ga = 0, 0, 0
        for _, m in group.iterrows():
            if m["HomeTeam"] == team:
                gf += m["FTHG"]; ga += m["FTAG"]
                if m["FTHG"] > m["FTAG"]: pts += 3
                elif m["FTHG"] == m["FTAG"]: pts += 1
            else:
                gf += m["FTAG"]; ga += m["FTHG"]
                if m["FTAG"] > m["FTHG"]: pts += 3
                elif m["FTAG"] == m["FTHG"]: pts += 1
        matches = len(group)
        rows.append({
            "team": team,
            "prev_season_label": season_label,   # ← source season
            "points": pts,
            "goals_for": gf,
            "goals_against": ga,
            "matches": matches,
            "ppg": pts / max(matches, 1),
            "gf_per_match": gf / max(matches, 1),
            "ga_per_match": ga / max(matches, 1),
        })
    end_of_season = pd.DataFrame(rows)

    # Map source season -> next (target) season
    season_order = sorted(end_of_season["prev_season_label"].unique())
    next_season = {s: season_order[i+1] for i, s in enumerate(season_order) if i+1 < len(season_order)}
    end_of_season["season_label"] = end_of_season["prev_season_label"].map(next_season)

    prev_lookup = (
        end_of_season
        .dropna(subset=["season_label"])
        .rename(columns={
            "ppg": "prev_season_ppg",
            "gf_per_match": "prev_season_gf_per_match",
            "ga_per_match": "prev_season_ga_per_match",
        })
        [["team", "season_label", "prev_season_label",
          "prev_season_ppg", "prev_season_gf_per_match", "prev_season_ga_per_match"]]
        .reset_index(drop=True)
    )
    return prev_lookup


def _iter_team_seasons(df):
    """Helper: iterate over (season, team) groups."""
    for season_label, season_group in df.groupby("season_label"):
        teams = sorted(set(season_group["HomeTeam"]).union(season_group["AwayTeam"]))
        for team in teams:
            mask = (season_group["HomeTeam"] == team) | (season_group["AwayTeam"] == team)
            yield (season_label, team), season_group[mask]


# ============================================================
# 5. CLASS WEIGHTS
# ============================================================

def compute_class_weights(y_int: pd.Series) -> dict:
    counts = y_int.value_counts().sort_index()
    total  = counts.sum()
    n_cls  = len(counts)
    return {int(cls): float(total / (n_cls * cnt)) for cls, cnt in counts.items()}


def compute_sample_weights(y_int: pd.Series) -> pd.Series:
    cw = compute_class_weights(y_int)
    return y_int.map(cw).astype(float)


# ============================================================
# 6. TESTING UTILITY
# ============================================================

def test_feature_builder(df: pd.DataFrame) -> None:
    assert "Date" in df.columns, "Missing Date column."
    assert df["Date"].is_monotonic_increasing, "Rows are not sorted by Date."


print("features.py module loaded.")
'''

module_path = os.path.join(SRC_DIR, 'features.py')
with open(module_path, 'w') as f:
    f.write(features_py)

print("Rewrote:", module_path)
print("Size (bytes):", os.path.getsize(module_path))

In [ ]:
import sys
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
import importlib
import features
importlib.reload(features)

print("Module reloaded.")

In [ ]:
import pandas as pd

s = pd.Series(['H', 'D', 'A', 'H', 'D'])
enc = features.encode_target(s)
dec = features.decode_target(enc)

print("Original :", s.tolist())
print("Encoded  :", enc.tolist())
print("Decoded  :", dec.tolist())
assert enc.tolist() == [2, 1, 0, 2, 1]
assert dec.tolist() == ['H', 'D', 'A', 'H', 'D']
print("✅ target encoding round-trips correctly.")

In [ ]:
y = pd.Series([2, 2, 2, 1, 1, 0])
cw = features.compute_class_weights(y)
sw = features.compute_sample_weights(y)

print("Class counts :", y.value_counts().sort_index().to_dict())
print("Class weights:", cw)
print("Sample weights (first 6):", sw.head(6).tolist())

assert cw[0] > cw[2], "Smaller class must have higher weight."
print("✅ class-weight logic correct.")

In [ ]:
TRAIN_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'train.csv')
train = pd.read_csv(TRAIN_PATH, parse_dates=['Date'])

prev = features.compute_prev_season_form(train)

print("Previous-season form table shape:", prev.shape)
print("Columns:", list(prev.columns))
print()
print(prev.head(10).to_string(index=False))

# Sanity check: Leicester 2016/17 prev_season_ppg should be very high (~2.13, title season)
leic = prev[(prev['team'] == 'Leicester') & (prev['season_label'] == '2016/17')]
print("\nLeicester 2016/17 previous-season form:")
print(leic.to_string(index=False))

assert not leic.empty, "Leicester 2016/17 missing from previous-season table."
assert leic['prev_season_ppg'].iloc[0] > 2.0, "Leicester prev_season_ppg should be > 2.0 (they won 2015/16)."
print("\n✅ previous-season form looks correct.")

In [ ]:
promoted_table = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}

train_promoted = {k: v for k, v in promoted_table.items() if k in train['season_label'].unique()}
baseline = features.compute_promoted_baseline(train, train_promoted, early_gameweeks=5)

print("Promoted-team baseline (from train only):")
for k, v in baseline.items():
    print(f"  {k}: {v:.3f}")

In [ ]:
print("=" * 60)
print("STEP 07A COMPLETE")
print("=" * 60)
print()
print("src/features.py is on Drive and importable.")
print("Fixed bug: duplicate season_label column in compute_prev_season_form.")
print("Next: Step 08A — rolling form features")

In [ ]:
# ============================================================
# STEP 08A — ROLLING FORM FEATURES
# Adds build_rolling_form() to features.py
# Builds train_form.csv for verification.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys
import numpy as np
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import importlib
import features
importlib.reload(features)

print("Features module ready.")

In [ ]:
rolling_form_code = '''

# ============================================================
# 7. ROLLING FORM FEATURES (added at Step 08A)
# ============================================================

def build_rolling_form(matches_df: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    """
    Build rolling form features for each team in the given matches DataFrame.

    For each match (row), produce 12 columns:
        home_form_pts_last{n}      away_form_pts_last{n}
        home_form_gf_last{n}       away_form_gf_last{n}
        home_form_ga_last{n}       away_form_ga_last{n}
        home_form_gd_last{n}       away_form_gd_last{n}
        home_form_wins_last{n}     away_form_wins_last{n}
        home_form_cs_last{n}       away_form_cs_last{n}

    CRITICAL: All rolling windows use ONLY matches strictly earlier
    than the current match. The current match's goals never appear in
    its own features.

    The output DataFrame has the same index as the input.
    Rows are aligned by index; do NOT reorder the input.
    """
    df = matches_df.copy()
    df = df.reset_index(drop=True)
    df["_row_id"] = df.index

    # ---------- Build a long-form per-team timeline ----------
    # For each match, the home team earns stats as home, the away team earns
    # stats as away. We stack these into a single team-match long table.

    home_side = pd.DataFrame({
        "row_id":   df["_row_id"],
        "Date":     df["Date"],
        "season_label": df["season_label"],
        "team":     df["HomeTeam"],
        "goals_for":    df["FTHG"],
        "goals_against":df["FTAG"],
    })
    home_side["pts"] = np.where(home_side["goals_for"] > home_side["goals_against"], 3,
                        np.where(home_side["goals_for"] == home_side["goals_against"], 1, 0))
    home_side["is_home"] = 1

    away_side = pd.DataFrame({
        "row_id":   df["_row_id"],
        "Date":     df["Date"],
        "season_label": df["season_label"],
        "team":     df["AwayTeam"],
        "goals_for":    df["FTAG"],
        "goals_against":df["FTHG"],
    })
    away_side["pts"] = np.where(away_side["goals_for"] > away_side["goals_against"], 3,
                        np.where(away_side["goals_for"] == away_side["goals_against"], 1, 0))
    away_side["is_home"] = 0

    long_df = pd.concat([home_side, away_side], ignore_index=True)
    long_df = long_df.sort_values(["team", "Date", "row_id"]).reset_index(drop=True)

    # Derived per-match stats from the team's perspective
    long_df["gd"]           = long_df["goals_for"] - long_df["goals_against"]
    long_df["win"]          = (long_df["goals_for"] > long_df["goals_against"]).astype(int)
    long_df["clean_sheet"]  = (long_df["goals_against"] == 0).astype(int)

    # ---------- Group by team; rolling with shift(1) ----------
    g = long_df.groupby("team", group_keys=False)

    long_df["form_pts_last"]  = g["pts"].apply(
        lambda s: s.shift(1).rolling(window=n, min_periods=1).mean()
    )
    long_df["form_gf_last"]   = g["goals_for"].apply(
        lambda s: s.shift(1).rolling(window=n, min_periods=1).mean()
    )
    long_df["form_ga_last"]   = g["goals_against"].apply(
        lambda s: s.shift(1).rolling(window=n, min_periods=1).mean()
    )
    long_df["form_gd_last"]   = g["gd"].apply(
        lambda s: s.shift(1).rolling(window=n, min_periods=1).mean()
    )
    long_df["form_wins_last"] = g["win"].apply(
        lambda s: s.shift(1).rolling(window=n, min_periods=1).sum()
    )
    long_df["form_cs_last"]   = g["clean_sheet"].apply(
        lambda s: s.shift(1).rolling(window=n, min_periods=1).sum()
    )

    # ---------- Split back to home/away rows, pivot into match-level ----------
    home_feats = long_df[long_df["is_home"] == 1].set_index("row_id")
    away_feats = long_df[long_df["is_home"] == 0].set_index("row_id")

    out = pd.DataFrame(index=df.index)
    for c in ["form_pts_last","form_gf_last","form_ga_last",
              "form_gd_last","form_wins_last","form_cs_last"]:
        suffix = f"_last{n}" if not c.endswith("_last") else c.replace("_last", f"_last{n}")
        out[f"home_{suffix}"] = home_feats[c].reindex(df.index).values
        out[f"away_{suffix}"] = away_feats[c].reindex(df.index).values

    return out
'''

module_path = os.path.join(SRC_DIR, 'features.py')
with open(module_path, 'a') as f:
    f.write(rolling_form_code)

importlib.reload(features)
print("Appended build_rolling_form to features.py")
print("Public functions now:", [n for n in dir(features) if not n.startswith("_") and callable(getattr(features, n))])

In [ ]:
train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])
print("Train shape:", train.shape)

roll = features.build_rolling_form(train, n=5)
print("Rolling features shape:", roll.shape)
print("Rolling columns:", list(roll.columns))
print("\nFirst 5 rows:")
print(roll.head().to_string())

In [ ]:
# ---------- Spot-check leakage guard on one team ----------
TEAM = "Man City"
SEASON = "2016/17"

season_matches = train[(train['season_label'] == SEASON) &
                       ((train['HomeTeam'] == TEAM) | (train['AwayTeam'] == TEAM))
                      ].sort_values('Date').reset_index(drop=True)

# Pick a mid-season match — say index 10 (the 11th match of the season)
target_i = 10
target_match = season_matches.iloc[target_i]
print("Target match:")
print(target_match[['Date','HomeTeam','AwayTeam','FTHG','FTAG','FTR']].to_string())
print()

# Show last 5 matches before it
history = season_matches.iloc[max(0, target_i-5):target_i]
print("Last", len(history), "matches BEFORE the target:")
print(history[['Date','HomeTeam','AwayTeam','FTHG','FTAG']].to_string(index=False))
print()

# Manual computation of the rolling features for this team over that history
def team_pts(row, team):
    if row['HomeTeam'] == team:
        if row['FTHG'] > row['FTAG']: return 3
        elif row['FTHG'] == row['FTAG']: return 1
        else: return 0
    else:
        if row['FTAG'] > row['FTHG']: return 3
        elif row['FTAG'] == row['FTHG']: return 1
        else: return 0

def team_gf(row, team):
    return row['FTHG'] if row['HomeTeam'] == team else row['FTAG']

def team_ga(row, team):
    return row['FTAG'] if row['HomeTeam'] == team else row['FTHG']

manual_pts = np.mean([team_pts(r, TEAM) for _, r in history.iterrows()])
manual_gf  = np.mean([team_gf(r, TEAM)  for _, r in history.iterrows()])
manual_ga  = np.mean([team_ga(r, TEAM)  for _, r in history.iterrows()])

# Pull the feature row (same row of train)
match_idx_in_train = train[(train['Date'] == target_match['Date']) &
                           (train['HomeTeam'] == target_match['HomeTeam']) &
                           (train['AwayTeam'] == target_match['AwayTeam'])].index[0]

if target_match['HomeTeam'] == TEAM:
    pts_col, gf_col, ga_col = 'home_form_pts_last5', 'home_form_gf_last5', 'home_form_ga_last5'
else:
    pts_col, gf_col, ga_col = 'away_form_pts_last5', 'away_form_gf_last5', 'away_form_ga_last5'

print(f"Feature row (index {match_idx_in_train}):")
print(f"  {pts_col:25s} = {roll.loc[match_idx_in_train, pts_col]:.4f}   (manual: {manual_pts:.4f})")
print(f"  {gf_col:25s} = {roll.loc[match_idx_in_train, gf_col]:.4f}   (manual: {manual_gf:.4f})")
print(f"  {ga_col:25s} = {roll.loc[match_idx_in_train, ga_col]:.4f}   (manual: {manual_ga:.4f})")

assert abs(roll.loc[match_idx_in_train, pts_col] - manual_pts) < 1e-6
assert abs(roll.loc[match_idx_in_train, gf_col]  - manual_gf ) < 1e-6
assert abs(roll.loc[match_idx_in_train, ga_col]  - manual_ga ) < 1e-6
print("\n✅ Leakage guard verified — rolling features match manual computation.")

In [ ]:
# ---------- Verify first-match-of-season has NaN ----------
first_of_season_idx = train.groupby(['season_label','HomeTeam']).head(1).index
first_of_season_idx = train.groupby(['season_label','AwayTeam']).head(1).index

first_of_either = sorted(set(train.groupby(['season_label','HomeTeam']).head(1).index) |
                         set(train.groupby(['season_label','AwayTeam']).head(1).index))

n_nan_first = roll.loc[first_of_either, 'home_form_pts_last5'].isna().sum() + \
              roll.loc[first_of_either, 'away_form_pts_last5'].isna().sum()
print(f"Rows that are a team's first match of season: {len(first_of_either)}")
print(f"NaN count in those rows (home+away pts cols): {n_nan_first}")

n_nan_total_home = roll['home_form_pts_last5'].isna().sum()
n_nan_total_away = roll['away_form_pts_last5'].isna().sum()
print(f"Total NaN across entire train (home): {n_nan_total_home}")
print(f"Total NaN across entire train (away): {n_nan_total_away}")

In [ ]:
# ---------- Apply 07A fill rules ----------
# Priority: previous-season form -> promoted baseline (from train only).

PROMOTED_TABLE = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}

train_promoted = {k: v for k, v in PROMOTED_TABLE.items() if k in train['season_label'].unique()}
promoted_baseline = features.compute_promoted_baseline(train, train_promoted, early_gameweeks=5)
print("Promoted baseline (train only):", promoted_baseline)

# Previous-season form lookup (from train only)
prev_form = features.compute_prev_season_form(train)
prev_map = prev_form.set_index(['team','season_label']).to_dict('index')

# Add promoted flags
train_fe = train.copy()
train_fe['home_is_promoted'] = train_fe.apply(
    lambda r: int(r['HomeTeam'] in PROMOTED_TABLE.get(r['season_label'], [])), axis=1)
train_fe['away_is_promoted'] = train_fe.apply(
    lambda r: int(r['AwayTeam'] in PROMOTED_TABLE.get(r['season_label'], [])), axis=1)

# Join rolling features
train_fe = pd.concat([train_fe, roll], axis=1)

# Fill NaN per row using the priority chain
def fill_form_features(row):
    for side, team_col in [('home','HomeTeam'), ('away','AwayTeam')]:
        team = row[team_col]
        season = row['season_label']
        is_promoted = row[f'{side}_is_promoted'] == 1

        # Which columns to fill
        cols = [f'{side}_form_pts_last5', f'{side}_form_gf_last5', f'{side}_form_ga_last5',
                f'{side}_form_gd_last5', f'{side}_form_wins_last5', f'{side}_form_cs_last5']

        if row[cols[0]] is not None and not pd.isna(row[cols[0]]):
            continue  # already have in-season history

        prev = prev_map.get((team, season))

        if prev is not None:
            # Use previous-season form
            row[f'{side}_form_pts_last5']  = prev['prev_season_ppg']
            row[f'{side}_form_gf_last5']   = prev['prev_season_gf_per_match']
            row[f'{side}_form_ga_last5']   = prev['prev_season_ga_per_match']
            row[f'{side}_form_gd_last5']   = prev['prev_season_gf_per_match'] - prev['prev_season_ga_per_match']
            row[f'{side}_form_wins_last5'] = np.nan  # unknown from prev-season table
            row[f'{side}_form_cs_last5']   = np.nan
        else:
            # Use promoted baseline
            row[f'{side}_form_pts_last5']  = np.nan  # unclear pts mapping
            row[f'{side}_form_gf_last5']   = promoted_baseline.get('promoted_baseline_goals_for', 1.0)
            row[f'{side}_form_ga_last5']   = promoted_baseline.get('promoted_baseline_goals_against', 1.70)
            row[f'{side}_form_gd_last5']   = (promoted_baseline.get('promoted_baseline_goals_for', 1.0)
                                              - promoted_baseline.get('promoted_baseline_goals_against', 1.70))
            row[f'{side}_form_wins_last5'] = np.nan
            row[f'{side}_form_cs_last5']   = np.nan
    return row

# Apply row-wise
train_fe = train_fe.apply(fill_form_features, axis=1)

# Report remaining NaN
remaining_nan = train_fe[[c for c in train_fe.columns if c.startswith(('home_form','away_form'))]].isna().sum()
print("\nRemaining NaN in form columns after fill:")
print(remaining_nan[remaining_nan > 0].to_string() if (remaining_nan > 0).any() else "(none)")

In [ ]:
# ---------- Final shape + save ----------
print("train_fe shape:", train_fe.shape)
print("New columns:", [c for c in train_fe.columns if c.startswith(('home_form','away_form','home_is_','away_is_'))])

out_path = os.path.join(PROC_DIR, 'train_form.csv')
train_fe.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("Size (bytes):", os.path.getsize(out_path))

In [ ]:
# ============================================================
# REWRITE src/features.py — final version for Step 08
# ============================================================

features_py = '''"""
features.py — shared preprocessing and feature-engineering utilities.

Rules:
  - No global state. Every function takes its inputs explicitly.
  - Every rolling feature is shifted by one match (no current-match leakage).
  - Rolling windows reset at season boundaries (season-scoped).
  - Early-season missing history is filled via weighted blend with
    previous-season form, or promoted baseline for promoted teams.
  - Assertions fail loudly if leakage is detected.
"""

from __future__ import annotations
import numpy as np
import pandas as pd


# ============================================================
# 1. TARGET ENCODING
# ============================================================

TARGET_MAP = {"A": 0, "D": 1, "H": 2}
TARGET_INV = {v: k for k, v in TARGET_MAP.items()}


def encode_target(series: pd.Series) -> pd.Series:
    """Map FTR string labels to integers. A=0, D=1, H=2."""
    return series.map(TARGET_MAP).astype(int)


def decode_target(series: pd.Series) -> pd.Series:
    """Reverse mapping."""
    return series.map(TARGET_INV)


# ============================================================
# 2. LEAKAGE GUARDS
# ============================================================

def assert_no_current_match_leakage(features_df: pd.DataFrame,
                                    matches_df: pd.DataFrame) -> None:
    """Verify row counts align. Full check in Step 08I."""
    assert len(features_df) == len(matches_df), \\
        f"Features rows ({len(features_df)}) != matches rows ({len(matches_df)})"


# ============================================================
# 3. PROMOTED-TEAM HANDLING
# ============================================================

def get_promoted_teams(promoted_table: dict, season_label: str) -> list:
    return list(promoted_table.get(season_label, []))


def compute_promoted_baseline(train_df: pd.DataFrame,
                              promoted_table: dict,
                              early_gameweeks: int = 5) -> dict:
    """
    Compute, from TRAIN ONLY, the average per-match stats of promoted teams
    during their first `early_gameweeks` matches of the season.

    Returns dict with keys:
      promoted_baseline_ppg, promoted_baseline_gf, promoted_baseline_ga
    """
    train = train_df.copy().sort_values("Date").reset_index(drop=True)

    rows = []
    for season_label, group in train.groupby("season_label"):
        promoted = get_promoted_teams(promoted_table, season_label)
        if not promoted:
            continue
        for team in promoted:
            team_matches = train[
                (train["season_label"] == season_label) &
                ((train["HomeTeam"] == team) | (train["AwayTeam"] == team))
            ].sort_values("Date").head(early_gameweeks)
            rows.append(team_matches)

    if not rows:
        return {"promoted_baseline_ppg": 1.0,
                "promoted_baseline_gf":  1.0,
                "promoted_baseline_ga":  1.7}

    bdf = pd.concat(rows, ignore_index=True)

    def promoted_pts(m):
        is_home = m["HomeTeam"] in get_promoted_teams(promoted_table, m["season_label"])
        gf = m["FTHG"] if is_home else m["FTAG"]
        ga = m["FTAG"] if is_home else m["FTHG"]
        if gf > ga: return 3
        if gf == ga: return 1
        return 0

    def promoted_gf(m):
        is_home = m["HomeTeam"] in get_promoted_teams(promoted_table, m["season_label"])
        return m["FTHG"] if is_home else m["FTAG"]

    def promoted_ga(m):
        is_home = m["HomeTeam"] in get_promoted_teams(promoted_table, m["season_label"])
        return m["FTAG"] if is_home else m["FTHG"]

    pts = bdf.apply(promoted_pts, axis=1).mean()
    gf  = bdf.apply(promoted_gf,  axis=1).mean()
    ga  = bdf.apply(promoted_ga,  axis=1).mean()

    return {
        "promoted_baseline_ppg": float(pts),
        "promoted_baseline_gf":  float(gf),
        "promoted_baseline_ga":  float(ga),
    }


# ============================================================
# 4. PREVIOUS-SEASON FORM (per team, per season)
# ============================================================

def compute_prev_season_form(train_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each team and each TARGET season, compute the team's end-of-season
    stats in the PREVIOUS season.

    Columns: team, season_label (target), prev_season_label (source),
             prev_season_ppg, prev_season_gf, prev_season_ga
    """
    rows = []
    for (season_label, team), group in _iter_team_seasons(train_df):
        pts, gf, ga = 0, 0, 0
        for _, m in group.iterrows():
            if m["HomeTeam"] == team:
                gf += m["FTHG"]; ga += m["FTAG"]
                if m["FTHG"] > m["FTAG"]: pts += 3
                elif m["FTHG"] == m["FTAG"]: pts += 1
            else:
                gf += m["FTAG"]; ga += m["FTHG"]
                if m["FTAG"] > m["FTHG"]: pts += 3
                elif m["FTAG"] == m["FTHG"]: pts += 1
        n = len(group)
        rows.append({
            "team": team,
            "prev_season_label": season_label,
            "ppg": pts / max(n, 1),
            "gf_per_match": gf / max(n, 1),
            "ga_per_match": ga / max(n, 1),
        })
    end_of_season = pd.DataFrame(rows)

    season_order = sorted(end_of_season["prev_season_label"].unique())
    next_season = {s: season_order[i+1] for i, s in enumerate(season_order) if i+1 < len(season_order)}
    end_of_season["season_label"] = end_of_season["prev_season_label"].map(next_season)

    prev_lookup = (
        end_of_season
        .dropna(subset=["season_label"])
        .rename(columns={
            "ppg":          "prev_season_ppg",
            "gf_per_match": "prev_season_gf",
            "ga_per_match": "prev_season_ga",
        })
        [["team", "season_label", "prev_season_label",
          "prev_season_ppg", "prev_season_gf", "prev_season_ga"]]
        .reset_index(drop=True)
    )
    return prev_lookup


def _iter_team_seasons(df):
    for season_label, season_group in df.groupby("season_label"):
        teams = sorted(set(season_group["HomeTeam"]).union(season_group["AwayTeam"]))
        for team in teams:
            mask = (season_group["HomeTeam"] == team) | (season_group["AwayTeam"] == team)
            yield (season_label, team), season_group[mask]


# ============================================================
# 5. ROLLING FORM (season-scoped)
# ============================================================

def build_rolling_form(matches_df: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    """
    Build season-scoped rolling form features per team.

    For each match, produce 8 columns (4 per side):
        home_form_pts_last{n}, away_form_pts_last{n}
        home_form_gf_last{n},  away_form_gf_last{n}
        home_form_ga_last{n},  away_form_ga_last{n}
        home_form_gd_last{n},  away_form_gd_last{n}

    Rolling window resets at season boundaries: only matches within the
    SAME season that are strictly EARLIER than the current match count
    toward the rolling window. Early-season rows have NaN (filled later
    by weighted blend).

    Output DataFrame has the same index as input.
    """
    df = matches_df.copy().reset_index(drop=True)
    df["_row_id"] = df.index

    # ---------- Long-form per-team timeline ----------
    def make_side(side):
        if side == "home":
            d = pd.DataFrame({
                "row_id": df["_row_id"],
                "Date":   df["Date"],
                "season_label": df["season_label"],
                "team":   df["HomeTeam"],
                "goals_for":     df["FTHG"],
                "goals_against": df["FTAG"],
                "is_home": 1,
            })
        else:
            d = pd.DataFrame({
                "row_id": df["_row_id"],
                "Date":   df["Date"],
                "season_label": df["season_label"],
                "team":   df["AwayTeam"],
                "goals_for":     df["FTAG"],
                "goals_against": df["FTHG"],
                "is_home": 0,
            })
        d["gd"] = d["goals_for"] - d["goals_against"]
        d["pts"] = np.where(d["goals_for"] > d["goals_against"], 3,
                     np.where(d["goals_for"] == d["goals_against"], 1, 0))
        return d

    long_df = pd.concat([make_side("home"), make_side("away")], ignore_index=True)
    long_df = long_df.sort_values(["team", "season_label", "Date", "row_id"]).reset_index(drop=True)

    # ---------- Rolling within (team, season) groups ----------
    def roll_col(col):
        return (long_df.groupby(["team", "season_label"])[col]
                       .transform(lambda s: s.shift(1).rolling(n, min_periods=1).mean()))

    long_df["form_pts_last"] = roll_col("pts")
    long_df["form_gf_last"]  = roll_col("goals_for")
    long_df["form_ga_last"]  = roll_col("goals_against")
    long_df["form_gd_last"]  = roll_col("gd")

    # ---------- Back to match-level ----------
    home_feats = long_df[long_df["is_home"] == 1].set_index("row_id")
    away_feats = long_df[long_df["is_home"] == 0].set_index("row_id")

    out = pd.DataFrame(index=df.index)
    mapping = {
        "form_pts_last": f"form_pts_last{n}",
        "form_gf_last":  f"form_gf_last{n}",
        "form_ga_last":  f"form_ga_last{n}",
        "form_gd_last":  f"form_gd_last{n}",
    }
    for src, dst in mapping.items():
        out[f"home_{dst}"] = home_feats[src].reindex(df.index).values
        out[f"away_{dst}"] = away_feats[src].reindex(df.index).values
    return out


# ============================================================
# 6. WEIGHTED BLEND WITH PREVIOUS-SEASON FORM
# ============================================================

def build_rolling_form_filled(matches_df: pd.DataFrame,
                              prev_form_df: pd.DataFrame,
                              promoted_baseline: dict,
                              promoted_table: dict,
                              n: int = 5) -> pd.DataFrame:
    """
    Build rolling form (season-scoped) and fill early-season NaN via
    weighted blend:

        value = (k * rolling_in_season + (n-k) * prev_season_form) / n
        where k = number of in-season matches so far.

    For rows where rolling is already populated (k = n), no change.

    For rows where a team is promoted OR has no previous-season entry,
    fill with promoted_baseline values.

    Returns a DataFrame with 8 columns (4 home + 4 away) + 2 promoted flags.
    """
    n_actual = n
    roll = build_rolling_form(matches_df, n=n_actual)

    # Lookup for previous-season form
    prev_map = prev_form_df.set_index(["team", "season_label"]).to_dict("index")

    # Promoted flags
    matches = matches_df.copy().reset_index(drop=True)
    matches["home_is_promoted"] = matches.apply(
        lambda r: int(r["HomeTeam"] in get_promoted_teams(promoted_table, r["season_label"])), axis=1)
    matches["away_is_promoted"] = matches.apply(
        lambda r: int(r["AwayTeam"] in get_promoted_teams(promoted_table, r["season_label"])), axis=1)

    # Compute k (number of prior in-season matches) per team per season
    def compute_k(df):
        df = df.copy().reset_index(drop=True)
        # For each team-season, matches played before current match
        def team_k(df, team_col):
            df = df.sort_values(["season_label", "Date"]).reset_index(drop=True)
            df["_k"] = df.groupby([team_col, "season_label"]).cumcount()
            return df[["_k"]].rename(columns={"_k": f"_k_{team_col}"})
        # We'll compute both home and away k
        return df

    matches = matches.sort_values(["season_label", "Date"]).reset_index(drop=True)
    matches["_k_home"] = matches.groupby(["HomeTeam", "season_label"]).cumcount()
    matches["_k_away"] = matches.groupby(["AwayTeam", "season_label"]).cumcount()

    # Because build_rolling_form expects original order, reorder matching
    # For simplicity, rebuild roll from sorted matches and reindex to original input.
    original_index = matches_df.index
    # Build roll on the sorted matches; then reorder back
    roll_sorted = build_rolling_form(matches, n=n_actual)
    roll_final  = pd.DataFrame(index=matches_df.index)
    for c in roll_sorted.columns:
        roll_final[c] = roll_sorted[c].values  # same length and order as `matches` (which equals original after sort)

    # Apply blend per side
    def blend_side(side, k_series):
        team_col = "HomeTeam" if side == "home" else "AwayTeam"
        for metric_src, metric_prev in [
            (f"form_pts_last{n_actual}", "prev_season_ppg"),
            (f"form_gf_last{n_actual}",  "prev_season_gf"),
            (f"form_ga_last{n_actual}",  "prev_season_ga"),
        ]:
            col = f"{side}_{metric_src}"
            values = roll_final[col].values.copy()
            for i, row in matches.iterrows():
                k = int(row[f"_k_{team_col.replace('Team','').lower()}"])
                if k >= n_actual:
                    continue  # already fully populated by in-season history
                rolling_val = values[i]
                if pd.isna(rolling_val):
                    # No in-season history at all
                    in_season_avg = 0.0
                else:
                    in_season_avg = rolling_val  # already an average over k matches
                team = row[team_col]
                season = row["season_label"]
                prev = prev_map.get((team, season))
                if prev is not None:
                    prev_val = prev[metric_prev]
                    blended = (k * in_season_avg + (n_actual - k) * prev_val) / n_actual
                else:
                    # Promoted team or no history: use promoted baseline
                    base_key = {
                        "form_pts_last" + str(n_actual): "promoted_baseline_ppg",
                        "form_gf_last"  + str(n_actual): "promoted_baseline_gf",
                        "form_ga_last"  + str(n_actual): "promoted_baseline_ga",
                    }[metric_src]
                    prev_val = promoted_baseline[base_key]
                    blended = (k * in_season_avg + (n_actual - k) * prev_val) / n_actual
                values[i] = blended
            roll_final[col] = values

        # GD derived after pts/gf/ga
        col_gd = f"{side}_form_gd_last{n_actual}"
        col_gf = f"{side}_form_gf_last{n_actual}"
        col_ga = f"{side}_form_ga_last{n_actual}"
        roll_final[col_gd] = roll_final[col_gf] - roll_final[col_ga]

    # Compute k series aligned with matches
    k_home = matches["_k_home"].reset_index(drop=True)
    k_away = matches["_k_away"].reset_index(drop=True)
    blend_side("home", k_home)
    blend_side("away", k_away)

    # Add promoted flags
    roll_final["home_is_promoted"] = matches["home_is_promoted"].reset_index(drop=True).values
    roll_final["away_is_promoted"] = matches["away_is_promoted"].reset_index(drop=True).values

    # Reindex back to original input index
    roll_final.index = matches_df.index
    return roll_final


# ============================================================
# 7. CLASS WEIGHTS
# ============================================================

def compute_class_weights(y_int: pd.Series) -> dict:
    counts = y_int.value_counts().sort_index()
    total  = counts.sum()
    n_cls  = len(counts)
    return {int(cls): float(total / (n_cls * cnt)) for cls, cnt in counts.items()}


def compute_sample_weights(y_int: pd.Series) -> pd.Series:
    cw = compute_class_weights(y_int)
    return y_int.map(cw).astype(float)


# ============================================================
# 8. TESTING UTILITY
# ============================================================

def test_feature_builder(df: pd.DataFrame) -> None:
    assert "Date" in df.columns, "Missing Date column."
    assert df["Date"].is_monotonic_increasing, "Rows are not sorted by Date."


print("features.py module loaded.")
'''

module_path = '/content/drive/MyDrive/epl_match_outcome/src/features.py'
with open(module_path, 'w') as f:
    f.write(features_py)

print("Wrote:", module_path)
print("Size (bytes):", __import__('os').path.getsize(module_path))

In [ ]:
import sys, os, importlib
import numpy as np
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import features
importlib.reload(features)

train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])
print("Train:", train.shape)

# --- Data we need ---
PROMOTED_TABLE = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}
train_promoted = {k: v for k, v in PROMOTED_TABLE.items() if k in train['season_label'].unique()}

prev_form      = features.compute_prev_season_form(train)
promoted_base  = features.compute_promoted_baseline(train, train_promoted, early_gameweeks=5)

print("Promoted baseline:", promoted_base)

# --- Build filled rolling form ---
train_form = features.build_rolling_form_filled(
    train, prev_form, promoted_base, PROMOTED_TABLE, n=5
)
print("\nBuilt form shape:", train_form.shape)
print("Columns:", list(train_form.columns))

# NaN check
nan_counts = train_form.isna().sum()
print("\nNaN counts:")
print(nan_counts[nan_counts > 0].to_string() if (nan_counts > 0).any() else "(none) ✅")

# Attach to train and save
train_out = pd.concat([train.reset_index(drop=True), train_form.reset_index(drop=True)], axis=1)
out_path = os.path.join(PROC_DIR, 'train_form.csv')
train_out.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("Size (bytes):", os.path.getsize(out_path))

In [ ]:
# Same Man City 2016/17 spot-check, but now season-scoped
TEAM, SEASON = "Man City", "2016/17"

season_matches = train[(train['season_label'] == SEASON) &
                       ((train['HomeTeam'] == TEAM) | (train['AwayTeam'] == TEAM))
                      ].sort_values('Date').reset_index(drop=True)

target_i = 10
target_match = season_matches.iloc[target_i]
print("Target match:", target_match[['Date','HomeTeam','AwayTeam','FTHG','FTAG','FTR']].to_dict())
print()

# Show last 5 matches BEFORE it, IN-SEASON only
history = season_matches.iloc[max(0, target_i-5):target_i]
print("Last", len(history), "matches before the target (in-season):")
print(history[['Date','HomeTeam','AwayTeam','FTHG','FTAG']].to_string(index=False))

def team_pts(row, team):
    if row['HomeTeam'] == team:
        return 3 if row['FTHG'] > row['FTAG'] else (1 if row['FTHG'] == row['FTAG'] else 0)
    else:
        return 3 if row['FTAG'] > row['FTHG'] else (1 if row['FTAG'] == row['FTHG'] else 0)

manual_pts = np.mean([team_pts(r, TEAM) for _, r in history.iterrows()])

# Find the feature row in train_form
mask = (train['Date'] == target_match['Date']) & \
       (train['HomeTeam'] == target_match['HomeTeam']) & \
       (train['AwayTeam'] == target_match['AwayTeam'])
idx = train.index[mask][0]

side = 'home' if target_match['HomeTeam'] == TEAM else 'away'
feat_pts = train_out.loc[idx, f'{side}_form_pts_last5']

print()
print(f"Feature {side}_form_pts_last5 = {feat_pts:.4f}")
print(f"Manual 5-match avg pts      = {manual_pts:.4f}")
assert abs(feat_pts - manual_pts) < 1e-6
print("✅ Leakage guard verified (season-scoped, no fill applied at k=5).")

In [ ]:
# Pick Man City's first match of 2016/17 (k=0, should be 100% prev-season form)
first_match_mask = (train['season_label'] == '2016/17') & \
                   ((train['HomeTeam'] == 'Man City') | (train['AwayTeam'] == 'Man City'))
first_idx = train[first_match_mask].sort_values('Date').index[0]

row = train_out.loc[first_idx]
print("Man City's first match of 2016/17:")
print(row[['Date','HomeTeam','AwayTeam','FTHG','FTAG','season_label']].to_dict())
print()
print("Filled rolling form features:")
for c in [c for c in train_out.columns if 'form' in c and ('Man City' in row['HomeTeam'] and c.startswith('home_') or 'Man City' in row['AwayTeam'] and c.startswith('away_'))]:
    print(f"  {c:35s} = {row[c]:.4f}")

# Man City 2015/16 final form (they finished 4th, ppg ~1.737)
leic_prev = prev_form[(prev_form['team'] == 'Man City') & (prev_form['season_label'] == '2016/17')]
print("\nMan City's 2015/16 final form (used for fill):")
print(leic_prev.to_string(index=False))

In [ ]:
venue_form_code = '''

# ============================================================
# 9. VENUE-SPECIFIC PREVIOUS-SEASON FORM (added at Step 08B)
# ============================================================

def compute_prev_season_form_venue(train_df: pd.DataFrame) -> pd.DataFrame:
    """
    Like compute_prev_season_form, but returns venue-specific stats
    (home-only and away-only for each team in each previous season).

    Columns:
      team, season_label (target), prev_season_label (source),
      prev_season_ppg_home, prev_season_gf_home, prev_season_ga_home,
      prev_season_ppg_away, prev_season_gf_away, prev_season_ga_away
    """
    rows = []
    for (season_label, team), group in _iter_team_seasons(train_df):
        home_group = group[group["HomeTeam"] == team]
        away_group = group[group["AwayTeam"] == team]

        def stats(sub, side):
            if len(sub) == 0:
                return (np.nan, np.nan, np.nan)
            if side == "home":
                gf_series = sub["FTHG"]; ga_series = sub["FTAG"]
            else:
                gf_series = sub["FTAG"]; ga_series = sub["FTHG"]
            pts = 0
            for _, m in sub.iterrows():
                gf = m["FTHG"] if side == "home" else m["FTAG"]
                ga = m["FTAG"] if side == "home" else m["FTHG"]
                if gf > ga: pts += 3
                elif gf == ga: pts += 1
            n = len(sub)
            return (pts / n, gf_series.mean(), ga_series.mean())

        ph, gfh, gah = stats(home_group, "home")
        pa, gfa, gaa = stats(away_group, "away")

        rows.append({
            "team": team,
            "prev_season_label": season_label,
            "ppg_home": ph, "gf_home": gfh, "ga_home": gah,
            "ppg_away": pa, "gf_away": gfa, "ga_away": gaa,
        })

    end_of_season = pd.DataFrame(rows)

    season_order = sorted(end_of_season["prev_season_label"].unique())
    next_season = {s: season_order[i+1] for i, s in enumerate(season_order) if i+1 < len(season_order)}
    end_of_season["season_label"] = end_of_season["prev_season_label"].map(next_season)

    prev_lookup = (
        end_of_season
        .dropna(subset=["season_label"])
        .rename(columns={
            "ppg_home": "prev_season_ppg_home",
            "gf_home":  "prev_season_gf_home",
            "ga_home":  "prev_season_ga_home",
            "ppg_away": "prev_season_ppg_away",
            "gf_away":  "prev_season_gf_away",
            "ga_away":  "prev_season_ga_away",
        })
        [["team", "season_label", "prev_season_label",
          "prev_season_ppg_home", "prev_season_gf_home", "prev_season_ga_home",
          "prev_season_ppg_away", "prev_season_gf_away", "prev_season_ga_away"]]
        .reset_index(drop=True)
    )
    return prev_lookup


# ============================================================
# 10. VENUE-SPECIFIC PROMOTED BASELINE (added at Step 08B)
# ============================================================

def compute_promoted_baseline_venue(train_df: pd.DataFrame,
                                    promoted_table: dict,
                                    early_gameweeks: int = 5) -> dict:
    """
    Same as compute_promoted_baseline but returns venue-specific values.
    """
    train = train_df.copy().sort_values("Date").reset_index(drop=True)

    home_rows, away_rows = [], []
    for season_label, group in train.groupby("season_label"):
        promoted = get_promoted_teams(promoted_table, season_label)
        if not promoted:
            continue
        for team in promoted:
            team_matches = train[
                (train["season_label"] == season_label) &
                ((train["HomeTeam"] == team) | (train["AwayTeam"] == team))
            ].sort_values("Date").head(early_gameweeks)
            home_rows.append(team_matches[team_matches["HomeTeam"] == team])
            away_rows.append(team_matches[team_matches["AwayTeam"] == team])

    def summarize(rows, side):
        non_empty = [r for r in rows if len(r) > 0]
        if not non_empty:
            return {"ppg": 1.0, "gf": 1.0, "ga": 1.7}
        bdf = pd.concat(non_empty, ignore_index=True)
        if side == "home":
            gf = bdf["FTHG"]; ga = bdf["FTAG"]
            pts = ((gf > ga).astype(int) * 3 + (gf == ga).astype(int) * 1).mean()
        else:
            gf = bdf["FTAG"]; ga = bdf["FTHG"]
            pts = ((gf > ga).astype(int) * 3 + (gf == ga).astype(int) * 1).mean()
        return {"ppg": float(pts), "gf": float(gf.mean()), "ga": float(ga.mean())}

    h = summarize(home_rows, "home")
    a = summarize(away_rows, "away")
    return {
        "promoted_baseline_ppg_home": h["ppg"],
        "promoted_baseline_gf_home":  h["gf"],
        "promoted_baseline_ga_home":  h["ga"],
        "promoted_baseline_ppg_away": a["ppg"],
        "promoted_baseline_gf_away":  a["gf"],
        "promoted_baseline_ga_away":  a["ga"],
    }


# ============================================================
# 11. VENUE-SPECIFIC ROLLING FORM (added at Step 08B)
# ============================================================

def build_venue_form_filled(matches_df: pd.DataFrame,
                            prev_form_venue_df: pd.DataFrame,
                            promoted_baseline_venue: dict,
                            promoted_table: dict,
                            n: int = 5) -> pd.DataFrame:
    """
    Compute venue-conditional rolling form:

      home_venue_*_last{n} = home team's last n HOME matches
      away_venue_*_last{n} = away team's last n AWAY matches

    Season-scoped. Early-season NaN filled via weighted blend with
    venue-specific previous-season form (or venue-specific promoted baseline).

    Returns 8 columns:
      home_venue_pts_last{n}, home_venue_gf_last{n}, home_venue_ga_last{n}, home_venue_gd_last{n}
      away_venue_pts_last{n}, away_venue_gf_last{n}, away_venue_ga_last{n}, away_venue_gd_last{n}
    """
    df = matches_df.copy().reset_index(drop=True)
    df = df.sort_values(["season_label", "Date"]).reset_index(drop=True)
    df["_row_id"] = df.index

    # ---- Long form, one row per (match, team) ----
    def make_side(side):
        if side == "home":
            d = pd.DataFrame({
                "row_id": df["_row_id"],
                "Date": df["Date"],
                "season_label": df["season_label"],
                "team": df["HomeTeam"],
                "goals_for": df["FTHG"],
                "goals_against": df["FTAG"],
                "is_home": 1,
            })
        else:
            d = pd.DataFrame({
                "row_id": df["_row_id"],
                "Date": df["Date"],
                "season_label": df["season_label"],
                "team": df["AwayTeam"],
                "goals_for": df["FTAG"],
                "goals_against": df["FTHG"],
                "is_home": 0,
            })
        d["gd"] = d["goals_for"] - d["goals_against"]
        d["pts"] = np.where(d["goals_for"] > d["goals_against"], 3,
                     np.where(d["goals_for"] == d["goals_against"], 1, 0))
        return d

    long_df = pd.concat([make_side("home"), make_side("away")], ignore_index=True)
    long_df = long_df.sort_values(["team", "is_home", "season_label", "Date", "row_id"]).reset_index(drop=True)

    # ---- Rolling within (team, is_home, season) ----
    gb_cols = ["team", "is_home", "season_label"]
    def roll_venue(col):
        return (long_df.groupby(gb_cols)[col]
                       .transform(lambda s: s.shift(1).rolling(n, min_periods=1).mean()))

    long_df["venue_pts_last"] = roll_venue("pts")
    long_df["venue_gf_last"]  = roll_venue("goals_for")
    long_df["venue_ga_last"]  = roll_venue("goals_against")
    long_df["venue_gd_last"]  = roll_venue("gd")

    # ---- Match-level output frame, aligned with `df` ----
    home_feats = long_df[long_df["is_home"] == 1].set_index("row_id")
    away_feats = long_df[long_df["is_home"] == 0].set_index("row_id")

    out = pd.DataFrame(index=df.index)
    for src, dst in [
        ("venue_pts_last", f"pts_last{n}"),
        ("venue_gf_last",  f"gf_last{n}"),
        ("venue_ga_last",  f"ga_last{n}"),
        ("venue_gd_last",  f"gd_last{n}"),
    ]:
        out[f"home_venue_{dst}"] = home_feats[src].reindex(df.index).values
        out[f"away_venue_{dst}"] = away_feats[src].reindex(df.index).values

    # ---- Compute k (in-season matches so far, per team per venue) ----
    df["_k_home_venue"] = df.groupby(["HomeTeam", "season_label"]).cumcount()
    df["_k_away_venue"] = df.groupby(["AwayTeam", "season_label"]).cumcount()
    # Count only same-venue matches: recompute via long table
    long_df["k"] = long_df.groupby(gb_cols).cumcount()
    home_k = long_df[long_df["is_home"] == 1].set_index("row_id")["k"].reindex(df.index).values
    away_k = long_df[long_df["is_home"] == 0].set_index("row_id")["k"].reindex(df.index).values

    # ---- Blend with venue-specific previous-season form ----
    prev_map = prev_form_venue_df.set_index(["team", "season_label"]).to_dict("index")

    def blend_side(side, k_arr):
        team_col = "HomeTeam" if side == "home" else "AwayTeam"
        venue_key = "home" if side == "home" else "away"
        prev_prefix = f"prev_season_ppg_{venue_key}", f"prev_season_gf_{venue_key}", f"prev_season_ga_{venue_key}"
        base_prefix = (f"promoted_baseline_ppg_{venue_key}",
                       f"promoted_baseline_gf_{venue_key}",
                       f"promoted_baseline_ga_{venue_key}")
        for metric, prev_key, base_key in zip(
            ["pts", "gf", "ga"], prev_prefix, base_prefix
        ):
            col = f"{side}_venue_{metric}_last{n}"
            values = out[col].values.copy()
            for i in range(len(df)):
                k = int(k_arr[i])
                if k >= n:
                    continue
                rolling_val = values[i]
                in_season_avg = 0.0 if pd.isna(rolling_val) else rolling_val
                team = df.iloc[i][team_col]
                season = df.iloc[i]["season_label"]
                prev = prev_map.get((team, season))
                if prev is not None:
                    prev_val = prev[prev_key]
                else:
                    prev_val = promoted_baseline_venue[base_key]
                values[i] = (k * in_season_avg + (n - k) * prev_val) / n
            out[col] = values

        # Derive gd from gf - ga
        out[f"{side}_venue_gd_last{n}"] = (
            out[f"{side}_venue_gf_last{n}"] - out[f"{side}_venue_ga_last{n}"]
        )

    blend_side("home", home_k)
    blend_side("away", away_k)

    # ---- Restore original order ----
    out.index = df.index
    return out
'''

module_path = '/content/drive/MyDrive/epl_match_outcome/src/features.py'
with open(module_path, 'a') as f:
    f.write(venue_form_code)

import importlib, sys, os
SRC_DIR = '/content/drive/MyDrive/epl_match_outcome/src'
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
import features
importlib.reload(features)

print("Appended venue-form functions.")
print("New public functions:", [n for n in dir(features)
                               if not n.startswith("_") and callable(getattr(features, n))])

In [ ]:
import pandas as pd, numpy as np, os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])

PROMOTED_TABLE = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}
train_promoted = {k: v for k, v in PROMOTED_TABLE.items() if k in train['season_label'].unique()}

# Compute venue-specific fallbacks from train only
prev_form_venue = features.compute_prev_season_form_venue(train)
promoted_base_v = features.compute_promoted_baseline_venue(train, train_promoted, early_gameweeks=5)

print("Venue-specific promoted baseline:")
for k, v in promoted_base_v.items():
    print(f"  {k}: {v:.3f}")

# Build venue-form features
venue_form = features.build_venue_form_filled(
    train, prev_form_venue, promoted_base_v, PROMOTED_TABLE, n=5
)

print("\nVenue form shape:", venue_form.shape)
print("Columns:", list(venue_form.columns))

nan_counts = venue_form.isna().sum()
print("\nNaN counts:")
print(nan_counts[nan_counts > 0].to_string() if (nan_counts > 0).any() else "(none) ✅")

# Merge with existing train_form.csv
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
train_form = train_form.drop(columns=[c for c in venue_form.columns if c in train_form.columns], errors='ignore')
train_form = pd.concat([train_form.reset_index(drop=True),
                        venue_form.reset_index(drop=True)], axis=1)

out_path = os.path.join(PROC_DIR, 'train_form.csv')
train_form.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("Total columns:", train_form.shape[1])
print("Form-related columns:", [c for c in train_form.columns
                                if 'form' in c or 'venue' in c or 'promoted' in c])

In [ ]:
# Burnley 2017/18 mid-season: strong home form, weak away form
TEAM, SEASON = "Burnley", "2017/18"
season_matches = train[(train['season_label'] == SEASON) &
                       ((train['HomeTeam'] == TEAM) | (train['AwayTeam'] == TEAM))
                      ].sort_values('Date').reset_index(drop=True)

# Pick a mid-season match (index ~15)
target_i = 15
target_match = season_matches.iloc[target_i]
mask = (train['Date'] == target_match['Date']) & \
       (train['HomeTeam'] == target_match['HomeTeam']) & \
       (train['AwayTeam'] == target_match['AwayTeam'])
idx = train.index[mask][0]

print("Target match:")
print(target_match[['Date','HomeTeam','AwayTeam','FTHG','FTAG','FTR']].to_dict())
print()

# If Burnley is home in this match:
if target_match['HomeTeam'] == TEAM:
    home_form = train_form.loc[idx, 'home_form_pts_last5']
    home_venue = train_form.loc[idx, 'home_venue_pts_last5']
    print(f"Burnley at home:  overall form = {home_form:.3f}   venue form = {home_venue:.3f}")
else:
    away_form = train_form.loc[idx, 'away_form_pts_last5']
    away_venue = train_form.loc[idx, 'away_venue_pts_last5']
    print(f"Burnley away:     overall form = {away_form:.3f}   venue form = {away_venue:.3f}")

# Show Burnley's home vs away matches so far this season
burnley_matches = season_matches.iloc[:target_i]
burnley_home = burnley_matches[burnley_matches['HomeTeam'] == TEAM]
burnley_away = burnley_matches[burnley_matches['AwayTeam'] == TEAM]
print(f"\nBurnley's matches so far: {len(burnley_home)} home, {len(burnley_away)} away")

def pts_from(matches, team, side):
    pts = []
    for _, m in matches.iterrows():
        gf = m['FTHG'] if side == 'home' else m['FTAG']
        ga = m['FTAG'] if side == 'home' else m['FTHG']
        pts.append(3 if gf > ga else (1 if gf == ga else 0))
    return pts

home_pts = pts_from(burnley_home, TEAM, 'home')
away_pts = pts_from(burnley_away, TEAM, 'away')
print(f"  Home points so far: {home_pts}  → avg {np.mean(home_pts):.2f}" if home_pts else "")
print(f"  Away points so far: {away_pts}  → avg {np.mean(away_pts):.2f}" if away_pts else "")
print()
print("(Burnley 2017/18 famously finished 7th with strong home form)")

In [ ]:
# Verify Burnley's home venue form matches the actual last-5-home average
mask = (train['Date'] == '2017-12-09') & \
       (train['HomeTeam'] == 'Burnley') & \
       (train['AwayTeam'] == 'Watford')
idx = train.index[mask][0]

burnley_home_before = train[
    (train['season_label'] == '2017/18') &
    (train['HomeTeam'] == 'Burnley') &
    (train['Date'] < '2017-12-09')
].sort_values('Date').tail(5)

def row_pts(row):
    return 3 if row['FTHG'] > row['FTAG'] else (1 if row['FTHG'] == row['FTAG'] else 0)

pts_list = [row_pts(r) for _, r in burnley_home_before.iterrows()]
print("Last 5 Burnley home matches before 2017-12-09:")
print(burnley_home_before[['Date','HomeTeam','AwayTeam','FTHG','FTAG']].to_string(index=False))
print("Points:", pts_list, "→ avg:", sum(pts_list)/len(pts_list))

print("\nFeature (train_form.csv) home_venue_pts_last5 =",
      train_form.loc[idx, 'home_venue_pts_last5'])

In [ ]:
match_stats_code = '''

# ============================================================
# 12. VENUE-SPECIFIC PREVIOUS-SEASON MATCH STATS (added at Step 08C)
# ============================================================

def compute_prev_season_match_stats_venue(train_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each team and each TARGET season, compute venue-specific match
    stats from the PREVIOUS season: shots, shots on target, corners.

    Returns columns:
      team, season_label (target), prev_season_label (source),
      prev_season_shots_home, prev_season_sot_home, prev_season_corners_home,
      prev_season_shots_away, prev_season_sot_away, prev_season_corners_away
    """
    rows = []
    for (season_label, team), group in _iter_team_seasons(train_df):
        home_group = group[group["HomeTeam"] == team]
        away_group = group[group["AwayTeam"] == team]

        def mean_or_nan(sub, cols):
            if len(sub) == 0:
                return tuple([np.nan] * len(cols))
            return tuple(float(sub[c].mean()) for c in cols)

        sh_h, sot_h, cn_h = mean_or_nan(home_group, ["HS", "HST", "HC"])
        sh_a, sot_a, cn_a = mean_or_nan(away_group, ["AS", "AST", "AC"])

        rows.append({
            "team": team,
            "prev_season_label": season_label,
            "shots_home": sh_h,  "sot_home": sot_h,  "corners_home": cn_h,
            "shots_away": sh_a,  "sot_away": sot_a,  "corners_away": cn_a,
        })

    end_of_season = pd.DataFrame(rows)

    season_order = sorted(end_of_season["prev_season_label"].unique())
    next_season = {s: season_order[i+1] for i, s in enumerate(season_order) if i+1 < len(season_order)}
    end_of_season["season_label"] = end_of_season["prev_season_label"].map(next_season)

    return (
        end_of_season
        .dropna(subset=["season_label"])
        .rename(columns={
            "shots_home":   "prev_season_shots_home",
            "sot_home":     "prev_season_sot_home",
            "corners_home": "prev_season_corners_home",
            "shots_away":   "prev_season_shots_away",
            "sot_away":     "prev_season_sot_away",
            "corners_away": "prev_season_corners_away",
        })
        [["team", "season_label", "prev_season_label",
          "prev_season_shots_home", "prev_season_sot_home", "prev_season_corners_home",
          "prev_season_shots_away", "prev_season_sot_away", "prev_season_corners_away"]]
        .reset_index(drop=True)
    )


# ============================================================
# 13. VENUE-SPECIFIC PROMOTED BASELINE FOR MATCH STATS
# ============================================================

def compute_promoted_baseline_match_stats(train_df: pd.DataFrame,
                                          promoted_table: dict,
                                          early_gameweeks: int = 5) -> dict:
    """
    Promoted-team baseline for shots, SOT, corners — venue-specific.
    """
    train = train_df.copy().sort_values("Date").reset_index(drop=True)

    home_rows, away_rows = [], []
    for season_label, group in train.groupby("season_label"):
        promoted = get_promoted_teams(promoted_table, season_label)
        if not promoted:
            continue
        for team in promoted:
            team_matches = train[
                (train["season_label"] == season_label) &
                ((train["HomeTeam"] == team) | (train["AwayTeam"] == team))
            ].sort_values("Date").head(early_gameweeks)
            home_rows.append(team_matches[team_matches["HomeTeam"] == team])
            away_rows.append(team_matches[team_matches["AwayTeam"] == team])

    def summarize(rows, cols):
        non_empty = [r for r in rows if len(r) > 0]
        if not non_empty:
            return {c: 0.0 for c in cols}
        bdf = pd.concat(non_empty, ignore_index=True)
        return {c: float(bdf[c].mean()) for c in cols}

    home_stats = summarize(home_rows, ["HS", "HST", "HC"])
    away_stats = summarize(away_rows, ["AS", "AST", "AC"])

    return {
        "promoted_baseline_shots_home":   home_stats["HS"],
        "promoted_baseline_sot_home":     home_stats["HST"],
        "promoted_baseline_corners_home": home_stats["HC"],
        "promoted_baseline_shots_away":   away_stats["AS"],
        "promoted_baseline_sot_away":     away_stats["AST"],
        "promoted_baseline_corners_away": away_stats["AC"],
    }


# ============================================================
# 14. ROLLING MATCH STATS (venue-specific, season-scoped)
# ============================================================

def build_match_stats_filled(matches_df: pd.DataFrame,
                             prev_stats_df: pd.DataFrame,
                             promoted_baseline_stats: dict,
                             promoted_table: dict,
                             n: int = 5) -> pd.DataFrame:
    """
    Rolling last-5 match stats, per team per venue, season-scoped.

    Output columns (8):
      home_form_shots_last{n},     away_form_shots_last{n}
      home_form_sot_last{n},       away_form_sot_last{n}
      home_form_corners_last{n},   away_form_corners_last{n}
      home_form_sot_ratio_last{n}, away_form_sot_ratio_last{n}
    """
    df = matches_df.copy().reset_index(drop=True).sort_values(
        ["season_label", "Date"]).reset_index(drop=True)
    df["_row_id"] = df.index

    # ---- Long form, per (match, team) with venue ----
    def make_side(side):
        if side == "home":
            d = pd.DataFrame({
                "row_id": df["_row_id"],
                "Date": df["Date"],
                "season_label": df["season_label"],
                "team": df["HomeTeam"],
                "shots": df["HS"], "sot": df["HST"], "corners": df["HC"],
                "is_home": 1,
            })
        else:
            d = pd.DataFrame({
                "row_id": df["_row_id"],
                "Date": df["Date"],
                "season_label": df["season_label"],
                "team": df["AwayTeam"],
                "shots": df["AS"], "sot": df["AST"], "corners": df["AC"],
                "is_home": 0,
            })
        d["sot_ratio"] = np.where(d["shots"] > 0, d["sot"] / d["shots"], np.nan)
        return d

    long_df = pd.concat([make_side("home"), make_side("away")], ignore_index=True)
    long_df = long_df.sort_values(["team", "is_home", "season_label", "Date", "row_id"]).reset_index(drop=True)

    gb_cols = ["team", "is_home", "season_label"]
    def roll_stat(col):
        return (long_df.groupby(gb_cols)[col]
                       .transform(lambda s: s.shift(1).rolling(n, min_periods=1).mean()))

    long_df["stat_shots_last"]   = roll_stat("shots")
    long_df["stat_sot_last"]     = roll_stat("sot")
    long_df["stat_corners_last"] = roll_stat("corners")
    long_df["stat_sot_ratio_last"] = roll_stat("sot_ratio")

    # ---- Back to match level ----
    home_feats = long_df[long_df["is_home"] == 1].set_index("row_id")
    away_feats = long_df[long_df["is_home"] == 0].set_index("row_id")

    out = pd.DataFrame(index=df.index)
    for src, dst in [
        ("stat_shots_last",     f"form_shots_last{n}"),
        ("stat_sot_last",       f"form_sot_last{n}"),
        ("stat_corners_last",   f"form_corners_last{n}"),
        ("stat_sot_ratio_last", f"form_sot_ratio_last{n}"),
    ]:
        out[f"home_{dst}"] = home_feats[src].reindex(df.index).values
        out[f"away_{dst}"] = away_feats[src].reindex(df.index).values

    # ---- k (in-season venue matches so far) ----
    long_df["k"] = long_df.groupby(gb_cols).cumcount()
    home_k = long_df[long_df["is_home"] == 1].set_index("row_id")["k"].reindex(df.index).values
    away_k = long_df[long_df["is_home"] == 0].set_index("row_id")["k"].reindex(df.index).values

    # ---- Blend with venue-specific previous-season stats ----
    prev_map = prev_stats_df.set_index(["team", "season_label"]).to_dict("index")

    stat_specs = [
        ("shots",     "form_shots_last{n}",     "prev_season_shots_{v}",     "promoted_baseline_shots_{v}"),
        ("sot",       "form_sot_last{n}",       "prev_season_sot_{v}",       "promoted_baseline_sot_{v}"),
        ("corners",   "form_corners_last{n}",   "prev_season_corners_{v}",   "promoted_baseline_corners_{v}"),
    ]

    def blend_side(side, k_arr):
        team_col = "HomeTeam" if side == "home" else "AwayTeam"
        venue = "home" if side == "home" else "away"
        for _, col_tmpl, prev_tmpl, base_tmpl in stat_specs:
            col = f"{side}_{col_tmpl.format(n=n)}"
            prev_key = prev_tmpl.format(v=venue)
            base_key = base_tmpl.format(v=venue)
            values = out[col].values.copy()
            for i in range(len(df)):
                k = int(k_arr[i])
                if k >= n:
                    continue
                rolling_val = values[i]
                in_season_avg = 0.0 if pd.isna(rolling_val) else rolling_val
                team = df.iloc[i][team_col]
                season = df.iloc[i]["season_label"]
                prev = prev_map.get((team, season))
                if prev is not None and not pd.isna(prev[prev_key]):
                    prev_val = prev[prev_key]
                else:
                    prev_val = promoted_baseline_stats[base_key]
                values[i] = (k * in_season_avg + (n - k) * prev_val) / n
            out[col] = values

        # sot_ratio: derive from blended shots and sot
        col_sot = f"{side}_form_sot_last{n}"
        col_sh  = f"{side}_form_shots_last{n}"
        col_ratio = f"{side}_form_sot_ratio_last{n}"
        safe_sh = out[col_sh].replace(0, np.nan)
        out[col_ratio] = (out[col_sot] / safe_sh).fillna(0.0)

    blend_side("home", home_k)
    blend_side("away", away_k)

    out.index = df.index
    # Restore original (pre-sort) row order
    out = out.reindex(matches_df.index)
    return out
'''

module_path = '/content/drive/MyDrive/epl_match_outcome/src/features.py'
with open(module_path, 'a') as f:
    f.write(match_stats_code)

import importlib, sys, os
SRC_DIR = '/content/drive/MyDrive/epl_match_outcome/src'
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
import features
importlib.reload(features)

print("Appended match-stat functions.")
print("Total public functions:", len([n for n in dir(features)
                                      if not n.startswith("_") and callable(getattr(features, n))]))

In [ ]:
import pandas as pd, numpy as np, os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])

PROMOTED_TABLE = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}
train_promoted = {k: v for k, v in PROMOTED_TABLE.items() if k in train['season_label'].unique()}

# Compute fallbacks from train
prev_stats_venue  = features.compute_prev_season_match_stats_venue(train)
promoted_base_ms  = features.compute_promoted_baseline_match_stats(train, train_promoted, early_gameweeks=5)

print("Promoted baseline for match stats:")
for k, v in promoted_base_ms.items():
    print(f"  {k}: {v:.3f}")

# Build
stats_form = features.build_match_stats_filled(
    train, prev_stats_venue, promoted_base_ms, PROMOTED_TABLE, n=5
)

print("\nMatch-stats form shape:", stats_form.shape)
print("Columns:", list(stats_form.columns))

nan_counts = stats_form.isna().sum()
print("\nNaN counts:")
print(nan_counts[nan_counts > 0].to_string() if (nan_counts > 0).any() else "(none) ✅")

# Merge into train_form.csv
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
train_form = train_form.drop(columns=[c for c in stats_form.columns if c in train_form.columns], errors='ignore')
train_form = pd.concat([train_form.reset_index(drop=True),
                        stats_form.reset_index(drop=True)], axis=1)

out_path = os.path.join(PROC_DIR, 'train_form.csv')
train_form.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("Total columns:", train_form.shape[1])
print("\nNew columns added this step:",
      [c for c in train_form.columns if 'form_shots' in c or 'form_sot' in c or 'form_corners' in c])

In [ ]:
# Spot-check: Burnley's shots form at the same match we checked earlier
mask = (train['Date'] == '2017-12-09') & \
       (train['HomeTeam'] == 'Burnley') & \
       (train['AwayTeam'] == 'Watford')
idx = train.index[mask][0]

# Burnley's last 5 home matches BEFORE this one
burnley_home_before = train[
    (train['season_label'] == '2017/18') &
    (train['HomeTeam'] == 'Burnley') &
    (train['Date'] < '2017-12-09')
].sort_values('Date').tail(5)

print("Last 5 Burnley home matches before 2017-12-09:")
print(burnley_home_before[['Date','HomeTeam','AwayTeam','HS','HST','HC','FTHG','FTAG']].to_string(index=False))
print()
print("Manual averages:")
print(f"  Shots:   {burnley_home_before['HS'].mean():.3f}")
print(f"  SOT:     {burnley_home_before['HST'].mean():.3f}")
print(f"  Corners: {burnley_home_before['HC'].mean():.3f}")

print("\nFeature values (train_form.csv):")
print(f"  home_form_shots_last5     = {train_form.loc[idx, 'home_form_shots_last5']:.3f}")
print(f"  home_form_sot_last5       = {train_form.loc[idx, 'home_form_sot_last5']:.3f}")
print(f"  home_form_corners_last5   = {train_form.loc[idx, 'home_form_corners_last5']:.3f}")

# Assertions
assert abs(train_form.loc[idx, 'home_form_shots_last5'] - burnley_home_before['HS'].mean()) < 1e-6
assert abs(train_form.loc[idx, 'home_form_sot_last5']   - burnley_home_before['HST'].mean()) < 1e-6
assert abs(train_form.loc[idx, 'home_form_corners_last5'] - burnley_home_before['HC'].mean()) < 1e-6
print("\n✅ Match-stat form verified against manual computation.")

In [ ]:
season_to_date_code = '''

# ============================================================
# 15. SEASON-TO-DATE FEATURES (added at Step 08D)
# ============================================================

def compute_prev_season_gd_per_game_venue(train_df: pd.DataFrame) -> pd.DataFrame:
    """
    Extend the venue-specific previous-season form with GD per game.
    Small helper that just derives from the existing lookup.
    """
    prev = compute_prev_season_form_venue(train_df)
    prev = prev.copy()
    prev["prev_season_gd_per_game_home"] = prev["prev_season_gf_home"] - prev["prev_season_ga_home"]
    prev["prev_season_gd_per_game_away"] = prev["prev_season_gf_away"] - prev["prev_season_ga_away"]
    return prev


def build_season_to_date_filled(matches_df: pd.DataFrame,
                                prev_form_venue_df: pd.DataFrame,
                                promoted_baseline_venue: dict,
                                promoted_table: dict) -> pd.DataFrame:
    """
    Cumulative season-to-date features, computed as of BEFORE the current
    match. Season-scoped (resets at season start).

    Output columns (6):
      home_season_ppg,           away_season_ppg
      home_season_gd_per_game,   away_season_gd_per_game
      home_season_matches_played, away_season_matches_played

    Filling rules:
      ppg and gd_per_game: C3 weighted blend using venue-specific prev-season form
      matches_played: actual count (no fill)
    """
    df = matches_df.copy().reset_index(drop=True).sort_values(
        ["season_label", "Date"]).reset_index(drop=True)
    df["_row_id"] = df.index

    # ---- Long form per (match, team) with venue ----
    def make_side(side):
        if side == "home":
            d = pd.DataFrame({
                "row_id": df["_row_id"],
                "Date": df["Date"],
                "season_label": df["season_label"],
                "team": df["HomeTeam"],
                "goals_for":     df["FTHG"],
                "goals_against": df["FTAG"],
                "is_home": 1,
            })
        else:
            d = pd.DataFrame({
                "row_id": df["_row_id"],
                "Date": df["Date"],
                "season_label": df["season_label"],
                "team": df["AwayTeam"],
                "goals_for":     df["FTAG"],
                "goals_against": df["FTHG"],
                "is_home": 0,
            })
        d["gd"] = d["goals_for"] - d["goals_against"]
        d["pts"] = np.where(d["goals_for"] > d["goals_against"], 3,
                     np.where(d["goals_for"] == d["goals_against"], 1, 0))
        return d

    long_df = pd.concat([make_side("home"), make_side("away")], ignore_index=True)
    long_df = long_df.sort_values(["team", "season_label", "Date", "row_id"]).reset_index(drop=True)

    # ---- Cumulative season-to-date (shifted by one match) ----
    gb = long_df.groupby(["team", "season_label"])

    long_df["_cum_pts"]  = gb["pts"].transform(lambda s: s.shift(1).expanding().sum())
    long_df["_cum_gd"]   = gb["gd"].transform(lambda s: s.shift(1).expanding().sum())
    long_df["_cum_n"]    = gb.cumcount()  # matches played BEFORE current match

    # ppg and gd_per_game = cum / n (n = matches played before current match)
    safe_n = long_df["_cum_n"].replace(0, np.nan)
    long_df["std_ppg"]  = long_df["_cum_pts"] / safe_n
    long_df["std_gdg"]  = long_df["_cum_gd"]  / safe_n
    long_df["std_n"]    = long_df["_cum_n"]

    # ---- To match-level ----
    home_feats = long_df[long_df["is_home"] == 1].set_index("row_id")
    away_feats = long_df[long_df["is_home"] == 0].set_index("row_id")

    out = pd.DataFrame(index=df.index)
    out["home_season_ppg"]              = home_feats["std_ppg"].reindex(df.index).values
    out["away_season_ppg"]              = away_feats["std_ppg"].reindex(df.index).values
    out["home_season_gd_per_game"]      = home_feats["std_gdg"].reindex(df.index).values
    out["away_season_gd_per_game"]      = away_feats["std_gdg"].reindex(df.index).values
    out["home_season_matches_played"]   = home_feats["std_n"].reindex(df.index).values
    out["away_season_matches_played"]   = away_feats["std_n"].reindex(df.index).values

    # ---- Fill ppg / gd_per_game via C3 blend (matches_played NOT filled) ----
    prev_map = prev_form_venue_df.set_index(["team", "season_label"]).to_dict("index")

    def blend_side(side, k_arr):
        team_col = "HomeTeam" if side == "home" else "AwayTeam"
        venue    = "home" if side == "home" else "away"
        specs = [
            ("ppg",         f"{side}_season_ppg",         f"prev_season_ppg_{venue}",         f"promoted_baseline_ppg_{venue}"),
            ("gd_per_game", f"{side}_season_gd_per_game", f"prev_season_gd_per_game_{venue}", None),
        ]
        for metric, col, prev_key, base_key in specs:
            values = out[col].values.copy()
            for i in range(len(df)):
                k = int(k_arr[i])
                if k >= 1:
                    continue  # has in-season data → ppg is genuine
                team = df.iloc[i][team_col]
                season = df.iloc[i]["season_label"]
                prev = prev_map.get((team, season))
                if prev is not None and not pd.isna(prev[prev_key]):
                    values[i] = prev[prev_key]
                else:
                    # promoted: use promoted baseline
                    if base_key is None:
                        # gd_per_game: derive from gf - ga baseline if needed
                        base_val = (promoted_baseline_venue.get(f"promoted_baseline_gf_{venue}", 1.0)
                                    - promoted_baseline_venue.get(f"promoted_baseline_ga_{venue}", 1.7))
                    else:
                        base_val = promoted_baseline_venue.get(base_key, 0.0)
                    values[i] = base_val
            out[col] = values

    home_k = home_feats["std_n"].reindex(df.index).values
    away_k = away_feats["std_n"].reindex(df.index).values
    blend_side("home", home_k)
    blend_side("away", away_k)

    # ---- Restore original row order ----
    out = out.reindex(matches_df.index)
    return out
'''

module_path = '/content/drive/MyDrive/epl_match_outcome/src/features.py'
with open(module_path, 'a') as f:
    f.write(season_to_date_code)

import importlib, sys, os
SRC_DIR = '/content/drive/MyDrive/epl_match_outcome/src'
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
import features
importlib.reload(features)

print("Appended season-to-date functions.")
print("Total public functions:", len([n for n in dir(features)
                                      if not n.startswith("_") and callable(getattr(features, n))]))

In [ ]:
import pandas as pd, numpy as np, os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])

PROMOTED_TABLE = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}
train_promoted = {k: v for k, v in PROMOTED_TABLE.items() if k in train['season_label'].unique()}

prev_form_venue = features.compute_prev_season_gd_per_game_venue(train)
promoted_base_v = features.compute_promoted_baseline_venue(train, train_promoted, early_gameweeks=5)

std_form = features.build_season_to_date_filled(
    train, prev_form_venue, promoted_base_v, PROMOTED_TABLE
)

print("Season-to-date shape:", std_form.shape)
print("Columns:", list(std_form.columns))

nan_counts = std_form.isna().sum()
print("\nNaN counts:")
print(nan_counts[nan_counts > 0].to_string() if (nan_counts > 0).any() else "(none) ✅")

# Save
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
train_form = train_form.drop(columns=[c for c in std_form.columns if c in train_form.columns], errors='ignore')
train_form = pd.concat([train_form.reset_index(drop=True),
                        std_form.reset_index(drop=True)], axis=1)

out_path = os.path.join(PROC_DIR, 'train_form.csv')
train_form.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("Total columns:", train_form.shape[1])

In [ ]:
# Take Man City at match 15 of 2016/17 — should have 14 matches played, real ppg
mask_season = train['season_label'] == '2016/17'
man_city_1617 = train[mask_season & ((train['HomeTeam']=='Man City') | (train['AwayTeam']=='Man City'))].sort_values('Date').reset_index(drop=True)
target = man_city_1617.iloc[14]

mask = (train['Date'] == target['Date']) & (train['HomeTeam'] == target['HomeTeam']) & (train['AwayTeam'] == target['AwayTeam'])
idx = train.index[mask][0]

side = 'home' if target['HomeTeam'] == 'Man City' else 'away'
print(f"Target match #{14}: {target['HomeTeam']} vs {target['AwayTeam']} on {target['Date'].date()}")
print()
print(f"Man City season-to-date BEFORE this match:")
print(f"  matches_played = {train_form.loc[idx, f'{side}_season_matches_played']}")
print(f"  ppg            = {train_form.loc[idx, f'{side}_season_ppg']:.3f}")
print(f"  gd_per_game    = {train_form.loc[idx, f'{side}_season_gd_per_game']:.3f}")

# Manual check
mc_before = man_city_1617.iloc[:14]
def pts_from(m, team):
    gf = m['FTHG'] if m['HomeTeam']==team else m['FTAG']
    ga = m['FTAG'] if m['HomeTeam']==team else m['FTHG']
    return 3 if gf > ga else (1 if gf == ga else 0)
def gd_from(m, team):
    gf = m['FTHG'] if m['HomeTeam']==team else m['FTAG']
    ga = m['FTAG'] if m['HomeTeam']==team else m['FTHG']
    return gf - ga

total_pts = sum(pts_from(m, 'Man City') for _, m in mc_before.iterrows())
total_gd  = sum(gd_from(m, 'Man City')  for _, m in mc_before.iterrows())
manual_ppg = total_pts / 14
manual_gdg = total_gd / 14

print(f"\nManual: {total_pts} pts, {total_gd} GD over 14 matches → ppg {manual_ppg:.3f}, gd/g {manual_gdg:.3f}")

assert abs(train_form.loc[idx, f'{side}_season_ppg'] - manual_ppg) < 1e-6
assert abs(train_form.loc[idx, f'{side}_season_gd_per_game'] - manual_gdg) < 1e-6
assert train_form.loc[idx, f'{side}_season_matches_played'] == 14
print("\n✅ Season-to-date verified against manual computation.")

In [ ]:
rest_days_code = '''

# ============================================================
# 16. REST DAYS (added at Step 08E)
# ============================================================

def build_rest_days(matches_df: pd.DataFrame,
                    cap: int = 14) -> pd.DataFrame:
    """
    For each match, compute days of rest for the home and away team
    since their previous match in the dataset.

    Rules:
      - Rest = (this match date) - (team's previous match date) in days.
      - Cap at `cap` (default 14) to tame the long tail.
      - First-ever match in dataset -> cap.
      - Season openers naturally resolve to cap.

    Output columns (2): home_rest_days, away_rest_days
    """
    df = matches_df.copy().reset_index(drop=True).sort_values(
        ["season_label", "Date"]).reset_index(drop=True)
    df["_row_id"] = df.index

    # Long form per (match, team)
    def make_side(side):
        team_col = "HomeTeam" if side == "home" else "AwayTeam"
        return pd.DataFrame({
            "row_id": df["_row_id"],
            "Date": df["Date"],
            "team": df[team_col],
            "is_home": 1 if side == "home" else 0,
        })

    long_df = pd.concat([make_side("home"), make_side("away")], ignore_index=True)
    long_df = long_df.sort_values(["team", "Date", "row_id"]).reset_index(drop=True)

    # Previous match date per team (shifted by one)
    long_df["prev_date"] = long_df.groupby("team")["Date"].shift(1)

    # Days since previous match
    long_df["rest_days"] = (long_df["Date"] - long_df["prev_date"]).dt.days

    # Fill NaN (first ever match) and cap
    long_df["rest_days"] = long_df["rest_days"].fillna(cap).clip(upper=cap)

    # Back to match-level
    home_feats = long_df[long_df["is_home"] == 1].set_index("row_id")
    away_feats = long_df[long_df["is_home"] == 0].set_index("row_id")

    out = pd.DataFrame(index=df.index)
    out["home_rest_days"] = home_feats["rest_days"].reindex(df.index).values
    out["away_rest_days"] = away_feats["rest_days"].reindex(df.index).values

    # Restore original order
    out = out.reindex(matches_df.index)
    return out
'''

module_path = '/content/drive/MyDrive/epl_match_outcome/src/features.py'
with open(module_path, 'a') as f:
    f.write(rest_days_code)

import importlib, sys, os
SRC_DIR = '/content/drive/MyDrive/epl_match_outcome/src'
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
import features
importlib.reload(features)

print("Appended rest-days function.")
print("Total public functions:", len([n for n in dir(features)
                                      if not n.startswith("_") and callable(getattr(features, n))]))

In [ ]:
import pandas as pd, numpy as np, os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])

rest = features.build_rest_days(train, cap=14)
print("Rest-days shape:", rest.shape)
print("Columns:", list(rest.columns))
print()
print(rest.describe().round(3).to_string())

# Distribution summary
print()
print("Distribution of home_rest_days:")
print(rest["home_rest_days"].value_counts().sort_index().to_string())

# Merge
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
train_form = train_form.drop(columns=[c for c in rest.columns if c in train_form.columns], errors='ignore')
train_form = pd.concat([train_form.reset_index(drop=True),
                        rest.reset_index(drop=True)], axis=1)

out_path = os.path.join(PROC_DIR, 'train_form.csv')
train_form.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("Total columns:", train_form.shape[1])

In [ ]:
# Pick a known short-rest or long-rest case
# Man City played Dec 31 2016, then Jan 2 2017 (2-day rest)
mask = (train['HomeTeam'] == 'Man City') & (train['Date'] == '2017-01-02')
if mask.any():
    idx = train.index[mask][0]
    print("Man City Jan 2, 2017 (should have short rest):")
    print(f"  home_rest_days = {train_form.loc[idx, 'home_rest_days']}")

# Season opener: Burnley's first match of 2017/18 (Aug 12, 2017)
mask = (train['HomeTeam'] == 'Chelsea') & (train['Date'] == '2017-08-12')
if mask.any():
    idx = train.index[mask][0]
    print("\nChelsea Aug 12, 2017 (season opener, should be capped at 14):")
    print(f"  home_rest_days = {train_form.loc[idx, 'home_rest_days']}")

# Verify cap: no value > 14
assert train_form['home_rest_days'].max() <= 14
assert train_form['away_rest_days'].max() <= 14
assert train_form['home_rest_days'].min() >= 0
print("\n✅ Rest-days in [0, 14]. Cap working.")

In [ ]:
elo_code = '''

# ============================================================
# 17. ELO RATINGS (added at Step 08F)
# ============================================================

def build_elo_features(matches_df: pd.DataFrame,
                       k_factor: float = 20.0,
                       home_bonus: float = 60.0,
                       initial_elo: float = 1500.0,
                       regression: float = 0.25) -> pd.DataFrame:
    """
    Compute pre-match Elo for home and away teams.

    Rules:
      - Each team starts at initial_elo (1500).
      - After each match, update both teams' Elo using the standard formula.
      - At end of each season, regress each team's Elo toward initial_elo
        by `regression` fraction.
      - Features for a match are the PRE-MATCH ratings, never post-match.

    Output columns (2): home_elo, away_elo
    """
    df = matches_df.copy().reset_index(drop=True).sort_values(
        ["Date"]).reset_index(drop=True)
    df["_row_id"] = df.index

    elo = {}
    last_season = None

    home_elos = [None] * len(df)
    away_elos = [None] * len(df)

    for i, row in df.iterrows():
        # If a new season is starting, regress everyone first
        if row["season_label"] != last_season:
            if last_season is not None:
                for t in list(elo.keys()):
                    elo[t] = elo[t] + regression * (initial_elo - elo[t])
            last_season = row["season_label"]

        home = row["HomeTeam"]; away = row["AwayTeam"]
        # Initialize if first time we see them
        if home not in elo: elo[home] = initial_elo
        if away not in elo: elo[away] = initial_elo

        elo_h = elo[home]; elo_a = elo[away]

        # Pre-match features
        home_elos[i] = elo_h
        away_elos[i] = elo_a

        # Expected score for home team
        E_h = 1.0 / (1.0 + 10.0 ** ((elo_a - elo_h - home_bonus) / 400.0))
        E_a = 1.0 - E_h

        # Actual score
        if row["FTHG"] > row["FTAG"]:
            S_h, S_a = 1.0, 0.0
        elif row["FTHG"] < row["FTAG"]:
            S_h, S_a = 0.0, 1.0
        else:
            S_h, S_a = 0.5, 0.5

        # Update
        elo[home] = elo_h + k_factor * (S_h - E_h)
        elo[away] = elo_a + k_factor * (S_a - E_a)

    out = pd.DataFrame(index=df.index)
    out["home_elo"] = home_elos
    out["away_elo"] = away_elos

    # Restore original order
    out = out.reindex(matches_df.index)
    return out


# ============================================================
# 18. PREVIOUS-SEASON PPG AS A DIRECT FEATURE (added at Step 08F)
# ============================================================

def build_prev_season_ppg_features(matches_df: pd.DataFrame,
                                   prev_form_venue_df: pd.DataFrame,
                                   promoted_baseline_venue: dict,
                                   promoted_table: dict) -> pd.DataFrame:
    """
    Direct feature: venue-specific previous-season PPG for both sides.

    For each match, look up the home team's previous-season home-venue PPG
    and the away team's previous-season away-venue PPG. Fall back to
    promoted baseline for promoted teams or teams without history.

    Output columns (2): home_prev_season_ppg, away_prev_season_ppg
    """
    df = matches_df.copy().reset_index(drop=True)
    prev_map = prev_form_venue_df.set_index(["team", "season_label"]).to_dict("index")

    def lookup(team, season, venue):
        prev = prev_map.get((team, season))
        if prev is not None and not pd.isna(prev[f"prev_season_ppg_{venue}"]):
            return prev[f"prev_season_ppg_{venue}"]
        return promoted_baseline_venue.get(f"promoted_baseline_ppg_{venue}", 1.0)

    out = pd.DataFrame(index=df.index)
    out["home_prev_season_ppg"] = df.apply(
        lambda r: lookup(r["HomeTeam"], r["season_label"], "home"), axis=1).values
    out["away_prev_season_ppg"] = df.apply(
        lambda r: lookup(r["AwayTeam"], r["season_label"], "away"), axis=1).values

    out.index = matches_df.index
    return out
'''

module_path = '/content/drive/MyDrive/epl_match_outcome/src/features.py'
with open(module_path, 'a') as f:
    f.write(elo_code)

import importlib, sys, os
SRC_DIR = '/content/drive/MyDrive/epl_match_outcome/src'
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
import features
importlib.reload(features)

print("Appended Elo and prev-season-PPG functions.")
print("Total public functions:", len([n for n in dir(features)
                                      if not n.startswith("_") and callable(getattr(features, n))]))

In [ ]:
import pandas as pd, numpy as np, os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])

PROMOTED_TABLE = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}
train_promoted = {k: v for k, v in PROMOTED_TABLE.items() if k in train['season_label'].unique()}

# Build Elo
elo_feats = features.build_elo_features(train)
print("Elo shape:", elo_feats.shape)
print(elo_feats.describe().round(2).to_string())

# Build prev-season PPG direct
prev_form_venue = features.compute_prev_season_form_venue(train)
promoted_base_v = features.compute_promoted_baseline_venue(train, train_promoted, early_gameweeks=5)
prev_ppg_feats  = features.build_prev_season_ppg_features(train, prev_form_venue, promoted_base_v, PROMOTED_TABLE)
print("\nPrev-season PPG shape:", prev_ppg_feats.shape)

# Merge
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
new_cols = list(elo_feats.columns) + list(prev_ppg_feats.columns)
train_form = train_form.drop(columns=[c for c in new_cols if c in train_form.columns], errors='ignore')
train_form = pd.concat([train_form.reset_index(drop=True),
                        elo_feats.reset_index(drop=True),
                        prev_ppg_feats.reset_index(drop=True)], axis=1)

nan_counts = train_form[new_cols].isna().sum()
print("\nNaN counts in new cols:")
print(nan_counts[nan_counts > 0].to_string() if (nan_counts > 0).any() else "(none) ✅")

out_path = os.path.join(PROC_DIR, 'train_form.csv')
train_form.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("Total columns:", train_form.shape[1])

In [ ]:
# Track Leicester's Elo across 2015/16 to 2016/17
leic_1516 = train[(train['season_label']=='2015/16') &
                  ((train['HomeTeam']=='Leicester') | (train['AwayTeam']=='Leicester'))].sort_values('Date')
leic_1617 = train[(train['season_label']=='2016/17') &
                  ((train['HomeTeam']=='Leicester') | (train['AwayTeam']=='Leicester'))].sort_values('Date')

print("Leicester Elo journey (first 3 of 2015/16, last 3 of 2015/16, first 3 of 2016/17):")
for label, subset in [("1516 first", leic_1516.head(3)), ("1516 last", leic_1516.tail(3)), ("1617 first", leic_1617.head(3))]:
    for _, row in subset.iterrows():
        idx = row.name
        side = 'home' if row['HomeTeam']=='Leicester' else 'away'
        elo_val = train_form.loc[idx, f'{side}_elo']
        print(f"  {label:10s} {row['Date'].date()} {row['HomeTeam']:15s} vs {row['AwayTeam']:15s}  Elo={elo_val:.1f}")

# Sanity: end of 2015/16 Leicester should be well above 1500 (they won the league)
leic_end_1516 = leic_1516.tail(1).index[0]
side = 'home' if train.loc[leic_end_1516,'HomeTeam']=='Leicester' else 'away'
post_elo = train_form.loc[leic_end_1516, f'{side}_elo']
print(f"\nLeicester Elo near end of 2015/16: {post_elo:.1f} (should be > 1600 after winning league)")

# Sanity: all ELOs should be in a reasonable range
assert train_form['home_elo'].between(1200, 2200).all()
assert train_form['away_elo'].between(1200, 2200).all()
print("\n✅ Elo values are in a reasonable range [1200, 2200].")

In [ ]:
# Diagnostic: check alignment between train and train_form for Leicester
leic_1516 = train[(train['season_label']=='2015/16') &
                  ((train['HomeTeam']=='Leicester') | (train['AwayTeam']=='Leicester'))].sort_values('Date')

print("Last 5 Leicester matches 2015/16, from train:")
last_5 = leic_1516.tail(5)
print(last_5[['Date','HomeTeam','AwayTeam','FTHG','FTAG']].to_string())

print("\nElo values at those same rows, from train_form:")
for idx, row in last_5.iterrows():
    side = 'home' if row['HomeTeam']=='Leicester' else 'away'
    # Direct .iloc lookup using row position
    pos = train.index.get_loc(idx)
    elo_val = train_form.iloc[pos][f'{side}_elo']
    print(f"  {row['Date'].date()} {row['HomeTeam']} vs {row['AwayTeam']}  → Leicester {side}_elo = {elo_val:.1f}")

# Also print the row from train_form directly for the target row
print("\nDirect check: train_form row for 2016-05-07 Leicester vs Everton")
mask = (train['Date']=='2016-05-07') & (train['HomeTeam']=='Leicester') & (train['AwayTeam']=='Everton')
if mask.any():
    idx2 = train.index[mask][0]
    print(f"train index: {idx2}")
    print(f"train_form home_elo: {train_form.loc[idx2, 'home_elo']:.1f}")
    print(f"train_form away_elo: {train_form.loc[idx2, 'away_elo']:.1f}")

In [ ]:
# Diagnostic: trace Leicester's Elo step-by-step through 2015/16
leic = train[(train['season_label']=='2015/16') &
             ((train['HomeTeam']=='Leicester') | (train['AwayTeam']=='Leicester'))].sort_values('Date')

print("Leicester Elo trace through 2015/16 (should always move by ≤20 per match, and up on wins):")
for idx, row in leic.iterrows():
    side = 'home' if row['HomeTeam']=='Leicester' else 'away'
    elo_val = train_form.loc[idx, f'{side}_elo']
    result = row['FTR']
    leicester_result = 'W' if (side=='home' and result=='H') or (side=='away' and result=='A') \
                       else ('D' if result=='D' else 'L')
    print(f"{row['Date'].date()} {row['HomeTeam']:15s} {row['FTHG']}-{row['FTAG']} {row['AwayTeam']:15s}  {leicester_result}  Elo={elo_val:.1f}")

In [ ]:
import os, re

SRC_DIR = '/content/drive/MyDrive/epl_match_outcome/src'
module_path = os.path.join(SRC_DIR, 'features.py')

with open(module_path, 'r') as f:
    src = f.read()

# New version of the function
new_elo = '''def build_elo_features(matches_df: pd.DataFrame,
                       k_factor: float = 20.0,
                       home_bonus: float = 60.0,
                       initial_elo: float = 1500.0,
                       regression: float = 0.25) -> pd.DataFrame:
    """
    Compute pre-match Elo for home and away teams.

    Rewritten to use purely positional iteration over a date-sorted copy,
    with no ambiguous index alignment.
    """
    df = matches_df.copy().reset_index(drop=True).sort_values(
        ["season_label", "Date"]).reset_index(drop=True)

    elo = {}
    last_season = None
    home_elos = [None] * len(df)
    away_elos = [None] * len(df)

    for pos in range(len(df)):
        row = df.iloc[pos]
        season = row["season_label"]
        home = row["HomeTeam"]
        away = row["AwayTeam"]

        # Season boundary: regress every team once
        if season != last_season:
            if last_season is not None:
                for t in list(elo.keys()):
                    elo[t] = elo[t] + regression * (initial_elo - elo[t])
            last_season = season

        if home not in elo: elo[home] = initial_elo
        if away not in elo: elo[away] = initial_elo

        elo_h = elo[home]
        elo_a = elo[away]

        # Store PRE-match Elo
        home_elos[pos] = elo_h
        away_elos[pos] = elo_a

        # Update
        E_h = 1.0 / (1.0 + 10.0 ** ((elo_a - elo_h - home_bonus) / 400.0))
        E_a = 1.0 - E_h
        if row["FTHG"] > row["FTAG"]:
            S_h, S_a = 1.0, 0.0
        elif row["FTHG"] < row["FTAG"]:
            S_h, S_a = 0.0, 1.0
        else:
            S_h, S_a = 0.5, 0.5

        elo[home] = elo_h + k_factor * (S_h - E_h)
        elo[away] = elo_a + k_factor * (S_a - E_a)

    out = pd.DataFrame({"home_elo": home_elos, "away_elo": away_elos},
                       index=df.index)
    # Map back to matches_df original row order
    out = out.reindex(matches_df.index)
    return out
'''

# Find the old build_elo_features block and replace it
# Pattern: from "def build_elo_features" up to the next "\ndef " (the next function)
pattern = re.compile(
    r'def build_elo_features\(.*?\n(?=\n# =+|\ndef |\nclass |\Z)',
    re.DOTALL
)

matches = pattern.findall(src)
print(f"Found {len(matches)} block(s) matching build_elo_features.")

if matches:
    src = pattern.sub(new_elo + "\n", src, count=1)
    with open(module_path, 'w') as f:
        f.write(src)
    print("Replaced build_elo_features.")
    print("New file size:", os.path.getsize(module_path))
else:
    print("Pattern not found. Let's inspect the file manually.")
    # Fallback: just find where the function starts and ends
    start = src.find("def build_elo_features")
    if start == -1:
        print("'def build_elo_features' not found at all.")
    else:
        print(f"'def build_elo_features' found at char {start}")
        print("Nearby context:")
        print(src[start:start+300])

In [ ]:
import sys, os, importlib
import pandas as pd
import numpy as np

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import features
importlib.reload(features)

train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])

# Recompute Elo
elo_feats = features.build_elo_features(train)
print("New Elo describe:")
print(elo_feats.describe().round(2).to_string())

# Rebuild prev-season PPG (unchanged)
PROMOTED_TABLE = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}
train_promoted = {k: v for k, v in PROMOTED_TABLE.items() if k in train['season_label'].unique()}
prev_form_venue = features.compute_prev_season_form_venue(train)
promoted_base_v = features.compute_promoted_baseline_venue(train, train_promoted, early_gameweeks=5)
prev_ppg_feats  = features.build_prev_season_ppg_features(train, prev_form_venue, promoted_base_v, PROMOTED_TABLE)

# Merge (drop old elo columns first, then re-add)
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
new_cols = ['home_elo', 'away_elo', 'home_prev_season_ppg', 'away_prev_season_ppg']
train_form = train_form.drop(columns=[c for c in new_cols if c in train_form.columns], errors='ignore')

train_form = pd.concat([
    train_form.reset_index(drop=True),
    elo_feats.reset_index(drop=True),
    prev_ppg_feats.reset_index(drop=True),
], axis=1)

out_path = os.path.join(PROC_DIR, 'train_form.csv')
train_form.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("Total columns:", train_form.shape[1])

In [ ]:
leic = train[(train['season_label']=='2015/16') &
             ((train['HomeTeam']=='Leicester') | (train['AwayTeam']=='Leicester'))].sort_values('Date')

print("Leicester Elo trace through 2015/16 (REWRITTEN Elo):")
print("Rule check: Elo should RISE on wins, FALL on losses, move by ≤20 per match.\n")

prev_elo = 1500.0
violations = 0
for idx, row in leic.iterrows():
    side = 'home' if row['HomeTeam']=='Leicester' else 'away'
    elo_val = train_form.loc[idx, f'{side}_elo']
    result = row['FTR']
    leicester_result = 'W' if ((side=='home' and result=='H') or (side=='away' and result=='A')) \
                       else ('D' if result=='D' else 'L')

    delta = elo_val - prev_elo
    flag = ""
    if pos := None:  # placeholder
        pass

    # Sanity checks
    if leicester_result == 'W' and delta < -0.01: flag = " ❌ WIN but Elo fell"
    if leicester_result == 'L' and delta > 0.01: flag = " ❌ LOSS but Elo rose"
    if abs(delta) > 21: flag = f" ❌ move >20 ({delta:+.1f})"

    if flag: violations += 1

    print(f"{row['Date'].date()} {row['HomeTeam']:15s} {row['FTHG']}-{row['FTAG']} {row['AwayTeam']:15s}  "
          f"{leicester_result}  Elo={elo_val:7.2f}  Δ={delta:+6.2f}{flag}")
    prev_elo = elo_val

print(f"\nViolations found: {violations}")
if violations == 0:
    print("✅ Elo trajectory is now correct.")
else:
    print("❌ Still broken. Send this trace back.")

In [ ]:
import pandas as pd, os
PROC_DIR = '/content/drive/MyDrive/epl_match_outcome/data/processed'
train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])

leic = train[(train['season_label']=='2015/16') &
             ((train['HomeTeam']=='Leicester') | (train['AwayTeam']=='Leicester'))].sort_values('Date')

print("Leicester Elo trace through 2015/16 — CORRECTED check")
print("(delta = pre_elo[i] - pre_elo[i-1]; this reflects RESULT OF MATCH i-1)\n")

prev_elo = 1500.0
prev_res = None
violations = 0
for idx, row in leic.iterrows():
    side = 'home' if row['HomeTeam']=='Leicester' else 'away'
    elo_val = train_form.loc[idx, f'{side}_elo']
    result = row['FTR']
    leic_res = 'W' if ((side=='home' and result=='H') or (side=='away' and result=='A')) \
               else ('D' if result=='D' else 'L')
    delta = elo_val - prev_elo
    check = ""
    if prev_res == 'W' and delta < -0.01: check = "❌ prev W but fell"; violations += 1
    if prev_res == 'L' and delta > 0.01: check = "❌ prev L but rose"; violations += 1
    if abs(delta) > 21: check = f"❌ move>20"; violations += 1

    print(f"{row['Date'].date()}  {row['HomeTeam']:12s} {row['FTHG']}-{row['FTAG']} {row['AwayTeam']:12s}  "
          f"{leic_res}  Elo={elo_val:8.2f}  Δ={delta:+6.2f}  prev_res={str(prev_res):4s}  {check}")
    prev_elo = elo_val
    prev_res = leic_res

print(f"\nViolations: {violations}")

In [ ]:
deltas_code = '''

# ============================================================
# 19. MATCHUP DELTAS (added at Step 08G)
# ============================================================

DELTA_SPECS = [
    ("delta_elo",                "home_elo",                "away_elo"),
    ("delta_form_pts_last5",     "home_form_pts_last5",     "away_form_pts_last5"),
    ("delta_form_gd_last5",      "home_form_gd_last5",      "away_form_gd_last5"),
    ("delta_venue_pts_last5",    "home_venue_pts_last5",    "away_venue_pts_last5"),
    ("delta_season_ppg",         "home_season_ppg",         "away_season_ppg"),
    ("delta_prev_season_ppg",    "home_prev_season_ppg",    "away_prev_season_ppg"),
    ("delta_form_shots_last5",   "home_form_shots_last5",   "away_form_shots_last5"),
    ("delta_form_sot_last5",     "home_form_sot_last5",     "away_form_sot_last5"),
    ("delta_rest_days",          "home_rest_days",          "away_rest_days"),
]


def build_deltas(feature_df: pd.DataFrame) -> pd.DataFrame:
    """
    Given a DataFrame that already contains home_* and away_* columns,
    compute the derived delta columns.

    delta_X = home_X - away_X

    Returns a DataFrame with only the delta columns, indexed like the input.
    """
    out = pd.DataFrame(index=feature_df.index)
    for name, home_col, away_col in DELTA_SPECS:
        assert home_col in feature_df.columns, f"Missing {home_col}"
        assert away_col in feature_df.columns, f"Missing {away_col}"
        out[name] = feature_df[home_col].values - feature_df[away_col].values
    return out
'''

module_path = '/content/drive/MyDrive/epl_match_outcome/src/features.py'
with open(module_path, 'a') as f:
    f.write(deltas_code)

import importlib, sys, os
SRC_DIR = '/content/drive/MyDrive/epl_match_outcome/src'
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
import features
importlib.reload(features)

print("Appended build_deltas.")
print("Total public functions:", len([n for n in dir(features)
                                      if not n.startswith("_") and callable(getattr(features, n))]))

In [ ]:
import pandas as pd, numpy as np, os

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])

deltas = features.build_deltas(train_form)
print("Delta shape:", deltas.shape)
print("Delta columns:", list(deltas.columns))
print()
print(deltas.describe().round(3).to_string())

nan_counts = deltas.isna().sum()
print("\nNaN counts:")
print(nan_counts[nan_counts > 0].to_string() if (nan_counts > 0).any() else "(none) ✅")

# Merge (drop old delta columns first)
train_form = train_form.drop(columns=[c for c in deltas.columns if c in train_form.columns], errors='ignore')
train_form = pd.concat([train_form.reset_index(drop=True),
                        deltas.reset_index(drop=True)], axis=1)

out_path = os.path.join(PROC_DIR, 'train_form.csv')
train_form.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("Total columns:", train_form.shape[1])

In [ ]:
# Take a match where Man City at home plays Southampton (or anyone weaker)
# Their delta_elo should be clearly positive.
mask = (train['HomeTeam'] == 'Man City') & (train['season_label'] == '2017/18')
if mask.any():
    idx = train[mask].sort_values('Date').index[10]  # 11th home match of season
    row = train_form.loc[idx]
    print(f"Match: {train.loc[idx,'HomeTeam']} vs {train.loc[idx,'AwayTeam']} on {train.loc[idx,'Date'].date()}")
    print()
    print("Home team metrics:")
    print(f"  elo                     = {row['home_elo']:.1f}")
    print(f"  form_pts_last5          = {row['home_form_pts_last5']:.2f}")
    print(f"  season_ppg              = {row['home_season_ppg']:.2f}")
    print()
    print("Away team metrics:")
    print(f"  elo                     = {row['away_elo']:.1f}")
    print(f"  form_pts_last5          = {row['away_form_pts_last5']:.2f}")
    print(f"  season_ppg              = {row['away_season_ppg']:.2f}")
    print()
    print("Deltas (home - away):")
    print(f"  delta_elo               = {row['delta_elo']:+.1f}")
    print(f"  delta_form_pts_last5    = {row['delta_form_pts_last5']:+.2f}")
    print(f"  delta_season_ppg        = {row['delta_season_ppg']:+.2f}")

    # Sanity: this should be a strongly positive delta_elo
    assert row['delta_elo'] > 0
    print("\n✅ Delta computation consistent (Man City favored at home → positive delta_elo).")

In [ ]:
# ============================================================
# STEP 07H — CELL 1 (FIXED)
# Add FTR_int, build val features safely on train+val combined.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys
import pandas as pd
import numpy as np

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import importlib
import features
importlib.reload(features)

TARGET_MAP = {"A": 0, "D": 1, "H": 2}
ALL_FEATURE_PREFIXES = ('home_', 'away_', 'delta_')

# ---------- Add FTR_int to train_form.csv ----------
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
train_form['FTR_int'] = train_form['FTR'].map(TARGET_MAP)
assert train_form['FTR_int'].notna().all(), "FTR_int has NaN."
train_form.to_csv(os.path.join(PROC_DIR, 'train_form.csv'), index=False)
print("Updated train_form.csv. Shape:", train_form.shape)
print("FTR distribution:", train_form['FTR_int'].value_counts().sort_index().to_dict())


# ---------- Build val features ----------
val = pd.read_csv(os.path.join(PROC_DIR, 'val.csv'), parse_dates=['Date'])
print("\nVal shape:", val.shape)

# Combine train_form + val
combined = pd.concat(
    [train_form.drop(columns=['FTR_int'], errors='ignore'), val],
    ignore_index=True
)

# *** THE FIX: drop pre-existing feature columns from combined ***
pre_existing = [c for c in combined.columns if c.startswith(ALL_FEATURE_PREFIXES)]
print(f"\nDropping {len(pre_existing)} pre-existing feature columns from combined.")
combined = combined.drop(columns=pre_existing)
combined = combined.sort_values('Date').reset_index(drop=True)
print("Combined shape after drop:", combined.shape)


# ---------- Rebuild all features on the combined set ----------
print("\nRebuilding features on train+val combined...")

PROMOTED_TABLE = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}
promoted_for_combined = {k: v for k, v in PROMOTED_TABLE.items()
                        if k in combined['season_label'].unique()}

roll             = features.build_rolling_form_filled(combined,
                                                      features.compute_prev_season_form(combined),
                                                      features.compute_promoted_baseline(combined, promoted_for_combined, 5),
                                                      PROMOTED_TABLE, n=5)
prev_form_v      = features.compute_prev_season_form_venue(combined)
promoted_base_v  = features.compute_promoted_baseline_venue(combined, promoted_for_combined, 5)
venue            = features.build_venue_form_filled(combined, prev_form_v, promoted_base_v, PROMOTED_TABLE, n=5)
prev_stats_v     = features.compute_prev_season_match_stats_venue(combined)
promoted_base_ms = features.compute_promoted_baseline_match_stats(combined, promoted_for_combined, 5)
stats            = features.build_match_stats_filled(combined, prev_stats_v, promoted_base_ms, PROMOTED_TABLE, n=5)
prev_gd          = features.compute_prev_season_gd_per_game_venue(combined)
std              = features.build_season_to_date_filled(combined, prev_gd, promoted_base_v, PROMOTED_TABLE)
rest             = features.build_rest_days(combined, cap=14)
elo              = features.build_elo_features(combined)
prev_ppg         = features.build_prev_season_ppg_features(combined, prev_form_v, promoted_base_v, PROMOTED_TABLE)

# Combine feature blocks
feat_blocks = [roll, venue, stats, std, rest, elo, prev_ppg]
combined_feats = pd.concat([b.reset_index(drop=True) for b in feat_blocks], axis=1)

# Sanity: no duplicate feature columns
assert not combined_feats.columns.duplicated().any(), \
    f"Duplicate columns in combined_feats: {combined_feats.columns[combined_feats.columns.duplicated()].tolist()}"

# Assemble full combined_feats: raw columns + feature columns
combined_feats = pd.concat([combined.reset_index(drop=True), combined_feats], axis=1)

# Build deltas
deltas = features.build_deltas(combined_feats)
combined_feats = pd.concat([combined_feats.reset_index(drop=True), deltas.reset_index(drop=True)], axis=1)

# Encode target
combined_feats['FTR_int'] = combined_feats['FTR'].map(TARGET_MAP)

# Slice back to val
val_form = combined_feats[combined_feats['season_label'].isin(['2023/24','2024/25'])].reset_index(drop=True)
val_form.to_csv(os.path.join(PROC_DIR, 'val_form.csv'), index=False)
print("\nSaved val_form.csv:", val_form.shape)

# NaN check
val_feat_cols = [c for c in val_form.columns if c.startswith(('home_','away_','delta_'))]
nan_total = val_form[val_feat_cols].isna().sum().sum()
print("NaN in val features:", nan_total)
assert nan_total == 0, f"Val has {nan_total} NaN."
print("✅ Val features built with no NaN.")

In [ ]:
pipeline_py = '''"""
pipeline.py — sklearn Pipeline for EPL match outcome prediction.

Design (Approach B):
  - Feature engineering is done separately (per split) via features.py.
  - This module handles only:
      * feature column selection
      * optional scaling
      * model training/prediction
  - The Pipeline is estimator-agnostic: swap the final estimator for
    LogisticRegression, XGBoost, RandomForest, etc.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier


MODEL_FEATURE_PREFIXES = ("home_", "away_", "delta_")
EXCLUDED_FEATURES = set()


def get_feature_columns(df: pd.DataFrame) -> list:
    cols = [c for c in df.columns
            if c.startswith(MODEL_FEATURE_PREFIXES) and c not in EXCLUDED_FEATURES]
    return cols


class OptionalScaler:
    def __init__(self, scale: bool = True):
        self.scale = scale
        self.scaler_ = StandardScaler() if scale else None

    def fit(self, X, y=None):
        if self.scale:
            self.scaler_.fit(X)
        return self

    def transform(self, X):
        if self.scale:
            return self.scaler_.transform(X)
        return X

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)

    def get_feature_names_out(self, input_features=None):
        if self.scale and input_features is not None:
            return np.asarray(input_features)
        return input_features


def build_pipeline(estimator=None, scale: bool = True) -> Pipeline:
    if estimator is None:
        estimator = DummyClassifier(strategy="most_frequent")
    return Pipeline([
        ("scaler", OptionalScaler(scale=scale)),
        ("model",  estimator),
    ])


def split_X_y(df: pd.DataFrame):
    feat_cols = get_feature_columns(df)
    X = df[feat_cols].copy()
    y = df["FTR_int"].copy() if "FTR_int" in df.columns else None
    return X, y
'''

module_path = os.path.join(SRC_DIR, 'pipeline.py')
with open(module_path, 'w') as f:
    f.write(pipeline_py)
print("Wrote:", module_path)
print("Size:", os.path.getsize(module_path))

In [ ]:
import importlib
import pipeline as pl
importlib.reload(pl)

from sklearn.metrics import accuracy_score, log_loss

train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
val_form   = pd.read_csv(os.path.join(PROC_DIR, 'val_form.csv'),   parse_dates=['Date'])

print("Train:", train_form.shape, "Val:", val_form.shape)

feat_cols = pl.get_feature_columns(train_form)
print(f"\nNumber of model features: {len(feat_cols)}")
print("First 10:", feat_cols[:10])
print("Last 10 :", feat_cols[-10:])

# Sanity: no leakage columns in feature list
leakage_cols = ['FTHG','FTAG','FTR','HTHG','HTAG','HTR','HS','AS','HST','AST',
                'HF','AF','HC','AC','HY','AY','HR','AR','Referee','Date',
                'HomeTeam','AwayTeam','season','season_label','FTR_int']
for c in leakage_cols:
    assert c not in feat_cols, f"LEAKAGE: {c} is in feature list!"
print("✅ No leakage columns in the feature list.")

X_train, y_train = pl.split_X_y(train_form)
X_val,   y_val   = pl.split_X_y(val_form)

pipe = pl.build_pipeline()  # DummyClassifier

pipe.fit(X_train, y_train)

pred_train = pipe.predict(X_train)
pred_val   = pipe.predict(X_val)
proba_val  = pipe.predict_proba(X_val)

print("\nPipeline ran end-to-end.")
print(f"Train accuracy (dummy): {accuracy_score(y_train, pred_train):.3f}")
print(f"Val   accuracy (dummy): {accuracy_score(y_val,   pred_val):.3f}")
print(f"Val   log loss (dummy): {log_loss(y_val, proba_val, labels=[0,1,2]):.4f}")
print("\nThese dummy numbers are our naive floor. Real models must beat them.")

In [ ]:
# Verify proba alignment + quick comparison of dummies
import importlib
import pipeline as pl
importlib.reload(pl)

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, log_loss

train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
val_form   = pd.read_csv(os.path.join(PROC_DIR, 'val_form.csv'),   parse_dates=['Date'])

X_train, y_train = pl.split_X_y(train_form)
X_val,   y_val   = pl.split_X_y(val_form)

# Fit dummy 'most_frequent'
pipe_mf = pl.build_pipeline(DummyClassifier(strategy='most_frequent'))
pipe_mf.fit(X_train, y_train)
proba_mf = pipe_mf.predict_proba(X_val)
print("classes_:", pipe_mf.classes_)
print("proba shape:", proba_mf.shape)
print("proba[0]:", proba_mf[0])
print("log loss (most_frequent):", log_loss(y_val, proba_mf, labels=pipe_mf.classes_))

# Fit dummy 'prior'
pipe_prior = pl.build_pipeline(DummyClassifier(strategy='prior'))
pipe_prior.fit(X_train, y_train)
proba_prior = pipe_prior.predict_proba(X_val)
print("\nproba_prior[0]:", proba_prior[0])
print("log loss (prior):", log_loss(y_val, proba_prior, labels=pipe_prior.classes_))

In [ ]:
import os, re

SRC_DIR = '/content/drive/MyDrive/epl_match_outcome/src'
module_path = os.path.join(SRC_DIR, 'pipeline.py')

with open(module_path, 'r') as f:
    src = f.read()

# Change the default
old = 'estimator = DummyClassifier(strategy="most_frequent")'
new = 'estimator = DummyClassifier(strategy="prior")  # class-frequency placeholder'

if old in src:
    src = src.replace(old, new, 1)
    with open(module_path, 'w') as f:
        f.write(src)
    print("✅ Updated default placeholder to strategy='prior'.")
else:
    print("⚠️  Pattern not found — the file may already be updated or wording changed.")
    print("Current line:", [l for l in src.splitlines() if 'DummyClassifier' in l])

In [ ]:
validation_py = '''"""
validation.py — time-aware CV splitting for the EPL outcome project.

Design:
  - SeasonSplit: expanding-window CV where each validation fold is exactly
    one complete season.
  - Fold i: train on seasons [0..i-1], validate on season [i].
  - No shuffling. No random splits. No future leakage.

Used by:
  - Steps 10-14 (model selection, training, tuning)
  - Step 15 (fixed-val final comparison uses the held-out val set, not this)
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from typing import Iterator, Tuple


class SeasonSplit:
    """
    Expanding-window time-series CV with one full season per validation fold.

    Parameters
    ----------
    season_labels : pd.Series
        Season label for each row, aligned with the data being split.
        Example values: '2015/16', '2016/17', ...
    min_train_seasons : int, default 1
        Minimum number of seasons required in the training set before the
        first fold can be generated.

    Yields
    ------
    (train_idx, val_idx) : tuple of np.ndarray
        Positional indices into the input data.
    """
    def __init__(self, season_labels: pd.Series, min_train_seasons: int = 1):
        self.season_labels = pd.Series(season_labels).reset_index(drop=True)
        self.min_train_seasons = min_train_seasons

        # Ordered unique seasons (they're already lexicographically sorted
        # for our naming convention, but sort explicitly to be safe)
        self.seasons_ = sorted(self.season_labels.unique())
        self.n_splits = max(0, len(self.seasons_) - min_train_seasons)

    def __iter__(self) -> Iterator[Tuple[np.ndarray, np.ndarray]]:
        for i in range(self.min_train_seasons, len(self.seasons_)):
            train_seasons = self.seasons_[:i]
            val_season    = self.seasons_[i]

            train_mask = self.season_labels.isin(train_seasons).values
            val_mask   = (self.season_labels == val_season).values

            train_idx = np.where(train_mask)[0]
            val_idx   = np.where(val_mask)[0]

            yield train_idx, val_idx

    def get_n_splits(self, X=None, y=None, groups=None) -> int:
        return self.n_splits


def assert_season_split_valid(season_labels: pd.Series,
                              train_idx: np.ndarray,
                              val_idx: np.ndarray) -> None:
    """
    Verify a single (train, val) split respects season boundaries:
      - Train seasons are all strictly earlier than val seasons.
      - No season appears in both.
    """
    s = pd.Series(season_labels).reset_index(drop=True)
    train_seasons = set(s.iloc[train_idx].unique())
    val_seasons   = set(s.iloc[val_idx].unique())

    assert len(val_seasons) == 1, f"Val fold spans multiple seasons: {val_seasons}"
    assert train_seasons.isdisjoint(val_seasons), \\
        f"Train and val seasons overlap: {train_seasons & val_seasons}"
    assert max(train_seasons) < min(val_seasons), \\
        f"Train seasons are not all earlier than val: train={train_seasons}, val={val_seasons}"


print("validation.py module loaded.")
'''

module_path = '/content/drive/MyDrive/epl_match_outcome/src/validation.py'
with open(module_path, 'w') as f:
    f.write(validation_py)

import os
print("Wrote:", module_path)
print("Size:", os.path.getsize(module_path))

In [ ]:
import sys, importlib
import pandas as pd
import numpy as np

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation
importlib.reload(validation)

# Load train
train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])
print("Train shape:", train.shape)
print("Seasons:", sorted(train['season_label'].unique()))

# Build splitter
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)
print(f"\nNumber of folds: {splitter.get_n_splits()}")

# Generate and verify each fold
print("\n=== Fold summary ===")
print(f"{'Fold':<5} {'Val season':<12} {'Train rows':<12} {'Val rows':<10} {'Train seasons':<55}")
print("-" * 100)

folds = []
for i, (tr_idx, va_idx) in enumerate(splitter):
    validation.assert_season_split_valid(train['season_label'], tr_idx, va_idx)
    val_season   = train['season_label'].iloc[va_idx].unique()[0]
    train_seasons = sorted(train['season_label'].iloc[tr_idx].unique())
    folds.append((tr_idx, va_idx, val_season, train_seasons))

    print(f"{i+1:<5} {val_season:<12} {len(tr_idx):<12} {len(va_idx):<10} {str(train_seasons):<55}")

print(f"\nTotal folds generated: {len(folds)}")
assert len(folds) == 7, f"Expected 7 folds, got {len(folds)}"

# Each val fold should be exactly 380 rows (one full season)
for _, va_idx, val_season, _ in folds:
    assert len(va_idx) == 380, f"Fold {val_season} has {len(va_idx)} val rows, expected 380"

# Fold sizes are expanding
train_sizes = [len(tr_idx) for tr_idx, _, _, _ in folds]
assert train_sizes == sorted(train_sizes), "Train sizes are not monotonically increasing"
print("✅ All folds respect season boundaries. Train sizes expand monotonically.")
print(f"   Train sizes: {train_sizes}")

In [ ]:
# Show the fold structure as a grid for visual verification
fold_df = pd.DataFrame({
    'season': sorted(train['season_label'].unique()),
})

for i, (tr_idx, va_idx, val_season, train_seasons) in enumerate(folds):
    fold_df[f'fold_{i+1}'] = fold_df['season'].apply(
        lambda s: 'val' if s == val_season else ('train' if s in train_seasons else '-')
    )

print("Fold layout (each column = one fold):")
print(fold_df.to_string(index=False))

In [ ]:
manifest_lines = [
    "# Validation Strategy — EPL Match Outcome Project",
    "",
    "## Summary",
    "",
    "- **Main metric:** log loss (proper scoring rule).",
    "- **Development CV:** expanding-window time-series, one full season per fold.",
    "- **Number of folds:** 7 (train spans 2015/16 – 2022/23 = 8 seasons).",
    "- **Fixed validation set:** 2023/24 + 2024/25 (760 matches, untouched until Step 15).",
    "- **Locked test set:** 2025/26 (380 matches, opened once at Step 17).",
    "",
    "## Fold structure",
    "",
    "| Fold | Validation season | Train rows | Val rows |",
    "| --- | --- | --- | --- |",
]
for i, (tr_idx, va_idx, val_season, _) in enumerate(folds):
    manifest_lines.append(f"| {i+1} | {val_season} | {len(tr_idx)} | {len(va_idx)} |")

manifest_lines += [
    "",
    "## Rules (non-negotiable)",
    "",
    "1. Folds respect season boundaries — every val fold is a complete season.",
    "2. Expanding window — folds train on increasingly more seasons.",
    "3. No shuffling, no stratified K-fold, no random row splits.",
    "4. The fixed validation set (2023/24 – 2024/25) is NOT used during",
    "   Steps 10-14 (model selection, training, tuning). It is opened once",
    "   at Step 15 for final model comparison.",
    "5. The test set (2025/26) is opened once at Step 17.",
    "",
    "## Implementation",
    "",
    "- Class: `src/validation.py::SeasonSplit`",
    "- Verifier: `src/validation.py::assert_season_split_valid`",
    "- All model selection in Steps 10-14 uses this splitter on the train set.",
]

manifest_path = os.path.join(DOCS_DIR, 'validation_manifest.md')
with open(manifest_path, 'w') as f:
    f.write("\n".join(manifest_lines))
print("Wrote:", manifest_path)
print("Size:", os.path.getsize(manifest_path))

In [ ]:
baselines_py = '''"""
baselines.py — reference models for EPL outcome prediction.

Every serious model (LogisticRegression, XGBoost, ...) must beat these
on validation log loss. If it doesn't, we haven't learned anything useful.

All classes are sklearn-compatible: they expose fit(X, y) and
predict / predict_proba.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression


# ============================================================
# 9.5A — Most frequent baseline
# ============================================================

class MostFrequentBaseline(BaseEstimator, ClassifierMixin):
    """Always predict the majority class with probability 1.0."""
    def fit(self, X, y):
        y = np.asarray(y)
        vals, counts = np.unique(y, return_counts=True)
        self.majority_class_ = int(vals[np.argmax(counts)])
        self.classes_ = np.array([0, 1, 2])
        return self

    def predict(self, X):
        return np.full(len(X), self.majority_class_, dtype=int)

    def predict_proba(self, X):
        p = np.zeros((len(X), 3), dtype=float)
        p[:, self.majority_class_] = 1.0
        return p


# ============================================================
# 9.5B — Prior baseline (class frequencies)
# ============================================================

class PriorBaseline(BaseEstimator, ClassifierMixin):
    """Always predict the training-set class frequencies."""
    def fit(self, X, y):
        y = np.asarray(y)
        vals, counts = np.unique(y, return_counts=True)
        priors = counts / counts.sum()
        self.classes_ = np.array([0, 1, 2])
        self.prior_proba_ = np.zeros(3)
        for v, p in zip(vals, priors):
            self.prior_proba_[int(v)] = p
        return self

    def predict(self, X):
        return np.full(len(X), int(np.argmax(self.prior_proba_)), dtype=int)

    def predict_proba(self, X):
        return np.tile(self.prior_proba_, (len(X), 1))


# ============================================================
# 9.5D — PPG rule baseline
# ============================================================

class PPGRuleBaseline(BaseEstimator, ClassifierMixin):
    """
    Predict H/D/A probabilities using season-to-date PPG difference.

    Approach:
      - sigmoid(k * (ppg_home - ppg_away)) → P(H) vs P(A) split
      - a fixed draw probability DRW applied around the split

    Parameters
    ----------
    k : float, default 2.0
        Slope of the sigmoid.
    draw_frac : float, default 0.24
        Fixed probability mass allocated to draws.
    home_col : str, default 'home_season_ppg'
    away_col : str, default 'away_season_ppg'
    """
    def __init__(self, k: float = 2.0, draw_frac: float = 0.24,
                 home_col: str = 'home_season_ppg',
                 away_col: str = 'away_season_ppg'):
        self.k = k
        self.draw_frac = draw_frac
        self.home_col = home_col
        self.away_col = away_col

    def fit(self, X, y):
        self.classes_ = np.array([0, 1, 2])
        # Optionally could fit k and draw_frac on train; here they're fixed.
        # We could refine later, but this is the baseline's simplicity by design.
        return self

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            ppg_h = X[self.home_col].values
            ppg_a = X[self.away_col].values
        else:
            raise ValueError("PPGRuleBaseline expects a DataFrame with named columns")

        diff = ppg_h - ppg_a
        # Sigmoid over PPG gap → gives (H vs A) split
        s = 1.0 / (1.0 + np.exp(-self.k * diff))
        d = self.draw_frac
        h = (1.0 - d) * s
        a = (1.0 - d) * (1.0 - s)

        out = np.zeros((len(X), 3))
        out[:, 0] = a
        out[:, 1] = d
        out[:, 2] = h
        return out

    def predict(self, X):
        p = self.predict_proba(X)
        return p.argmax(axis=1)


# ============================================================
# 9.5E — Elo baseline
# ============================================================

class EloBaseline(BaseEstimator, ClassifierMixin):
    """
    Predict H/D/A probabilities from Elo difference using a small
    multinomial logistic regression.

    Parameters
    ----------
    feature_cols : list of str, default ['delta_elo']
        Columns to feed into the small logistic regression.
    """
    def __init__(self, feature_cols=None):
        self.feature_cols = feature_cols or ['delta_elo']

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            X_sub = X[self.feature_cols].values
        else:
            raise ValueError("EloBaseline expects a DataFrame with named columns")

        self.model_ = LogisticRegression(
            multi_class='multinomial', max_iter=1000, random_state=42
        )
        self.model_.fit(X_sub, y)
        self.classes_ = self.model_.classes_
        return self

    def predict_proba(self, X):
        X_sub = X[self.feature_cols].values
        return self.model_.predict_proba(X_sub)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


print("baselines.py module loaded.")
'''

module_path = '/content/drive/MyDrive/epl_match_outcome/src/baselines.py'
with open(module_path, 'w') as f:
    f.write(baselines_py)

import os
print("Wrote:", module_path)
print("Size:", os.path.getsize(module_path))

In [ ]:
baselines_py = '''"""
baselines.py — reference models for EPL outcome prediction.

Every serious model (LogisticRegression, XGBoost, ...) must beat these
on validation log loss. If it doesn't, we haven't learned anything useful.

All classes are sklearn-compatible: they expose fit(X, y) and
predict / predict_proba.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression


# ============================================================
# 9.5A — Most frequent baseline
# ============================================================

class MostFrequentBaseline(BaseEstimator, ClassifierMixin):
    """Always predict the majority class with probability 1.0."""
    def fit(self, X, y):
        y = np.asarray(y)
        vals, counts = np.unique(y, return_counts=True)
        self.majority_class_ = int(vals[np.argmax(counts)])
        self.classes_ = np.array([0, 1, 2])
        return self

    def predict(self, X):
        return np.full(len(X), self.majority_class_, dtype=int)

    def predict_proba(self, X):
        p = np.zeros((len(X), 3), dtype=float)
        p[:, self.majority_class_] = 1.0
        return p


# ============================================================
# 9.5B — Prior baseline (class frequencies)
# ============================================================

class PriorBaseline(BaseEstimator, ClassifierMixin):
    """Always predict the training-set class frequencies."""
    def fit(self, X, y):
        y = np.asarray(y)
        vals, counts = np.unique(y, return_counts=True)
        priors = counts / counts.sum()
        self.classes_ = np.array([0, 1, 2])
        self.prior_proba_ = np.zeros(3)
        for v, p in zip(vals, priors):
            self.prior_proba_[int(v)] = p
        return self

    def predict(self, X):
        return np.full(len(X), int(np.argmax(self.prior_proba_)), dtype=int)

    def predict_proba(self, X):
        return np.tile(self.prior_proba_, (len(X), 1))


# ============================================================
# 9.5D — PPG rule baseline
# ============================================================

class PPGRuleBaseline(BaseEstimator, ClassifierMixin):
    """
    Predict H/D/A probabilities using season-to-date PPG difference.

    Approach:
      - sigmoid(k * (ppg_home - ppg_away)) -> P(H) vs P(A) split
      - a fixed draw probability DRW applied around the split
    """
    def __init__(self, k: float = 2.0, draw_frac: float = 0.24,
                 home_col: str = 'home_season_ppg',
                 away_col: str = 'away_season_ppg'):
        self.k = k
        self.draw_frac = draw_frac
        self.home_col = home_col
        self.away_col = away_col

    def fit(self, X, y):
        self.classes_ = np.array([0, 1, 2])
        return self

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            ppg_h = X[self.home_col].values
            ppg_a = X[self.away_col].values
        else:
            raise ValueError("PPGRuleBaseline expects a DataFrame with named columns")

        diff = ppg_h - ppg_a
        s = 1.0 / (1.0 + np.exp(-self.k * diff))
        d = self.draw_frac
        h = (1.0 - d) * s
        a = (1.0 - d) * (1.0 - s)

        out = np.zeros((len(X), 3))
        out[:, 0] = a
        out[:, 1] = d
        out[:, 2] = h
        return out

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


# ============================================================
# 9.5E — Elo baseline (logistic regression on Elo difference)
# ============================================================

class EloBaseline(BaseEstimator, ClassifierMixin):
    """
    Predict H/D/A probabilities from Elo difference using a small
    multinomial logistic regression.
    """
    def __init__(self, feature_cols=None):
        self.feature_cols = feature_cols or ['delta_elo']

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            X_sub = X[self.feature_cols].values
        else:
            raise ValueError("EloBaseline expects a DataFrame with named columns")

        # NOTE: multi_class argument intentionally omitted (sklearn 1.5+
        # defaults to multinomial and warns on the explicit argument).
        self.model_ = LogisticRegression(max_iter=1000, random_state=42)
        self.model_.fit(X_sub, y)
        self.classes_ = self.model_.classes_
        return self

    def predict_proba(self, X):
        X_sub = X[self.feature_cols].values
        return self.model_.predict_proba(X_sub)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


print("baselines.py module loaded.")
'''

import os
module_path = '/content/drive/MyDrive/epl_match_outcome/src/baselines.py'
with open(module_path, 'w') as f:
    f.write(baselines_py)
print("Wrote:", module_path)
print("Size:", os.path.getsize(module_path))

In [ ]:
import os, sys, importlib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, baselines, pipeline
importlib.reload(validation)
importlib.reload(baselines)
importlib.reload(pipeline)

# Load train features
train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
print("Train shape:", train.shape)

# Splitter
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)

# Baselines
def make_baselines():
    return {
        'MostFrequent': baselines.MostFrequentBaseline(),
        'Prior':        baselines.PriorBaseline(),
        'PPGRule':      baselines.PPGRuleBaseline(k=2.0, draw_frac=0.24),
        'EloBaseline':  baselines.EloBaseline(feature_cols=['delta_elo']),
    }

records = []
for fold_i, (tr_idx, va_idx) in enumerate(splitter):
    val_season = train['season_label'].iloc[va_idx].unique()[0]
    tr = train.iloc[tr_idx].reset_index(drop=True)
    va = train.iloc[va_idx].reset_index(drop=True)
    y_tr = tr['FTR_int'].values
    y_va = va['FTR_int'].values

    for name, model in make_baselines().items():
        if name in ('PPGRule', 'EloBaseline'):
            model.fit(tr, y_tr)
            proba = model.predict_proba(va)
        else:
            feat_cols = [c for c in tr.columns if c.startswith(('home_', 'away_', 'delta_'))]
            model.fit(tr[feat_cols].values, y_tr)
            proba = model.predict_proba(va[feat_cols].values)

        records.append({
            'fold':       fold_i + 1,
            'val_season': val_season,
            'baseline':   name,
            'accuracy':   accuracy_score(y_va, proba.argmax(axis=1)),
            'log_loss':   log_loss(y_va, proba, labels=[0, 1, 2]),
        })

results = pd.DataFrame(records)

print("\n=== Per-fold baseline log loss ===")
print(results.pivot(index='fold', columns='baseline', values='log_loss').round(4).to_string())

print("\n=== Summary ===")
summary = results.groupby('baseline').agg(
    ll_mean=('log_loss', 'mean'),
    ll_std=('log_loss', 'std'),
    acc_mean=('accuracy', 'mean'),
    acc_std=('accuracy', 'std'),
).round(4)
print(summary.to_string())

# Save
results_path = os.path.join(RESULTS_DIR, 'baseline_results.csv')
results.to_csv(results_path, index=False)
print("\nSaved:", results_path)

In [ ]:
lines = [
    "# Baseline Results — Step 9.5",
    "",
    "All baselines evaluated on 7-fold expanding-window CV on the train set",
    "(2015/16 – 2022/23). No val or test data touched.",
    "",
    "Main metric: **log loss** (lower is better).",
    "",
    "## Summary",
    "",
    "| Baseline | Log loss (mean) | Log loss (std) | Accuracy (mean) |",
    "| --- | ---: | ---: | ---: |",
]
for b, row in summary.iterrows():
    lines.append(f"| {b} | {row['ll_mean']:.4f} | {row['ll_std']:.4f} | {row['acc_mean']:.4f} |")

lines += [
    "",
    "## Interpretation",
    "",
    "- **MostFrequent** hits the accuracy floor (~0.45) but log loss is",
    "  degenerate (~19.7) — hard 0/1 probabilities make log loss unusable.",
    "- **Prior** is the real log-loss floor: **1.0633**.",
    "- **PPGRule** beats Prior by 0.020 (~2%) — simple season-strength signal works.",
    "- **EloBaseline** beats Prior by 0.090 (~8.5%) — Elo strength is a strong",
    "  football-native predictor. **This is the bar.**",
    "",
    "## What this means for modeling",
    "",
    "Any model we train at Step 10+ must beat **0.9734 (EloBaseline, mean)** on",
    "the same 7-fold CV. If a 47-feature model can't beat a 1-feature Elo model,",
    "we've learned nothing beyond team strength.",
    "",
    "## Per-fold pattern",
    "",
    "- Fold 3 (val = 2018/19) is easiest for Elo — large top-vs-rest gap that season.",
    "- Fold 5 (val = 2020/21) is hardest for every baseline — COVID-affected season,",
    "  home advantage collapsed, form patterns broke.",
    "",
    "## Artifacts",
    "",
    "- `src/baselines.py`",
    "- `results/baseline_results.csv`",
]

manifest_path = os.path.join(DOCS_DIR, 'baseline_results.md')
with open(manifest_path, 'w') as f:
    f.write("\n".join(lines))
print("Wrote:", manifest_path)
print("Size:", os.path.getsize(manifest_path))

In [ ]:
models_py = '''"""
models.py — factory functions for the three candidate models,
plus a CV evaluation helper.

All models are sklearn-compatible estimators. Scaling is applied only
for models that need it (LogReg).
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, log_loss

# XGBoost import (graceful failure if not installed)
try:
    from xgboost import XGBClassifier
    _XGB_AVAILABLE = True
except ImportError:
    _XGB_AVAILABLE = False


SEED = 42


# ============================================================
# Model factories
# ============================================================

def make_logreg(l1_ratio: float = 0.5, C: float = 1.0,
                class_weight: str | None = "balanced",
                max_iter: int = 2000):
    """
    Multinomial Logistic Regression with ElasticNet regularization.
    Includes StandardScaler inside the pipeline (LR needs it).
    """
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=l1_ratio,
            C=C,
            max_iter=max_iter,
            class_weight=class_weight,
            random_state=SEED,
        )),
    ])


def make_rf(n_estimators: int = 500,
            min_samples_leaf: int = 5,
            min_samples_split: int = 10,
            class_weight: str | None = "balanced_subsample",
            max_depth: int | None = None):
    """
    Random Forest Classifier.
    Tree-based, does not need scaling.
    """
    return RandomForestClassifier(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        class_weight=class_weight,
        max_depth=max_depth,
        random_state=SEED,
        n_jobs=-1,
    )


def make_xgb(n_estimators: int = 500,
             max_depth: int = 4,
             learning_rate: float = 0.05,
             subsample: float = 0.8,
             colsample_bytree: float = 0.8,
             min_child_weight: float = 5.0,
             reg_lambda: float = 1.0,
             reg_alpha: float = 0.0,
             objective: str = "multi:softprob",
             num_class: int = 3,
             eval_metric: str = "mlogloss"):
    """
    XGBoost Classifier for 3-class classification.
    Uses sklearn API; no scaling needed.
    """
    if not _XGB_AVAILABLE:
        raise ImportError("xgboost is not installed. Run: !pip install xgboost")
    return XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        reg_lambda=reg_lambda,
        reg_alpha=reg_alpha,
        objective=objective,
        num_class=num_class,
        eval_metric=eval_metric,
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist",
        verbosity=0,
    )


# ============================================================
# Feature column selector (matches pipeline.py convention)
# ============================================================

FEATURE_PREFIXES = ("home_", "away_", "delta_")


def select_features(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c.startswith(FEATURE_PREFIXES)]


# ============================================================
# CV evaluation helper
# ============================================================

def evaluate_cv(model_factory,
                train_df: pd.DataFrame,
                splitter,
                feature_cols: list | None = None,
                sample_weights: bool = False) -> pd.DataFrame:
    """
    Run a model through a CV splitter.

    Parameters
    ----------
    model_factory : callable
        Returns a fresh, unfitted estimator on each call.
    train_df : pd.DataFrame
        The training feature DataFrame (must have FTR_int and season_label).
    splitter : iterable of (train_idx, val_idx)
        Our SeasonSplit instance.
    feature_cols : list of str, optional
        Feature columns. If None, uses select_features().
    sample_weights : bool, default False
        If True, pass sample_weight to model.fit for models that accept it
        (XGBoost uses this for class balancing).

    Returns
    -------
    DataFrame with columns: fold, val_season, accuracy, log_loss.
    """
    if feature_cols is None:
        feature_cols = select_features(train_df)

    records = []
    for fold_i, (tr_idx, va_idx) in enumerate(splitter):
        tr = train_df.iloc[tr_idx].reset_index(drop=True)
        va = train_df.iloc[va_idx].reset_index(drop=True)

        X_tr = tr[feature_cols].values
        y_tr = tr["FTR_int"].values
        X_va = va[feature_cols].values
        y_va = va["FTR_int"].values

        model = model_factory()

        # Optional per-sample weights for XGBoost / RF class balancing
        if sample_weights and hasattr(model, "fit"):
            counts = np.bincount(y_tr, minlength=3).astype(float)
            class_w = len(y_tr) / (3.0 * np.where(counts > 0, counts, 1.0))
            w_tr = class_w[y_tr]
            try:
                model.fit(X_tr, y_tr, sample_weight=w_tr)
            except TypeError:
                # model doesn't accept sample_weight
                model.fit(X_tr, y_tr)
        else:
            model.fit(X_tr, y_tr)

        proba = model.predict_proba(X_va)
        preds = proba.argmax(axis=1)

        records.append({
            "fold": fold_i + 1,
            "val_season": tr.iloc[0]["season_label"] if False else va.iloc[0]["season_label"],
            "accuracy": accuracy_score(y_va, preds),
            "log_loss": log_loss(y_va, proba, labels=[0, 1, 2]),
        })

    return pd.DataFrame(records)


print("models.py module loaded. XGBoost available:", _XGB_AVAILABLE)
'''

import os
module_path = '/content/drive/MyDrive/epl_match_outcome/src/models.py'
with open(module_path, 'w') as f:
    f.write(models_py)
print("Wrote:", module_path)
print("Size:", os.path.getsize(module_path))

In [ ]:
import os, sys, importlib
import numpy as np
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, models
importlib.reload(validation)
importlib.reload(models)

# XGBoost install check
if not models._XGB_AVAILABLE:
    print("Installing xgboost...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "xgboost"], check=True)
    importlib.reload(models)

train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)
print("Train shape:", train.shape)
print("Number of folds:", splitter.get_n_splits())

# Feature columns
feat_cols = models.select_features(train)
print("Number of features:", len(feat_cols))

# Run all three models
print("\nTraining Logistic Regression...")
res_lr = models.evaluate_cv(models.make_logreg, train, splitter, feat_cols)

print("Training Random Forest...")
res_rf = models.evaluate_cv(models.make_rf, train, splitter, feat_cols)

print("Training XGBoost...")
res_xgb = models.evaluate_cv(models.make_xgb, train, splitter, feat_cols, sample_weights=True)

# Tag and combine
for df, name in [(res_lr, "LogReg"), (res_rf, "RandomForest"), (res_xgb, "XGBoost")]:
    df["model"] = name

all_res = pd.concat([res_lr, res_rf, res_xgb], ignore_index=True)

print("\n=== Per-fold log loss ===")
pivot = all_res.pivot(index='fold', columns='model', values='log_loss').round(4)
print(pivot.to_string())

print("\n=== Per-fold accuracy ===")
pivot_acc = all_res.pivot(index='fold', columns='model', values='accuracy').round(4)
print(pivot_acc.to_string())

print("\n=== Summary ===")
summary = all_res.groupby('model').agg(
    ll_mean=('log_loss', 'mean'),
    ll_std=('log_loss', 'std'),
    acc_mean=('accuracy', 'mean'),
    acc_std=('accuracy', 'std'),
).round(4)
print(summary.to_string())

# Save
all_res.to_csv(os.path.join(RESULTS_DIR, 'model_cv_results.csv'), index=False)
print("\nSaved:", os.path.join(RESULTS_DIR, 'model_cv_results.csv'))

In [ ]:
DOCS_DIR = os.path.join(PROJECT_ROOT, 'docs')

# Add baseline results for comparison
baseline = pd.read_csv(os.path.join(RESULTS_DIR, 'baseline_results.csv'))
baseline_only = baseline[baseline['baseline'].isin(['Prior', 'PPGRule', 'EloBaseline'])].copy()
baseline_only = baseline_only.rename(columns={'baseline': 'model'})
baseline_summary = baseline_only.groupby('model').agg(
    ll_mean=('log_loss', 'mean'),
    ll_std=('log_loss', 'std'),
    acc_mean=('accuracy', 'mean'),
    acc_std=('accuracy', 'std'),
).round(4)

# Combined table
all_summary = pd.concat([baseline_summary, summary])
all_summary = all_summary.sort_values('ll_mean')

print("=== COMBINED: baselines + models ===")
print(all_summary.to_string())
print()
print(f"Best model by log loss: {all_summary['ll_mean'].idxmin()}")
print(f"Best baseline by log loss: {baseline_summary['ll_mean'].idxmin()}")

# Improvement over Elo baseline
elo_ll = baseline_summary.loc['EloBaseline', 'll_mean']
print(f"\nElo baseline log loss: {elo_ll:.4f}")
for m in summary.index:
    ll = summary.loc[m, 'll_mean']
    delta = ll - elo_ll
    pct = (ll - elo_ll) / elo_ll * 100
    print(f"  {m:15s}: {ll:.4f}  (Δ {delta:+.4f}, {pct:+.2f}% vs Elo baseline)")

# Write markdown
lines = [
    "# Model CV Results — Step 11",
    "",
    "All models evaluated on 7-fold expanding-window CV on the train set",
    "(2015/16 – 2022/23). Same folds as the baselines.",
    "",
    "Main metric: **log loss** (lower is better).",
    "",
    "## Combined table (baselines + models)",
    "",
    "| Model | Log loss (mean) | Log loss (std) | Accuracy (mean) |",
    "| --- | ---: | ---: | ---: |",
]
for m, row in all_summary.iterrows():
    lines.append(f"| {m} | {row['ll_mean']:.4f} | {row['ll_std']:.4f} | {row['acc_mean']:.4f} |")

lines += [
    "",
    "## Improvement vs EloBaseline",
    "",
    f"EloBaseline log loss: **{elo_ll:.4f}**",
    "",
    "| Model | Log loss | Δ vs Elo | % vs Elo |",
    "| --- | ---: | ---: | ---: |",
]
for m in summary.index:
    ll = summary.loc[m, 'll_mean']
    delta = ll - elo_ll
    pct = (ll - elo_ll) / elo_ll * 100
    lines.append(f"| {m} | {ll:.4f} | {delta:+.4f} | {pct:+.2f}% |")

lines += [
    "",
    "## Interpretation",
    "",
    "See per-fold table for stability analysis. Key questions:",
    "",
    "1. Does the full Logistic Regression beat the Elo-only baseline?",
    "   - If yes: the 46 additional features carry real signal beyond strength.",
    "   - If no: we're not learning anything useful beyond Elo.",
    "2. Does XGBoost beat Logistic Regression meaningfully?",
    "   - If marginal: prefer LR (simpler, more interpretable) per Step 15 rule.",
    "   - If clear: consider XGBoost for final model.",
    "3. Is Random Forest competitive?",
    "   - If yes: keep as candidate.",
    "   - If poorly calibrated (expected): report as-is without adding calibration.",
    "",
    "## Artifacts",
    "",
    "- `src/models.py`",
    "- `results/model_cv_results.csv`",
]

manifest_path = os.path.join(DOCS_DIR, 'model_cv_results.md')
with open(manifest_path, 'w') as f:
    f.write("\n".join(lines))
print("\nWrote:", manifest_path)

In [ ]:
import os, sys, importlib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, models
importlib.reload(validation)
importlib.reload(models)

train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)

# Get fold 7 (last fold)
folds = list(splitter)
tr_idx, va_idx = folds[-1]   # fold 7
tr = train.iloc[tr_idx].reset_index(drop=True)
va = train.iloc[va_idx].reset_index(drop=True)

print(f"Fold 7: train={len(tr)} rows, val={len(va)} rows, val season={va['season_label'].iloc[0]}")
print()

feat_cols_all = models.select_features(train)
feat_cols_delta = [c for c in feat_cols_all if c.startswith('delta_')]
print(f"All features: {len(feat_cols_all)}")
print(f"Delta features: {len(feat_cols_delta)}")
print()


def run(model, features, use_weights=False, name=""):
    X_tr = tr[features].values; y_tr = tr['FTR_int'].values
    X_va = va[features].values; y_va = va['FTR_int'].values

    if use_weights:
        counts = np.bincount(y_tr, minlength=3).astype(float)
        cw = len(y_tr) / (3.0 * np.where(counts > 0, counts, 1.0))
        w = cw[y_tr]
        try:
            model.fit(X_tr, y_tr, sample_weight=w)
        except TypeError:
            model.fit(X_tr, y_tr)
    else:
        model.fit(X_tr, y_tr)

    proba = model.predict_proba(X_va)
    acc = accuracy_score(y_va, proba.argmax(axis=1))
    ll = log_loss(y_va, proba, labels=[0, 1, 2])
    print(f"{name:55s} ll={ll:.4f}  acc={acc:.4f}")
    return ll, acc


results = {}

print("=== Reference: EloBaseline on fold 7 ===")
from sklearn.linear_model import LogisticRegression as LR_
from sklearn.preprocessing import StandardScaler as SS_
elo_pipe = Pipeline([("s", SS_()), ("m", LR_(max_iter=1000, random_state=42))])
results['EloBaseline'] = run(elo_pipe, ['delta_elo'], name='EloBaseline')

print("\n=== E1 — XGBoost small (100 trees, depth 3, lr 0.1) ===")
xgb_small = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                          subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                          reg_lambda=1.0, objective='multi:softprob', num_class=3,
                          eval_metric='mlogloss', random_state=42, n_jobs=-1,
                          tree_method='hist', verbosity=0)
results['E1_xgb_small'] = run(xgb_small, feat_cols_all, name='E1 XGBoost small')

print("\n=== E2 — LogReg without class weights ===")
lr_no_w = Pipeline([("s", SS_()), ("m", LR_(penalty='elasticnet', solver='saga',
                                            l1_ratio=0.5, C=1.0, max_iter=2000,
                                            class_weight=None, random_state=42))])
results['E2_lr_no_weights'] = run(lr_no_w, feat_cols_all, name='E2 LogReg no class weights')

print("\n=== E3 — RF without class weights ===")
rf_no_w = RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
                                 min_samples_split=10, class_weight=None,
                                 random_state=42, n_jobs=-1)
results['E3_rf_no_weights'] = run(rf_no_w, feat_cols_all, name='E3 RF no class weights')

print("\n=== E4 — LogReg l1_ratio=1.0 (pure Lasso) ===")
lr_l1 = Pipeline([("s", SS_()), ("m", LR_(penalty='elasticnet', solver='saga',
                                          l1_ratio=1.0, C=1.0, max_iter=2000,
                                          class_weight=None, random_state=42))])
results['E4_lr_l1'] = run(lr_l1, feat_cols_all, name='E4 LogReg L1 only')

print("\n=== E5 — LogReg C=10 (less regularization) ===")
lr_c10 = Pipeline([("s", SS_()), ("m", LR_(penalty='elasticnet', solver='saga',
                                           l1_ratio=0.5, C=10.0, max_iter=2000,
                                           class_weight=None, random_state=42))])
results['E5_lr_C10'] = run(lr_c10, feat_cols_all, name='E5 LogReg C=10')

print("\n=== E6 — LogReg on deltas only (9 features) ===")
lr_delta = Pipeline([("s", SS_()), ("m", LR_(penalty='elasticnet', solver='saga',
                                             l1_ratio=0.5, C=1.0, max_iter=2000,
                                             class_weight=None, random_state=42))])
results['E6_lr_deltas'] = run(lr_delta, feat_cols_delta, name='E6 LogReg deltas only')

print("\n=== E7 — XGBoost small with sample weights ===")
xgb_small_w = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                            subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                            reg_lambda=1.0, objective='multi:softprob', num_class=3,
                            eval_metric='mlogloss', random_state=42, n_jobs=-1,
                            tree_method='hist', verbosity=0)
results['E7_xgb_small_weights'] = run(xgb_small_w, feat_cols_all, use_weights=True,
                                      name='E7 XGBoost small + sample weights')

print("\n=== E8 — RF deeper (min_samples_leaf=1, 200 trees) ===")
rf_deep = RandomForestClassifier(n_estimators=200, min_samples_leaf=1,
                                 min_samples_split=2, class_weight=None,
                                 random_state=42, n_jobs=-1)
results['E8_rf_deep'] = run(rf_deep, feat_cols_all, name='E8 RF deep')

print("\n=== Summary (all on fold 7) ===")
summ = pd.DataFrame([(k, v[0], v[1]) for k, v in results.items()],
                    columns=['experiment', 'log_loss', 'accuracy'])
summ = summ.sort_values('log_loss').reset_index(drop=True)
print(summ.round(4).to_string(index=False))

# Save
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
summ.to_csv(os.path.join(RESULTS_DIR, 'diagnostic_fold7.csv'), index=False)
print("\nSaved:", os.path.join(RESULTS_DIR, 'diagnostic_fold7.csv'))

In [ ]:
import os, sys, importlib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, models
importlib.reload(validation)
importlib.reload(models)

train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)
tr_idx, va_idx = list(splitter)[-1]  # fold 7
tr = train.iloc[tr_idx].reset_index(drop=True)
va = train.iloc[va_idx].reset_index(drop=True)
y_tr = tr['FTR_int'].values
y_va = va['FTR_int'].values

print(f"Fold 7: train={len(tr)} rows, val={len(va)} rows, val season={va['season_label'].iloc[0]}")
print()

# ---------- Subsets ----------
all_feat = models.select_features(train)
delta_feat = [c for c in all_feat if c.startswith('delta_')]
core_feat = [
    'delta_elo',
    'delta_form_pts_last5',
    'delta_venue_pts_last5',
    'delta_season_ppg',
    'delta_prev_season_ppg',
    'delta_rest_days',
    'home_is_promoted',
    'away_is_promoted',
]

subsets = {
    'S1_all47':  all_feat,
    'S2_deltas': delta_feat,
    'S3_core8':  core_feat,
}

# Sanity: core8 all present
missing = [c for c in core_feat if c not in train.columns]
assert not missing, f"Missing core features: {missing}"

print("Subset sizes:")
for name, feats in subsets.items():
    print(f"  {name:12s}: {len(feats)} features")
print()

# ---------- Models ----------
def lr_factory():
    return Pipeline([
        ("s", StandardScaler()),
        ("m", LogisticRegression(penalty='elasticnet', solver='saga',
                                 l1_ratio=0.5, C=1.0, max_iter=2000,
                                 class_weight=None, random_state=42))
    ])

def rf_factory():
    return RandomForestClassifier(n_estimators=500,
                                  min_samples_leaf=5,
                                  min_samples_split=10,
                                  class_weight=None,
                                  random_state=42, n_jobs=-1)

# ---------- Reference ----------
print("=== Reference ===")
elo_pipe = Pipeline([
    ("s", StandardScaler()),
    ("m", LogisticRegression(max_iter=1000, random_state=42))
])
elo_pipe.fit(tr[['delta_elo']].values, y_tr)
p = elo_pipe.predict_proba(va[['delta_elo']].values)
ll_elo = log_loss(y_va, p, labels=[0, 1, 2])
acc_elo = accuracy_score(y_va, p.argmax(axis=1))
print(f"EloBaseline (delta_elo only)               ll={ll_elo:.4f}  acc={acc_elo:.4f}")
print()

# ---------- Diagnostic grid ----------
print("=== Feature subset × model grid ===")
records = [{
    'subset': 'EloBaseline', 'model': 'LogReg', 'n_features': 1,
    'log_loss': ll_elo, 'accuracy': acc_elo,
}]

for sub_name, feats in subsets.items():
    X_tr = tr[feats].values
    X_va = va[feats].values

    # LogReg
    lr = lr_factory()
    lr.fit(X_tr, y_tr)
    p_lr = lr.predict_proba(X_va)
    ll_lr = log_loss(y_va, p_lr, labels=[0, 1, 2])
    acc_lr = accuracy_score(y_va, p_lr.argmax(axis=1))
    print(f"{sub_name:12s} × LogReg        n={len(feats):2d}  ll={ll_lr:.4f}  acc={acc_lr:.4f}")
    records.append({'subset': sub_name, 'model': 'LogReg', 'n_features': len(feats),
                    'log_loss': ll_lr, 'accuracy': acc_lr})

    # RandomForest
    rf = rf_factory()
    rf.fit(X_tr, y_tr)
    p_rf = rf.predict_proba(X_va)
    ll_rf = log_loss(y_va, p_rf, labels=[0, 1, 2])
    acc_rf = accuracy_score(y_va, p_rf.argmax(axis=1))
    print(f"{sub_name:12s} × RandomForest  n={len(feats):2d}  ll={ll_rf:.4f}  acc={acc_rf:.4f}")
    records.append({'subset': sub_name, 'model': 'RandomForest', 'n_features': len(feats),
                    'log_loss': ll_rf, 'accuracy': acc_rf})

print()

# ---------- Summary ----------
results = pd.DataFrame(records)
results_sorted = results.sort_values('log_loss').reset_index(drop=True)
print("=== Ranking (sorted by log loss) ===")
print(results_sorted.round(4).to_string(index=False))

# Save
out_path = os.path.join(RESULTS_DIR, 'feature_subset_diagnostic_fold7.csv')
results.to_csv(out_path, index=False)
print("\nSaved:", out_path)

# ---------- Comparison vs EloBaseline ----------
print("\n=== Comparison vs EloBaseline (0.9737) ===")
for _, r in results_sorted.iterrows():
    if r['subset'] == 'EloBaseline':
        continue
    delta = r['log_loss'] - ll_elo
    marker = " ✅ beats Elo" if delta < 0 else ""
    print(f"{r['subset']:12s} × {r['model']:13s}: {r['log_loss']:.4f}  (Δ {delta:+.4f}){marker}")

In [ ]:
mov_elo_code = '''

# ============================================================
# 20. MARGIN-OF-VICTORY ELO (added at Step 12c)
# ============================================================

def build_mov_elo_features(matches_df: pd.DataFrame,
                           k_factor: float = 20.0,
                           home_bonus: float = 60.0,
                           initial_elo: float = 1500.0,
                           regression: float = 0.25) -> pd.DataFrame:
    """
    Margin-of-victory Elo. Same as build_elo_features, but the K-factor
    is scaled by goal difference:

        mult = 1 + log(1 + goal_diff)   # 0 -> 1.0, 1 -> 1.69, 3 -> 2.39
        update = K * mult * (S - E)

    This makes convincing wins move the rating more than narrow wins.
    """
    import numpy as np
    df = matches_df.copy().reset_index(drop=True).sort_values(
        ["season_label", "Date"]).reset_index(drop=True)

    elo = {}
    last_season = None
    home_elos = [None] * len(df)
    away_elos = [None] * len(df)

    for pos in range(len(df)):
        row = df.iloc[pos]
        season = row["season_label"]
        home = row["HomeTeam"]; away = row["AwayTeam"]

        if season != last_season:
            if last_season is not None:
                for t in list(elo.keys()):
                    elo[t] = elo[t] + regression * (initial_elo - elo[t])
            last_season = season

        if home not in elo: elo[home] = initial_elo
        if away not in elo: elo[away] = initial_elo

        elo_h = elo[home]; elo_a = elo[away]
        home_elos[pos] = elo_h
        away_elos[pos] = elo_a

        E_h = 1.0 / (1.0 + 10.0 ** ((elo_a - elo_h - home_bonus) / 400.0))
        E_a = 1.0 - E_h

        if row["FTHG"] > row["FTAG"]:
            S_h, S_a = 1.0, 0.0
        elif row["FTHG"] < row["FTAG"]:
            S_h, S_a = 0.0, 1.0
        else:
            S_h, S_a = 0.5, 0.5

        gd = abs(int(row["FTHG"]) - int(row["FTAG"]))
        mult = 1.0 + np.log1p(gd)   # log(1) = 0 for a draw, log(2)=0.69 for 1-goal, etc.

        elo[home] = elo_h + k_factor * mult * (S_h - E_h)
        elo[away] = elo_a + k_factor * mult * (S_a - E_a)

    out = pd.DataFrame({
        "home_elo_mov": home_elos,
        "away_elo_mov": away_elos,
    }, index=df.index)
    out["delta_elo_mov"] = out["home_elo_mov"] - out["away_elo_mov"]
    out = out.reindex(matches_df.index)
    return out
'''

module_path = '/content/drive/MyDrive/epl_match_outcome/src/features.py'
with open(module_path, 'a') as f:
    f.write(mov_elo_code)

import importlib, sys, os
SRC_DIR = '/content/drive/MyDrive/epl_match_outcome/src'
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
import features
importlib.reload(features)

print("Appended MOV-Elo builder.")
print("Total public functions:", len([n for n in dir(features)
                                      if not n.startswith("_") and callable(getattr(features, n))]))

In [ ]:
import os
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')

# Build MOV-Elo on the same combined frame we used for the other features.
# We'll concatenate train_form + val_form rows (already contains all features),
# but we need to recompute MOV-Elo from scratch on the combined data.
train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])
val   = pd.read_csv(os.path.join(PROC_DIR, 'val.csv'),   parse_dates=['Date'])
combined = pd.concat([train, val], ignore_index=True).sort_values('Date').reset_index(drop=True)

mov_elo = features.build_mov_elo_features(combined)
print("MOV-Elo shape:", mov_elo.shape)
print(mov_elo.describe().round(2).to_string())

# Split back
train_mov = mov_elo.iloc[:len(train)].reset_index(drop=True)
val_mov   = mov_elo.iloc[len(train):].reset_index(drop=True)

# Sanity: lengths match
assert len(train_mov) == len(train)
assert len(val_mov) == len(val)

# Merge into existing train_form.csv and val_form.csv
for path, mov in [(os.path.join(PROC_DIR, 'train_form.csv'), train_mov),
                  (os.path.join(PROC_DIR, 'val_form.csv'),   val_mov)]:
    df = pd.read_csv(path, parse_dates=['Date'])
    df = df.drop(columns=[c for c in mov.columns if c in df.columns], errors='ignore')
    df = pd.concat([df.reset_index(drop=True), mov.reset_index(drop=True)], axis=1)
    df.to_csv(path, index=False)
    print(f"Updated {os.path.basename(path)}. Shape: {df.shape}, "
          f"has delta_elo_mov: {'delta_elo_mov' in df.columns}")

In [ ]:
import importlib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, models
importlib.reload(validation)
importlib.reload(models)

train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)

all_feat = models.select_features(train)
delta_feat = [c for c in all_feat if c.startswith('delta_')]
core_feat = [
    'delta_elo', 'delta_form_pts_last5', 'delta_venue_pts_last5',
    'delta_season_ppg', 'delta_prev_season_ppg', 'delta_rest_days',
    'home_is_promoted', 'away_is_promoted',
]

def lr_factory():
    return Pipeline([
        ("s", StandardScaler()),
        ("m", LogisticRegression(penalty='elasticnet', solver='saga',
                                 l1_ratio=0.5, C=1.0, max_iter=2000,
                                 class_weight=None, random_state=42))
    ])

def rf_factory():
    return RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
                                  min_samples_split=10, class_weight=None,
                                  random_state=42, n_jobs=-1)

# ---------- Candidates to test on all folds ----------
candidates = {
    'EloBaseline(delta_elo)':       (lr_factory, ['delta_elo']),
    'EloMOVE(delta_elo_mov)':       (lr_factory, ['delta_elo_mov']),
    'EloBoth(delta_elo + mov)':     (lr_factory, ['delta_elo', 'delta_elo_mov']),
    'Deltas_LR(9)':                 (lr_factory, delta_feat),
    'Core8_LR':                     (lr_factory, core_feat),
    'All47_RF':                     (rf_factory, all_feat),
}

records = []
for fold_i, (tr_idx, va_idx) in enumerate(splitter):
    val_season = train['season_label'].iloc[va_idx].unique()[0]
    tr = train.iloc[tr_idx].reset_index(drop=True)
    va = train.iloc[va_idx].reset_index(drop=True)
    y_tr = tr['FTR_int'].values
    y_va = va['FTR_int'].values

    for name, (factory, feats) in candidates.items():
        # Ensure all requested features exist
        missing = [c for c in feats if c not in tr.columns]
        if missing:
            print(f"SKIP {name}: missing features {missing}")
            continue

        m = factory()
        m.fit(tr[feats].values, y_tr)
        p = m.predict_proba(va[feats].values)

        records.append({
            'fold': fold_i + 1,
            'val_season': val_season,
            'model': name,
            'log_loss': log_loss(y_va, p, labels=[0, 1, 2]),
            'accuracy': accuracy_score(y_va, p.argmax(axis=1)),
        })

results = pd.DataFrame(records)

print("=== Per-fold log loss ===")
print(results.pivot(index='fold', columns='model', values='log_loss').round(4).to_string())

print("\n=== Summary (mean ± std) ===")
summary = results.groupby('model').agg(
    ll_mean=('log_loss', 'mean'),
    ll_std=('log_loss', 'std'),
    acc_mean=('accuracy', 'mean'),
).round(4)
print(summary.sort_values('ll_mean').to_string())

# Save
results.to_csv(os.path.join(RESULTS_DIR, 'foldwide_candidate_results.csv'), index=False)
print("\nSaved:", os.path.join(RESULTS_DIR, 'foldwide_candidate_results.csv'))

# Compare to current EloBaseline
elo_ll = summary.loc['EloBaseline(delta_elo)', 'll_mean']
print(f"\n=== Δ vs EloBaseline ({elo_ll:.4f}) ===")
for m in summary.index:
    if m == 'EloBaseline(delta_elo)':
        continue
    ll = summary.loc[m, 'll_mean']
    delta = ll - elo_ll
    marker = " ✅ BEATS" if delta < -0.0001 else ""
    print(f"{m:30s}: {ll:.4f}  (Δ {delta:+.4f}){marker}")

In [ ]:
import os, sys, importlib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, features
importlib.reload(validation)
importlib.reload(features)


# ---------- Extended Elo builder with custom (K, home_bonus) ----------
def build_elo_custom(matches_df, k_factor, home_bonus,
                     initial_elo=1500.0, regression=0.25):
    df = matches_df.copy().reset_index(drop=True).sort_values(
        ["season_label", "Date"]).reset_index(drop=True)

    elo = {}
    last_season = None
    home_elos = [None] * len(df)
    away_elos = [None] * len(df)

    for pos in range(len(df)):
        row = df.iloc[pos]
        season = row["season_label"]
        home = row["HomeTeam"]; away = row["AwayTeam"]

        if season != last_season:
            if last_season is not None:
                for t in list(elo.keys()):
                    elo[t] = elo[t] + regression * (initial_elo - elo[t])
            last_season = season

        if home not in elo: elo[home] = initial_elo
        if away not in elo: elo[away] = initial_elo

        elo_h = elo[home]; elo_a = elo[away]
        home_elos[pos] = elo_h
        away_elos[pos] = elo_a

        E_h = 1.0 / (1.0 + 10.0 ** ((elo_a - elo_h - home_bonus) / 400.0))
        E_a = 1.0 - E_h
        if row["FTHG"] > row["FTAG"]:
            S_h, S_a = 1.0, 0.0
        elif row["FTHG"] < row["FTAG"]:
            S_h, S_a = 0.0, 1.0
        else:
            S_h, S_a = 0.5, 0.5

        elo[home] = elo_h + k_factor * (S_h - E_h)
        elo[away] = elo_a + k_factor * (S_a - E_a)

    out = pd.DataFrame({
        "home_elo_cust": home_elos,
        "away_elo_cust": away_elos,
    })
    out["delta_elo_cust"] = out["home_elo_cust"] - out["away_elo_cust"]
    return out


# ---------- Setup: use train_form.csv (has FTR_int) ----------
train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
print("Train shape:", train.shape)
print("Has FTR_int:", 'FTR_int' in train.columns)

splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)

configs = [
    (15, 60, "K=15, HB=60"),
    (20, 60, "K=20, HB=60 (current)"),
    (25, 60, "K=25, HB=60"),
    (30, 60, "K=30, HB=60"),
    (20, 40, "K=20, HB=40"),
    (20, 80, "K=20, HB=80"),
    (20, 100, "K=20, HB=100"),
]

records = []
for k, hb, label in configs:
    for fold_i, (tr_idx, va_idx) in enumerate(splitter):
        tr = train.iloc[tr_idx].reset_index(drop=True).sort_values('Date')
        va = train.iloc[va_idx].reset_index(drop=True).sort_values('Date')

        # Compute Elo on train + val combined so val rows see full history
        combined = pd.concat([tr, va], ignore_index=True).sort_values(
            ['season_label', 'Date']).reset_index(drop=True)
        elo_df = build_elo_custom(combined, k_factor=k, home_bonus=hb)

        # Split back
        tr_elo = elo_df.iloc[:len(tr)].reset_index(drop=True)
        va_elo = elo_df.iloc[len(tr):].reset_index(drop=True)

        X_tr = tr_elo[['delta_elo_cust']].values
        y_tr = tr['FTR_int'].values
        X_va = va_elo[['delta_elo_cust']].values
        y_va = va['FTR_int'].values

        pipe = Pipeline([
            ("s", StandardScaler()),
            ("m", LogisticRegression(max_iter=1000, random_state=42)),
        ])
        pipe.fit(X_tr, y_tr)
        p = pipe.predict_proba(X_va)

        records.append({
            'config': label,
            'fold': fold_i + 1,
            'val_season': va['season_label'].iloc[0],
            'log_loss': log_loss(y_va, p, labels=[0, 1, 2]),
            'accuracy': accuracy_score(y_va, p.argmax(axis=1)),
        })

results = pd.DataFrame(records)

print("\n=== Per-fold log loss ===")
print(results.pivot(index='fold', columns='config', values='log_loss').round(4).to_string())

print("\n=== Summary (mean ± std) ===")
summary = results.groupby('config').agg(
    ll_mean=('log_loss', 'mean'),
    ll_std=('log_loss', 'std'),
    acc_mean=('accuracy', 'mean'),
).round(4)
print(summary.sort_values('ll_mean').to_string())

# Save
results.to_csv(os.path.join(RESULTS_DIR, 'elo_sweep_results.csv'), index=False)
print("\nSaved:", os.path.join(RESULTS_DIR, 'elo_sweep_results.csv'))

# Best config
best = summary.sort_values('ll_mean').index[0]
best_ll = summary.loc[best, 'll_mean']
current_ll = summary.loc['K=20, HB=60 (current)', 'll_mean']
print(f"\nBest config: {best}  (ll = {best_ll:.4f})")
print(f"Current:     K=20, HB=60 (current)  (ll = {current_ll:.4f})")
print(f"Improvement: {current_ll - best_ll:+.4f}")

In [ ]:
import os, sys, importlib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, log_loss, confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, models
importlib.reload(validation)
importlib.reload(models)

train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)

delta_feat = [c for c in models.select_features(train) if c.startswith('delta_')]
all_feat = models.select_features(train)

def lr_factory():
    return Pipeline([
        ("s", StandardScaler()),
        ("m", LogisticRegression(penalty='elasticnet', solver='saga',
                                 l1_ratio=0.5, C=1.0, max_iter=2000,
                                 class_weight=None, random_state=42))
    ])

def rf_factory():
    return RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
                                  min_samples_split=10, class_weight=None,
                                  random_state=42, n_jobs=-1)

candidates = {
    'EloBaseline':    (lambda: Pipeline([("s", StandardScaler()),
                                         ("m", LogisticRegression(max_iter=1000, random_state=42))]),
                       ['delta_elo']),
    'Deltas_LR':      (lr_factory, delta_feat),
    'All47_RF':       (rf_factory, all_feat),
}

# Aggregate predictions over all folds
all_preds = {name: {'y_true': [], 'y_pred': [], 'y_proba': []} for name in candidates}

for fold_i, (tr_idx, va_idx) in enumerate(splitter):
    tr = train.iloc[tr_idx].reset_index(drop=True)
    va = train.iloc[va_idx].reset_index(drop=True)
    y_tr = tr['FTR_int'].values
    y_va = va['FTR_int'].values

    for name, (factory, feats) in candidates.items():
        m = factory()
        m.fit(tr[feats].values, y_tr)
        p = m.predict_proba(va[feats].values)
        all_preds[name]['y_true'].extend(y_va)
        all_preds[name]['y_pred'].extend(p.argmax(axis=1))
        all_preds[name]['y_proba'].extend(p)

# Compute metrics on the aggregate
rows = []
for name, d in all_preds.items():
    y_true = np.array(d['y_true'])
    y_pred = np.array(d['y_pred'])
    y_proba = np.array(d['y_proba'])

    acc = accuracy_score(y_true, y_pred)
    ll = log_loss(y_true, y_proba, labels=[0, 1, 2])
    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], zero_division=0
    )
    macro_f1 = f1.mean()

    # Per-class rows
    for cls_idx, cls_name in enumerate(['A', 'D', 'H']):
        rows.append({
            'model': name,
            'metric_scope': f'class_{cls_name}',
            'accuracy': np.nan,
            'log_loss': np.nan,
            'precision': prec[cls_idx],
            'recall': rec[cls_idx],
            'f1': f1[cls_idx],
            'support': support[cls_idx],
        })

    # Overall row
    rows.append({
        'model': name,
        'metric_scope': 'overall',
        'accuracy': acc,
        'log_loss': ll,
        'precision': np.nan,
        'recall': np.nan,
        'f1': macro_f1,
        'support': len(y_true),
    })

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(os.path.join(RESULTS_DIR, 'step12_metrics.csv'), index=False)

print("=== Overall metrics ===")
print(metrics_df[metrics_df['metric_scope'] == 'overall']
      [['model', 'accuracy', 'log_loss', 'f1']]
      .rename(columns={'f1': 'macro_f1'})
      .to_string(index=False))

print("\n=== Per-class metrics ===")
print(metrics_df[metrics_df['metric_scope'].str.startswith('class_')]
      [['model', 'metric_scope', 'precision', 'recall', 'f1', 'support']]
      .round(3).to_string(index=False))

print("\nSaved:", os.path.join(RESULTS_DIR, 'step12_metrics.csv'))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
class_labels = ['A', 'D', 'H']

for ax, (name, d) in zip(axes, all_preds.items()):
    cm = confusion_matrix(d['y_true'], d['y_pred'], labels=[0, 1, 2])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels,
                ax=ax, cbar=False)
    ax.set_title(f"{name}\n(aggregate over 7 folds)")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step12_confusion_matrices.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)

# Also print numeric versions
for name, d in all_preds.items():
    cm = confusion_matrix(d['y_true'], d['y_pred'], labels=[0, 1, 2])
    print(f"\n{name} confusion matrix:")
    print(pd.DataFrame(cm, index=['true_A','true_D','true_H'],
                       columns=['pred_A','pred_D','pred_H']).to_string())

In [ ]:
from sklearn.calibration import calibration_curve

name = 'EloBaseline'
d = all_preds[name]
y_true = np.array(d['y_true'])
y_proba = np.array(d['y_proba'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for cls_idx, cls_name in enumerate(['A', 'D', 'H']):
    y_binary = (y_true == cls_idx).astype(int)
    prob_true, prob_pred = calibration_curve(y_binary, y_proba[:, cls_idx],
                                              n_bins=10, strategy='uniform')
    axes[cls_idx].plot(prob_pred, prob_true, marker='o', label='Model')
    axes[cls_idx].plot([0, 1], [0, 1], 'k--', label='Perfect')
    axes[cls_idx].set_title(f"Calibration — class {cls_name}")
    axes[cls_idx].set_xlabel("Predicted probability")
    axes[cls_idx].set_ylabel("Empirical frequency")
    axes[cls_idx].legend()
    axes[cls_idx].grid(alpha=0.3)

plt.tight_layout()
cal_path = os.path.join(RESULTS_DIR, 'step12_calibration_EloBaseline.png')
plt.savefig(cal_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", cal_path)

# Also compare calibration of all three for the H class
fig, ax = plt.subplots(figsize=(7, 6))
for name, d in all_preds.items():
    y_true = np.array(d['y_true'])
    y_proba = np.array(d['y_proba'])
    y_binary = (y_true == 2).astype(int)   # Home win
    prob_true, prob_pred = calibration_curve(y_binary, y_proba[:, 2],
                                              n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, marker='o', label=name)
ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
ax.set_title("Calibration — class H (home win)")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Empirical frequency")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step12_calibration_H_comparison.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)

In [ ]:
print("=== Predicted class distribution vs actual ===")
actual_dist = pd.Series(np.array(all_preds['EloBaseline']['y_true'])).value_counts(normalize=True).sort_index()
print(f"Actual:  A={actual_dist[0]:.3f}, D={actual_dist[1]:.3f}, H={actual_dist[2]:.3f}")

for name, d in all_preds.items():
    preds = np.array(d['y_pred'])
    dist = pd.Series(preds).value_counts(normalize=True).sort_index()
    # Fill in zeros for missing
    dist = dist.reindex([0, 1, 2], fill_value=0)
    print(f"{name:15s}:  A={dist[0]:.3f}, D={dist[1]:.3f}, H={dist[2]:.3f}")

print("\nNote: a model that never predicts D will have poor D recall.")
print("The H/D/A base rates are approximately 32% / 23% / 45%.")

In [ ]:
DOCS_DIR = os.path.join(PROJECT_ROOT, 'docs')

overall = metrics_df[metrics_df['metric_scope'] == 'overall']
per_class = metrics_df[metrics_df['metric_scope'].str.startswith('class_')]

lines = [
    "# Step 12 — Formal Evaluation",
    "",
    "Evaluated on aggregated predictions over all 7 CV folds.",
    "Each fold = one full season. Total evaluation set: 2660 matches.",
    "",
    "## Overall metrics",
    "",
    "| Model | Log loss | Accuracy | Macro F1 |",
    "| --- | ---: | ---: | ---: |",
]
for _, r in overall.iterrows():
    lines.append(f"| {r['model']} | {r['log_loss']:.4f} | {r['accuracy']:.4f} | {r['f1']:.4f} |")

lines += [
    "",
    "## Per-class metrics",
    "",
    "| Model | Class | Precision | Recall | F1 | Support |",
    "| --- | --- | ---: | ---: | ---: | ---: |",
]
for _, r in per_class.iterrows():
    cls = r['metric_scope'].replace('class_', '')
    lines.append(f"| {r['model']} | {cls} | {r['precision']:.3f} | {r['recall']:.3f} | {r['f1']:.3f} | {int(r['support'])} |")

lines += [
    "",
    "## Calibration",
    "",
    "See `results/step12_calibration_EloBaseline.png` and",
    "`results/step12_calibration_H_comparison.png`.",
    "",
    "## Interpretation",
    "",
    "The Draw class is consistently the hardest to predict — all models",
    "show lower recall and precision on D than on A or H. This matches",
    "the known difficulty of predicting draws in football.",
    "",
    "## Artifacts",
    "",
    "- results/step12_metrics.csv",
    "- results/step12_confusion_matrices.png",
    "- results/step12_calibration_EloBaseline.png",
    "- results/step12_calibration_H_comparison.png",
]

with open(os.path.join(DOCS_DIR, 'step12_evaluation.md'), 'w') as f:
    f.write("\n".join(lines))
print("Wrote:", os.path.join(DOCS_DIR, 'step12_evaluation.md'))

In [ ]:
import os, sys, importlib
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, models
importlib.reload(validation)
importlib.reload(models)

train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)

delta_feat = [c for c in models.select_features(train) if c.startswith('delta_')]
all_feat = models.select_features(train)

def lr_factory():
    return Pipeline([
        ("s", StandardScaler()),
        ("m", LogisticRegression(penalty='elasticnet', solver='saga',
                                 l1_ratio=0.5, C=1.0, max_iter=2000,
                                 class_weight=None, random_state=42))
    ])

def rf_factory():
    return RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
                                  min_samples_split=10, class_weight=None,
                                  random_state=42, n_jobs=-1)

candidates = {
    'EloBaseline': (lambda: Pipeline([("s", StandardScaler()),
                                       ("m", LogisticRegression(max_iter=1000, random_state=42))]),
                    ['delta_elo']),
    'Deltas_LR':   (lr_factory, delta_feat),
    'All47_RF':    (rf_factory, all_feat),
}

# ---------- Per-fold train vs val ----------
rows = []
for fold_i, (tr_idx, va_idx) in enumerate(splitter):
    val_season = train['season_label'].iloc[va_idx].unique()[0]
    tr = train.iloc[tr_idx].reset_index(drop=True)
    va = train.iloc[va_idx].reset_index(drop=True)
    y_tr = tr['FTR_int'].values
    y_va = va['FTR_int'].values

    for name, (factory, feats) in candidates.items():
        m = factory()
        m.fit(tr[feats].values, y_tr)

        # Train performance
        p_tr = m.predict_proba(tr[feats].values)
        tr_ll = log_loss(y_tr, p_tr, labels=[0, 1, 2])

        # Val performance
        p_va = m.predict_proba(va[feats].values)
        va_ll = log_loss(y_va, p_va, labels=[0, 1, 2])

        rows.append({
            'fold': fold_i + 1,
            'val_season': val_season,
            'model': name,
            'train_ll': tr_ll,
            'val_ll': va_ll,
            'gap': va_ll - tr_ll,
        })

gap_df = pd.DataFrame(rows)

print("=== Per-fold train vs val log loss ===")
pivot = gap_df.pivot_table(index='fold', columns='model', values=['train_ll', 'val_ll', 'gap'])
print(pivot.round(4).to_string())

print("\n=== Mean summary ===")
summary = gap_df.groupby('model').agg(
    train_ll_mean=('train_ll', 'mean'),
    val_ll_mean=('val_ll', 'mean'),
    gap_mean=('gap', 'mean'),
    gap_std=('gap', 'std'),
).round(4)
print(summary.to_string())

gap_df.to_csv(os.path.join(RESULTS_DIR, 'step13_gap.csv'), index=False)
print("\nSaved:", os.path.join(RESULTS_DIR, 'step13_gap.csv'))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, (factory, feats)) in zip(axes, candidates.items()):
    # Collect (train_size, train_ll, val_ll) across folds
    train_sizes = []
    train_losses = []
    val_losses = []

    for tr_idx, va_idx in splitter:
        tr = train.iloc[tr_idx].reset_index(drop=True)
        va = train.iloc[va_idx].reset_index(drop=True)
        y_tr = tr['FTR_int'].values
        y_va = va['FTR_int'].values

        m = factory()
        m.fit(tr[feats].values, y_tr)

        p_tr = m.predict_proba(tr[feats].values)
        p_va = m.predict_proba(va[feats].values)

        train_sizes.append(len(tr))
        train_losses.append(log_loss(y_tr, p_tr, labels=[0, 1, 2]))
        val_losses.append(log_loss(y_va, p_va, labels=[0, 1, 2]))

    ax.plot(train_sizes, train_losses, marker='o', label='Train', color='#1f77b4')
    ax.plot(train_sizes, val_losses, marker='s', label='Val', color='#d62728')
    ax.set_xlabel("Training set size")
    ax.set_ylabel("Log loss")
    ax.set_title(f"Learning curve — {name}")
    ax.grid(alpha=0.3)
    ax.legend()
    ax.axhline(0.95, color='gray', linestyle='--', alpha=0.5, label='0.95 ref')

plt.tight_layout()
lc_path = os.path.join(RESULTS_DIR, 'step13_learning_curves.png')
plt.savefig(lc_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", lc_path)

In [ ]:
DOCS_DIR = os.path.join(PROJECT_ROOT, 'docs')

lines = [
    "# Step 13 — Overfitting / Underfitting Analysis",
    "",
    "Computed train vs val log loss per fold, and learning curves.",
    "",
    "## Train vs val summary (mean across 7 folds)",
    "",
    "| Model | Train log loss | Val log loss | Gap | Gap std |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for m, r in summary.iterrows():
    lines.append(f"| {m} | {r['train_ll_mean']:.4f} | {r['val_ll_mean']:.4f} | {r['gap_mean']:+.4f} | {r['gap_std']:.4f} |")

lines += [
    "",
    "## Interpretation",
    "",
    "- A **large positive gap** (val >> train) indicates overfitting.",
    "- A **near-zero gap** with high loss indicates the model is at the",
    "  information ceiling — more capacity or tuning won't help.",
    "- The chart `results/step13_learning_curves.png` shows how train and",
    "  val losses evolve as training data grows.",
    "",
    "## Artifacts",
    "",
    "- results/step13_gap.csv",
    "- results/step13_learning_curves.png",
]

with open(os.path.join(DOCS_DIR, 'step13_overfitting.md'), 'w') as f:
    f.write("\n".join(lines))
print("Wrote:", os.path.join(DOCS_DIR, 'step13_overfitting.md'))

In [ ]:
import os, sys, importlib, time
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, models
importlib.reload(validation)
importlib.reload(models)

train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)

delta_feat = [c for c in models.select_features(train) if c.startswith('delta_')]

# ---------- LR Grid ----------
C_values = [0.1, 0.5, 1.0, 5.0, 10.0]
l1_values = [0.2, 0.5, 0.8]

records = []
t0 = time.time()
total = len(C_values) * len(l1_values) * splitter.get_n_splits()
done = 0

for C in C_values:
    for l1 in l1_values:
        fold_lls = []
        for tr_idx, va_idx in splitter:
            tr = train.iloc[tr_idx].reset_index(drop=True)
            va = train.iloc[va_idx].reset_index(drop=True)

            pipe = Pipeline([
                ("s", StandardScaler()),
                ("m", LogisticRegression(penalty='elasticnet', solver='saga',
                                         l1_ratio=l1, C=C, max_iter=2000,
                                         class_weight=None, random_state=42))
            ])
            pipe.fit(tr[delta_feat].values, tr['FTR_int'].values)
            p = pipe.predict_proba(va[delta_feat].values)
            fold_lls.append(log_loss(va['FTR_int'].values, p, labels=[0, 1, 2]))
            done += 1
            if done % 10 == 0:
                elapsed = time.time() - t0
                print(f"  {done}/{total} fits done, elapsed {elapsed:.1f}s")

        records.append({
            'C': C, 'l1_ratio': l1,
            'll_mean': np.mean(fold_lls),
            'll_std': np.std(fold_lls),
        })

lr_results = pd.DataFrame(records).sort_values('ll_mean').reset_index(drop=True)
print("\n=== Deltas_LR tuning results ===")
print(lr_results.round(4).to_string(index=False))

lr_results.to_csv(os.path.join(RESULTS_DIR, 'tuning_lr.csv'), index=False)
print("\nSaved:", os.path.join(RESULTS_DIR, 'tuning_lr.csv'))
print(f"\nBest LR: C={lr_results.iloc[0]['C']}, l1_ratio={lr_results.iloc[0]['l1_ratio']}, "
      f"ll={lr_results.iloc[0]['ll_mean']:.4f}")
print(f"EloBaseline reference: 0.9734")
print(f"Δ best LR vs Elo: {lr_results.iloc[0]['ll_mean'] - 0.9734:+.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# ---------- RF Grid ----------
min_leaf_values = [10, 20, 40]
max_depth_values = [None, 10, 8]
n_estimators_fixed = 300

all_feat = models.select_features(train)

rf_records = []
t0 = time.time()
total = len(min_leaf_values) * len(max_depth_values) * splitter.get_n_splits()
done = 0

for msl in min_leaf_values:
    for md in max_depth_values:
        fold_lls = []
        for tr_idx, va_idx in splitter:
            tr = train.iloc[tr_idx].reset_index(drop=True)
            va = train.iloc[va_idx].reset_index(drop=True)

            rf = RandomForestClassifier(
                n_estimators=n_estimators_fixed,
                min_samples_leaf=msl,
                min_samples_split=max(2, msl),
                max_depth=md,
                class_weight=None,
                random_state=42, n_jobs=-1,
            )
            rf.fit(tr[all_feat].values, tr['FTR_int'].values)
            p = rf.predict_proba(va[all_feat].values)
            fold_lls.append(log_loss(va['FTR_int'].values, p, labels=[0, 1, 2]))
            done += 1
            if done % 5 == 0:
                elapsed = time.time() - t0
                print(f"  {done}/{total} fits done, elapsed {elapsed:.1f}s")

        rf_records.append({
            'min_samples_leaf': msl,
            'max_depth': str(md),
            'll_mean': np.mean(fold_lls),
            'll_std': np.std(fold_lls),
        })

rf_results = pd.DataFrame(rf_records).sort_values('ll_mean').reset_index(drop=True)
print("\n=== All47_RF tuning results ===")
print(rf_results.round(4).to_string(index=False))

rf_results.to_csv(os.path.join(RESULTS_DIR, 'tuning_rf.csv'), index=False)
print("\nSaved:", os.path.join(RESULTS_DIR, 'tuning_rf.csv'))
print(f"\nBest RF: min_samples_leaf={rf_results.iloc[0]['min_samples_leaf']}, "
      f"max_depth={rf_results.iloc[0]['max_depth']}, "
      f"ll={rf_results.iloc[0]['ll_mean']:.4f}")
print(f"EloBaseline reference: 0.9734")
print(f"Δ best RF vs Elo: {rf_results.iloc[0]['ll_mean'] - 0.9734:+.4f}")

In [ ]:
DOCS_DIR = os.path.join(PROJECT_ROOT, 'docs')

best_lr = lr_results.iloc[0]
best_rf = rf_results.iloc[0]

lines = [
    "# Step 14 — Hyperparameter Tuning",
    "",
    "Grid search on the same 7-fold CV used throughout.",
    "EloBaseline reference (champion): **0.9734** log loss.",
    "",
    "## Deltas_LR grid (15 configs)",
    "",
    "| C | l1_ratio | Log loss (mean) | Log loss (std) |",
    "| ---: | ---: | ---: | ---: |",
]
for _, r in lr_results.iterrows():
    lines.append(f"| {r['C']} | {r['l1_ratio']} | {r['ll_mean']:.4f} | {r['ll_std']:.4f} |")

lines += [
    "",
    f"**Best LR:** C={best_lr['C']}, l1_ratio={best_lr['l1_ratio']}, "
    f"log loss = **{best_lr['ll_mean']:.4f}** (Δ vs Elo {best_lr['ll_mean']-0.9734:+.4f})",
    "",
    "## All47_RF grid (9 configs, 300 trees each)",
    "",
    "| min_samples_leaf | max_depth | Log loss (mean) | Log loss (std) |",
    "| ---: | ---: | ---: | ---: |",
]
for _, r in rf_results.iterrows():
    lines.append(f"| {r['min_samples_leaf']} | {r['max_depth']} | {r['ll_mean']:.4f} | {r['ll_std']:.4f} |")

lines += [
    "",
    f"**Best RF:** min_samples_leaf={best_rf['min_samples_leaf']}, "
    f"max_depth={best_rf['max_depth']}, log loss = **{best_rf['ll_mean']:.4f}** "
    f"(Δ vs Elo {best_rf['ll_mean']-0.9734:+.4f})",
    "",
    "## Interpretation",
    "",
    "Compare the best tuned configuration of each family against EloBaseline.",
    "If neither beats EloBaseline by ≥ 0.005, EloBaseline remains champion.",
    "",
    "## Artifacts",
    "",
    "- results/tuning_lr.csv",
    "- results/tuning_rf.csv",
]

with open(os.path.join(DOCS_DIR, 'step14_tuning.md'), 'w') as f:
    f.write("\n".join(lines))
print("Wrote:", os.path.join(DOCS_DIR, 'step14_tuning.md'))

In [ ]:
import os, sys, importlib, time
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, models
importlib.reload(validation)
importlib.reload(models)

train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)
delta_feat = [c for c in models.select_features(train) if c.startswith('delta_')]

C_values = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
l1_values = [0.7, 0.8, 0.9, 1.0]

records = []
t0 = time.time()
total = len(C_values) * len(l1_values) * splitter.get_n_splits()
done = 0

for C in C_values:
    for l1 in l1_values:
        fold_lls = []
        for tr_idx, va_idx in splitter:
            tr = train.iloc[tr_idx].reset_index(drop=True)
            va = train.iloc[va_idx].reset_index(drop=True)

            pipe = Pipeline([
                ("s", StandardScaler()),
                ("m", LogisticRegression(penalty='elasticnet', solver='saga',
                                         l1_ratio=l1, C=C, max_iter=3000,
                                         class_weight=None, random_state=42))
            ])
            pipe.fit(tr[delta_feat].values, tr['FTR_int'].values)
            p = pipe.predict_proba(va[delta_feat].values)
            fold_lls.append(log_loss(va['FTR_int'].values, p, labels=[0, 1, 2]))
            done += 1

        records.append({
            'C': C, 'l1_ratio': l1,
            'll_mean': np.mean(fold_lls),
            'll_std': np.std(fold_lls),
        })

lr_ext = pd.DataFrame(records).sort_values('ll_mean').reset_index(drop=True)
print(f"Done in {time.time()-t0:.1f}s")
print("\n=== Extended Deltas_LR grid ===")
print(lr_ext.round(4).to_string(index=False))

lr_ext.to_csv(os.path.join(RESULTS_DIR, 'tuning_lr_extended.csv'), index=False)
print("\nSaved:", os.path.join(RESULTS_DIR, 'tuning_lr_extended.csv'))

best = lr_ext.iloc[0]
print(f"\nBest: C={best['C']}, l1_ratio={best['l1_ratio']}, ll={best['ll_mean']:.4f}")
print(f"EloBaseline: 0.9734")
print(f"Δ best vs Elo: {best['ll_mean'] - 0.9734:+.4f}")

In [ ]:
best_C = best['C']
best_l1 = best['l1_ratio']
print(f"Verifying per-fold: Deltas_LR (C={best_C}, l1_ratio={best_l1}) vs EloBaseline")
print()

# Collect per-fold results for both models
per_fold = []
for fold_i, (tr_idx, va_idx) in enumerate(splitter):
    val_season = train['season_label'].iloc[va_idx].unique()[0]
    tr = train.iloc[tr_idx].reset_index(drop=True)
    va = train.iloc[va_idx].reset_index(drop=True)
    y_tr = tr['FTR_int'].values
    y_va = va['FTR_int'].values

    # Deltas_LR winning config
    pipe_lr = Pipeline([
        ("s", StandardScaler()),
        ("m", LogisticRegression(penalty='elasticnet', solver='saga',
                                 l1_ratio=best_l1, C=best_C, max_iter=3000,
                                 class_weight=None, random_state=42))
    ])
    pipe_lr.fit(tr[delta_feat].values, y_tr)
    p_lr = pipe_lr.predict_proba(va[delta_feat].values)
    ll_lr = log_loss(y_va, p_lr, labels=[0, 1, 2])

    # EloBaseline
    pipe_elo = Pipeline([
        ("s", StandardScaler()),
        ("m", LogisticRegression(max_iter=1000, random_state=42))
    ])
    pipe_elo.fit(tr[['delta_elo']].values, y_tr)
    p_elo = pipe_elo.predict_proba(va[['delta_elo']].values)
    ll_elo = log_loss(y_va, p_elo, labels=[0, 1, 2])

    per_fold.append({
        'fold': fold_i + 1,
        'val_season': val_season,
        'EloBaseline': ll_elo,
        'Deltas_LR_tuned': ll_lr,
        'delta': ll_lr - ll_elo,
        'winner': 'Deltas_LR' if ll_lr < ll_elo else 'EloBaseline',
    })

per_fold_df = pd.DataFrame(per_fold)
print("=== Per-fold comparison ===")
print(per_fold_df.round(4).to_string(index=False))
print()

n_lr_wins = (per_fold_df['winner'] == 'Deltas_LR').sum()
n_elo_wins = (per_fold_df['winner'] == 'EloBaseline').sum()
print(f"Deltas_LR wins: {n_lr_wins} folds")
print(f"EloBaseline wins: {n_elo_wins} folds")
print(f"Mean delta (LR - Elo): {per_fold_df['delta'].mean():+.4f}")

per_fold_df.to_csv(os.path.join(RESULTS_DIR, 'step14b_perfold.csv'), index=False)
print("\nSaved:", os.path.join(RESULTS_DIR, 'step14b_perfold.csv'))

In [ ]:
DOCS_DIR = os.path.join(PROJECT_ROOT, 'docs')

lines = [
    "# Step 14b — Extended LR grid + per-fold verification",
    "",
    "## Extended grid results",
    "",
    "| C | l1_ratio | Log loss (mean) | Std |",
    "| ---: | ---: | ---: | ---: |",
]
for _, r in lr_ext.iterrows():
    lines.append(f"| {r['C']} | {r['l1_ratio']} | {r['ll_mean']:.4f} | {r['ll_std']:.4f} |")

lines += [
    "",
    f"**Best config:** C={best['C']}, l1_ratio={best['l1_ratio']}",
    f"**Log loss:** {best['ll_mean']:.4f}",
    f"**Δ vs EloBaseline (0.9734):** {best['ll_mean']-0.9734:+.4f}",
    "",
    "## Per-fold verification",
    "",
    "| Fold | Val season | EloBaseline | Deltas_LR (tuned) | Delta | Winner |",
    "| ---: | --- | ---: | ---: | ---: | --- |",
]
for _, r in per_fold_df.iterrows():
    lines.append(f"| {int(r['fold'])} | {r['val_season']} | {r['EloBaseline']:.4f} | "
                 f"{r['Deltas_LR_tuned']:.4f} | {r['delta']:+.4f} | {r['winner']} |")

lines += [
    "",
    f"**Deltas_LR wins:** {n_lr_wins} folds",
    f"**EloBaseline wins:** {n_elo_wins} folds",
    f"**Mean delta:** {per_fold_df['delta'].mean():+.4f}",
    "",
    "## Conclusion",
    "",
    "If the win is consistent across most folds, the tuned Deltas_LR is our champion.",
    "If the win is concentrated in 1-2 folds, the improvement may not generalize.",
    "",
]

with open(os.path.join(DOCS_DIR, 'step14b_results.md'), 'w') as f:
    f.write("\n".join(lines))
print("Wrote:", os.path.join(DOCS_DIR, 'step14b_results.md'))

In [ ]:
import os, sys, importlib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, log_loss, confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation, models
importlib.reload(validation)
importlib.reload(models)

train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
splitter = validation.SeasonSplit(train['season_label'], min_train_seasons=1)

delta_feat = [c for c in models.select_features(train) if c.startswith('delta_')]
all_feat = models.select_features(train)

# Define final candidate factories with the tuned configs
def make_elo():
    return Pipeline([("s", StandardScaler()),
                     ("m", LogisticRegression(max_iter=1000, random_state=42))])

def make_deltas_lr():
    return Pipeline([("s", StandardScaler()),
                     ("m", LogisticRegression(penalty='elasticnet', solver='saga',
                                              l1_ratio=1.0, C=0.1, max_iter=3000,
                                              class_weight=None, random_state=42))])

def make_all47_rf():
    return RandomForestClassifier(n_estimators=300,
                                  min_samples_leaf=20,
                                  min_samples_split=20,
                                  max_depth=8,
                                  class_weight=None,
                                  random_state=42, n_jobs=-1)

candidates = {
    'EloBaseline':      (make_elo,       ['delta_elo']),
    'Deltas_LR_tuned':  (make_deltas_lr, delta_feat),
    'All47_RF_tuned':   (make_all47_rf,  all_feat),
}

# Aggregate predictions
agg = {name: {'y_true': [], 'y_pred': [], 'y_proba': []} for name in candidates}

for fold_i, (tr_idx, va_idx) in enumerate(splitter):
    tr = train.iloc[tr_idx].reset_index(drop=True)
    va = train.iloc[va_idx].reset_index(drop=True)
    y_tr = tr['FTR_int'].values
    y_va = va['FTR_int'].values

    for name, (factory, feats) in candidates.items():
        m = factory()
        m.fit(tr[feats].values, y_tr)
        p = m.predict_proba(va[feats].values)
        agg[name]['y_true'].extend(y_va)
        agg[name]['y_pred'].extend(p.argmax(axis=1))
        agg[name]['y_proba'].extend(p)

# Overall metrics
rows = []
for name, d in agg.items():
    y_true = np.array(d['y_true'])
    y_pred = np.array(d['y_pred'])
    y_proba = np.array(d['y_proba'])

    acc = accuracy_score(y_true, y_pred)
    ll = log_loss(y_true, y_proba, labels=[0, 1, 2])
    _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2],
                                                  zero_division=0)
    rows.append({
        'model': name,
        'n_features': len(candidates[name][1]),
        'log_loss': ll,
        'accuracy': acc,
        'macro_f1': f1.mean(),
    })

overall = pd.DataFrame(rows).sort_values('log_loss').reset_index(drop=True)
print("=== Overall comparison ===")
print(overall.round(4).to_string(index=False))

overall.to_csv(os.path.join(RESULTS_DIR, 'step15_overall.csv'), index=False)

In [ ]:
champion_name = 'Deltas_LR_tuned'
d = agg[champion_name]
y_true = np.array(d['y_true'])
y_pred = np.array(d['y_pred'])
y_proba = np.array(d['y_proba'])

# Per-class
prec, rec, f1, sup = precision_recall_fscore_support(
    y_true, y_pred, labels=[0, 1, 2], zero_division=0
)
per_class = pd.DataFrame({
    'class': ['A', 'D', 'H'],
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'support': sup,
})
print(f"=== {champion_name} per-class metrics ===")
print(per_class.round(3).to_string(index=False))

per_class.to_csv(os.path.join(RESULTS_DIR, 'step15_per_class.csv'), index=False)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
print(f"\n=== {champion_name} confusion matrix ===")
print(pd.DataFrame(cm, index=['true_A', 'true_D', 'true_H'],
                   columns=['pred_A', 'pred_D', 'pred_H']).to_string())

In [ ]:
lines = [
    "# Step 15 — Model Comparison",
    "",
    "## Overall comparison (7-fold CV, aggregate over all folds)",
    "",
    "| Model | Features | Log loss | Accuracy | Macro F1 |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for _, r in overall.iterrows():
    lines.append(f"| {r['model']} | {r['n_features']} | {r['log_loss']:.4f} | "
                 f"{r['accuracy']:.4f} | {r['macro_f1']:.4f} |")

lines += [
    "",
    "## Champion",
    "",
    f"**{champion_name}**",
    "",
    "- 9 delta features, all derived from earlier matches only",
    "- Logistic Regression with ElasticNet (C=0.1, l1_ratio=1.0 = pure Lasso)",
    "- Wins 5 of 7 folds against EloBaseline",
    "- Mean Δ vs EloBaseline: −0.0034",
    "- Improves meaningfully on the hardest fold (2020/21 COVID: −0.0123)",
    "",
    "## Per-class metrics (champion)",
    "",
    "| Class | Precision | Recall | F1 | Support |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for _, r in per_class.iterrows():
    lines.append(f"| {r['class']} | {r['precision']:.3f} | {r['recall']:.3f} | "
                 f"{r['f1']:.3f} | {int(r['support'])} |")

lines += [
    "",
    "## Confusion matrix (champion)",
    "",
    "| Actual \\ Predicted | A | D | H |",
    "| --- | ---: | ---: | ---: |",
    f"| A | {cm[0,0]} | {cm[0,1]} | {cm[0,2]} |",
    f"| D | {cm[1,0]} | {cm[1,1]} | {cm[1,2]} |",
    f"| H | {cm[2,0]} | {cm[2,1]} | {cm[2,2]} |",
    "",
    "## Why this is the champion",
    "",
    "1. **Best log loss** on the primary metric.",
    "2. **Consistent across folds** (5 of 7 wins).",
    "3. **Interpretable** (9 features with Lasso coefficients).",
    "4. **Small, deployable** — 9-feature LR is trivial to serve.",
    "5. **Generalizes** — no evidence of overfitting.",
    "",
    "## Honest caveats",
    "",
    "- The improvement over EloBaseline is modest (0.0035).",
    "- It is smaller than the fold-to-fold variance (~0.005).",
    "- Some seasons (2019/20, 2021/22) show EloBaseline marginally better.",
    "- The value of the champion is not a huge margin but a **consistent**",
    "  direction of improvement over a strong baseline.",
    "",
    "## Artifacts",
    "",
    "- results/step15_overall.csv",
    "- results/step15_per_class.csv",
    "- results/step15_champion_plots.png",
]
with open(os.path.join(DOCS_DIR, 'step15_model_comparison.md'), 'w') as f:
    f.write("\n".join(lines))
print("Wrote:", os.path.join(DOCS_DIR, 'step15_model_comparison.md'))

# Append to decision log
decision_log = os.path.join(DOCS_DIR, 'decision_log.md')
append_block = """

## Step 15 — Champion model selected
**Champion:** Deltas_LR_tuned (9 delta features, LR with ElasticNet, C=0.1, l1_ratio=1.0).
**Log loss (7-fold CV):** 0.9699 (mean), 0.0342 (std).
**EloBaseline (champion runner-up):** 0.9734.
**Δ:** -0.0035.
**Per-fold wins:** 5 of 7.

**Runner-up:** EloBaseline (single-feature LR on delta_elo).
**Second runner-up:** All47_RF_tuned (log loss 0.9777).

**Rationale for champion:**
- Best log loss on the primary metric.
- Consistent across folds (5/7 wins).
- Interpretable (9 features, Lasso coefficients).
- Modest but real improvement over a strong Elo baseline.

**Honest caveats:**
- The improvement (0.0035) is smaller than fold-to-fold variance.
- Not a dramatic win — a small, consistent edge.
"""
with open(decision_log, 'a') as f:
    f.write(append_block)
print("\nAppended champion decision to:", decision_log)

In [ ]:
# ============================================================
# STEP 16 — FINAL MODEL TRAINING
# Refit champion on train + val (2015/16 - 2024/25).
# Freeze. Save to models/. No test data touched.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, datetime
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(MODELS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import features
import importlib
importlib.reload(features)

# ---------- Load train_form and val_form ----------
train = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
val   = pd.read_csv(os.path.join(PROC_DIR, 'val_form.csv'),   parse_dates=['Date'])

print(f"Train: {train.shape}, Val: {val.shape}")

# ---------- Define the 9 delta features (frozen) ----------
CHAMPION_FEATURES = [
    'delta_elo',
    'delta_form_pts_last5',
    'delta_form_gd_last5',
    'delta_venue_pts_last5',
    'delta_season_ppg',
    'delta_prev_season_ppg',
    'delta_form_shots_last5',
    'delta_form_sot_last5',
    'delta_rest_days',
]

# Sanity: all features present in both train and val
missing_tr = [c for c in CHAMPION_FEATURES if c not in train.columns]
missing_va = [c for c in CHAMPION_FEATURES if c not in val.columns]
assert not missing_tr, f"Missing features in train: {missing_tr}"
assert not missing_va, f"Missing features in val: {missing_va}"
print(f"\nAll {len(CHAMPION_FEATURES)} champion features present in both splits.")

# ---------- Combine train + val ----------
combined = pd.concat([train, val], ignore_index=True)
combined = combined.sort_values('Date').reset_index(drop=True)
print(f"Combined training shape: {combined.shape}")

X_train_final = combined[CHAMPION_FEATURES].values
y_train_final = combined['FTR_int'].values
print(f"X: {X_train_final.shape}, y: {y_train_final.shape}")
print(f"Target distribution: {pd.Series(y_train_final).value_counts(normalize=True).round(4).to_dict()}")

# ---------- Fit the champion ----------
champion = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        l1_ratio=1.0,
        C=0.1,
        max_iter=3000,
        class_weight=None,
        random_state=42,
    ))
])

champion.fit(X_train_final, y_train_final)

# Sanity check: prediction sanity + in-sample performance
proba_train = champion.predict_proba(X_train_final)
from sklearn.metrics import log_loss, accuracy_score
print(f"\nIn-sample (should be ~0.965-0.970):")
print(f"  log loss:  {log_loss(y_train_final, proba_train, labels=[0,1,2]):.4f}")
print(f"  accuracy:  {accuracy_score(y_train_final, proba_train.argmax(axis=1)):.4f}")

print(f"\nCoefficient magnitudes (model.coef_):")
coefs = champion.named_steps['model'].coef_
for cls_idx, cls_name in enumerate(['A', 'D', 'H']):
    nonzero = (np.abs(coefs[cls_idx]) > 1e-6).sum()
    print(f"  Class {cls_name}: {nonzero} nonzero coefficients out of {coefs.shape[1]}")

In [ ]:
# ---------- Save ----------
model_path = os.path.join(MODELS_DIR, 'champion_final.joblib')
joblib.dump(champion, model_path)
print(f"Saved model: {model_path}")
print(f"Size: {os.path.getsize(model_path)} bytes")

# ---------- Feature list ----------
features_path = os.path.join(MODELS_DIR, 'champion_features.json')
with open(features_path, 'w') as f:
    json.dump({
        'features': CHAMPION_FEATURES,
        'n_features': len(CHAMPION_FEATURES),
        'target_map': {'A': 0, 'D': 1, 'H': 2},
        'created': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    }, f, indent=2)
print(f"Saved features: {features_path}")

# ---------- Metadata ----------
metadata = {
    'model_name': 'Deltas_LR_tuned',
    'model_class': 'sklearn.pipeline.Pipeline',
    'pipeline_steps': ['StandardScaler', 'LogisticRegression'],
    'hyperparameters': {
        'penalty': 'elasticnet',
        'solver': 'saga',
        'l1_ratio': 1.0,
        'C': 0.1,
        'max_iter': 3000,
        'class_weight': None,
        'random_state': 42,
    },
    'training': {
        'seasons': ['2015/16', '2016/17', '2017/18', '2018/19', '2019/20',
                    '2020/21', '2021/22', '2022/23', '2023/24', '2024/25'],
        'rows': int(len(combined)),
        'feature_count': len(CHAMPION_FEATURES),
    },
    'cv_performance': {
        'log_loss_mean': 0.9699,
        'log_loss_std': 0.0342,
        'elo_baseline_log_loss': 0.9734,
        'delta_vs_elo': -0.0035,
        'folds_won': '5/7',
    },
    'created_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
}
metadata_path = os.path.join(MODELS_DIR, 'champion_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata: {metadata_path}")

# ---------- SHA-256 hashes ----------
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

print("\n=== SHA-256 hashes ===")
hashes = {}
for p in [model_path, features_path, metadata_path]:
    h = sha256(p)
    hashes[os.path.basename(p)] = h
    print(f"  {os.path.basename(p):30s}  {h}")

In [ ]:
# ---------- Verify that saved model can be loaded and produces same predictions ----------
loaded = joblib.load(model_path)
proba_loaded = loaded.predict_proba(X_train_final)

# Compare
assert np.allclose(proba_train, proba_loaded, atol=1e-9), "Loaded model differs from in-memory!"
print("✅ Loaded model produces identical predictions to in-memory model.")

# Verify feature list matches
with open(features_path) as f:
    feat_meta = json.load(f)
assert feat_meta['features'] == CHAMPION_FEATURES, "Feature list mismatch!"
print("✅ Feature list matches frozen champion features.")

In [ ]:
DOCS_DIR = os.path.join(PROJECT_ROOT, 'docs')

lines = [
    "# Step 16 — Final Model Training",
    "",
    "The champion (Deltas_LR_tuned) was refit on **train + validation**",
    "(2015/16 – 2024/25, 3800 matches). Test data (2025/26) is untouched.",
    "",
    "## Champion configuration",
    "",
    "| Parameter | Value |",
    "| --- | --- |",
    "| Model | Multinomial Logistic Regression |",
    "| Penalty | ElasticNet |",
    "| Solver | saga |",
    "| C | 0.1 |",
    "| l1_ratio | 1.0 (pure Lasso) |",
    "| class_weight | None |",
    "| max_iter | 3000 |",
    "| random_state | 42 |",
    "| Features | 9 delta features |",
    "| Training rows | 3800 |",
    "",
    "## Features",
    "",
]
for f in CHAMPION_FEATURES:
    lines.append(f"- `{f}`")

lines += [
    "",
    "## In-sample performance (on train + val)",
    "",
    f"- Log loss: {log_loss(y_train_final, proba_train, labels=[0,1,2]):.4f}",
    f"- Accuracy: {accuracy_score(y_train_final, proba_train.argmax(axis=1)):.4f}",
    "",
    "## Files saved",
    "",
    "- `models/champion_final.joblib` — the fitted pipeline",
    "- `models/champion_features.json` — the exact 9 feature names",
    "- `models/champion_metadata.json` — config, training info, CV performance",
    "",
    "## SHA-256 hashes",
    "",
    "```",
]
for name, h in hashes.items():
    lines.append(f"{h}  {name}")
lines += [
    "```",
    "",
    "## Test lock rules",
    "",
    "This is the frozen model. **It will NOT be changed after Step 17.**",
    "Any post-test modification requires a new held-out evaluation set (2026/27).",
]
with open(os.path.join(DOCS_DIR, 'step16_final_model.md'), 'w') as f:
    f.write("\n".join(lines))
print("Wrote:", os.path.join(DOCS_DIR, 'step16_final_model.md'))

In [ ]:
# ============================================================
# STEP 17 — FINAL MODEL TESTING 🔒
# First and only opening of the test set.
# Build features for all three splits (chronologically), slice test.
# Load frozen champion, predict, evaluate, report.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, importlib
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (
    accuracy_score, log_loss, confusion_matrix,
    precision_recall_fscore_support
)

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import features, validation, models
importlib.reload(features)
importlib.reload(validation)
importlib.reload(models)

print("=" * 60)
print("STEP 17 — OPENING THE LOCKED TEST SET (once)")
print("=" * 60)

# ---------- Load all three splits ----------
train = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'), parse_dates=['Date'])
val   = pd.read_csv(os.path.join(PROC_DIR, 'val.csv'),   parse_dates=['Date'])
test  = pd.read_csv(os.path.join(PROC_DIR, 'test.csv'),  parse_dates=['Date'])

print(f"\nTrain: {train.shape}")
print(f"Val  : {val.shape}")
print(f"Test : {test.shape}")
print(f"\nTest seasons: {sorted(test['season_label'].unique())}")
print(f"Test date range: {test['Date'].min().date()} → {test['Date'].max().date()}")


# ---------- Build features on all three (chronological stream) ----------
combined = pd.concat([train, val, test], ignore_index=True).sort_values('Date').reset_index(drop=True)
print(f"\nCombined for feature build: {combined.shape}")

PROMOTED_TABLE = {
    '2016/17': ['Burnley','Hull','Middlesbrough'],
    '2017/18': ['Brighton','Huddersfield','Newcastle'],
    '2018/19': ['Cardiff','Fulham','Wolves'],
    '2019/20': ['Aston Villa','Norwich','Sheffield United'],
    '2020/21': ['Fulham','Leeds','West Brom'],
    '2021/22': ['Brentford','Norwich','Watford'],
    '2022/23': ['Bournemouth','Fulham',"Nott'm Forest"],
    '2023/24': ['Burnley','Luton','Sheffield United'],
    '2024/25': ['Ipswich','Leicester','Southampton'],
    '2025/26': ['Burnley','Leeds','Sunderland'],
}
promoted_for_combined = {k: v for k, v in PROMOTED_TABLE.items()
                        if k in combined['season_label'].unique()}

print("\nBuilding features on combined stream...")
roll             = features.build_rolling_form_filled(
                       combined,
                       features.compute_prev_season_form(combined),
                       features.compute_promoted_baseline(combined, promoted_for_combined, 5),
                       PROMOTED_TABLE, n=5)
prev_form_v      = features.compute_prev_season_form_venue(combined)
promoted_base_v  = features.compute_promoted_baseline_venue(combined, promoted_for_combined, 5)
venue            = features.build_venue_form_filled(combined, prev_form_v, promoted_base_v, PROMOTED_TABLE, n=5)
prev_stats_v     = features.compute_prev_season_match_stats_venue(combined)
promoted_base_ms = features.compute_promoted_baseline_match_stats(combined, promoted_for_combined, 5)
stats            = features.build_match_stats_filled(combined, prev_stats_v, promoted_base_ms, PROMOTED_TABLE, n=5)
prev_gd          = features.compute_prev_season_gd_per_game_venue(combined)
std              = features.build_season_to_date_filled(combined, prev_gd, promoted_base_v, PROMOTED_TABLE)
rest             = features.build_rest_days(combined, cap=14)
elo              = features.build_elo_features(combined)
prev_ppg         = features.build_prev_season_ppg_features(combined, prev_form_v, promoted_base_v, PROMOTED_TABLE)

feat_blocks = [roll, venue, stats, std, rest, elo, prev_ppg]
combined_feats = pd.concat([b.reset_index(drop=True) for b in feat_blocks], axis=1)
combined_feats = pd.concat([combined.reset_index(drop=True), combined_feats], axis=1)

# Deltas
deltas = features.build_deltas(combined_feats)
combined_feats = pd.concat([combined_feats.reset_index(drop=True), deltas.reset_index(drop=True)], axis=1)

# Encode target
TARGET_MAP = {"A": 0, "D": 1, "H": 2}
combined_feats['FTR_int'] = combined_feats['FTR'].map(TARGET_MAP)

# Slice out test
test_feats = combined_feats[combined_feats['season_label'] == '2025/26'].reset_index(drop=True)
print(f"\nTest features shape: {test_feats.shape}")
print(f"Test date range: {test_feats['Date'].min().date()} → {test_feats['Date'].max().date()}")

# Save test_form.csv (for Step 20/21/22 reuse)
test_feats.to_csv(os.path.join(PROC_DIR, 'test_form.csv'), index=False)
print(f"\nSaved: {os.path.join(PROC_DIR, 'test_form.csv')}")

# NaN check
champion_features = [
    'delta_elo', 'delta_form_pts_last5', 'delta_form_gd_last5',
    'delta_venue_pts_last5', 'delta_season_ppg', 'delta_prev_season_ppg',
    'delta_form_shots_last5', 'delta_form_sot_last5', 'delta_rest_days',
]
nan_test = test_feats[champion_features].isna().sum().sum()
print(f"NaN in test champion features: {nan_test}")
assert nan_test == 0, "Test features contain NaN!"
print("✅ Test features built without NaN.")

In [ ]:
# ---------- Load the frozen champion ----------
model_path = os.path.join(MODELS_DIR, 'champion_final.joblib')
feat_path  = os.path.join(MODELS_DIR, 'champion_features.json')

champion = joblib.load(model_path)
with open(feat_path) as f:
    feat_meta = json.load(f)

assert feat_meta['features'] == champion_features, "Feature list mismatch!"
print(f"Loaded champion: {feat_meta['n_features']} features")
print(f"Features: {champion_features}")

# ---------- Predict ----------
X_test = test_feats[champion_features].values
y_test = test_feats['FTR_int'].values

proba_test = champion.predict_proba(X_test)
pred_test  = proba_test.argmax(axis=1)

# ---------- Metrics ----------
acc = accuracy_score(y_test, pred_test)
ll = log_loss(y_test, proba_test, labels=[0, 1, 2])

print("\n" + "=" * 60)
print("FINAL TEST RESULT 🔒")
print("=" * 60)
print(f"\nTest log loss : {ll:.4f}")
print(f"Test accuracy : {acc:.4f}")

# Per-class
prec, rec, f1, sup = precision_recall_fscore_support(
    y_test, pred_test, labels=[0, 1, 2], zero_division=0
)
print("\nPer-class metrics:")
for i, cls in enumerate(['A', 'D', 'H']):
    print(f"  {cls}: precision={prec[i]:.3f}, recall={rec[i]:.3f}, f1={f1[i]:.3f}, support={sup[i]}")

# Confusion matrix
cm = confusion_matrix(y_test, pred_test, labels=[0, 1, 2])
print("\nConfusion matrix:")
print(pd.DataFrame(cm, index=['true_A','true_D','true_H'],
                   columns=['pred_A','pred_D','pred_H']).to_string())

In [ ]:
# ---------- Baselines for comparison ----------
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

# We need to fit baselines on train+val (same as champion), then evaluate on test.
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
val_form   = pd.read_csv(os.path.join(PROC_DIR, 'val_form.csv'),   parse_dates=['Date'])
tv = pd.concat([train_form, val_form], ignore_index=True)

y_tv = tv['FTR_int'].values
y_test_actual = test_feats['FTR_int'].values

# Most frequent
mf = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(tv), 1)), y_tv)
p_mf = mf.predict_proba(np.zeros((len(test_feats), 1)))

# Prior
pr = DummyClassifier(strategy='prior').fit(np.zeros((len(tv), 1)), y_tv)
p_pr = pr.predict_proba(np.zeros((len(test_feats), 1)))

# EloBaseline
elo_pipe = Pipeline([("s", StandardScaler()),
                     ("m", LogisticRegression(max_iter=1000, random_state=42))])
elo_pipe.fit(tv[['delta_elo']].values, y_tv)
p_elo = elo_pipe.predict_proba(test_feats[['delta_elo']].values)

# Evaluate each
print("=== Test set: baseline comparison ===")
print(f"{'Model':<20s} {'Log loss':>10s} {'Accuracy':>10s}")
print("-" * 42)
for name, p in [('MostFrequent', p_mf), ('Prior', p_pr), ('EloBaseline', p_elo)]:
    ll_b = log_loss(y_test_actual, p, labels=[0, 1, 2])
    acc_b = accuracy_score(y_test_actual, p.argmax(axis=1))
    print(f"{name:<20s} {ll_b:>10.4f} {acc_b:>10.4f}")

# Champion
print(f"{'CHAMPION':<20s} {ll:>10.4f} {acc:>10.4f}")
print()
print(f"Champion improvement over EloBaseline: {log_loss(y_test_actual, p_elo, labels=[0,1,2]) - ll:+.4f}")

In [ ]:
# ---------- Bootstrap 95% CI on test log loss ----------
np.random.seed(42)
n = len(y_test)
B = 1000
lls = []
for _ in range(B):
    idx = np.random.randint(0, n, size=n)
    ll_b = log_loss(y_test[idx], proba_test[idx], labels=[0, 1, 2])
    lls.append(ll_b)

lls = np.array(lls)
lo, hi = np.percentile(lls, [2.5, 97.5])
print(f"Test log loss:  {ll:.4f}")
print(f"95% CI (bootstrap): [{lo:.4f}, {hi:.4f}]")
print(f"Bootstrap std:    {lls.std():.4f}")

# Same for EloBaseline
lls_elo = []
for _ in range(B):
    idx = np.random.randint(0, n, size=n)
    ll_b = log_loss(y_test[idx], p_elo[idx], labels=[0, 1, 2])
    lls_elo.append(ll_b)
lls_elo = np.array(lls_elo)
lo_elo, hi_elo = np.percentile(lls_elo, [2.5, 97.5])
print(f"\nEloBaseline log loss: {log_loss(y_test_actual, p_elo, labels=[0,1,2]):.4f}")
print(f"95% CI (bootstrap):   [{lo_elo:.4f}, {hi_elo:.4f}]")

In [ ]:
# ---------- Write docs/step17_final_test.md ----------
ll_elo_test = log_loss(y_test_actual, p_elo, labels=[0, 1, 2])
acc_elo_test = accuracy_score(y_test_actual, p_elo.argmax(axis=1))

lines = [
    "# Step 17 — Final Model Testing 🔒",
    "",
    "The locked test set (2025/26, 380 matches) was opened **once** on",
    f"`{pd.Timestamp.utcnow().date()}`. The frozen champion was evaluated.",
    "",
    "**No further model changes are permitted after this evaluation.**",
    "",
    "## Test set",
    "",
    f"- Season: 2025/26",
    f"- Matches: {len(test_feats)}",
    f"- Date range: {test_feats['Date'].min().date()} → {test_feats['Date'].max().date()}",
    "",
    "## Champion performance on test",
    "",
    f"- **Log loss: {ll:.4f}**",
    f"- Accuracy: {acc:.4f}",
    f"- 95% CI (bootstrap): [{lo:.4f}, {hi:.4f}]",
    "",
    "## Per-class metrics",
    "",
    "| Class | Precision | Recall | F1 | Support |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for i, cls in enumerate(['A', 'D', 'H']):
    lines.append(f"| {cls} | {prec[i]:.3f} | {rec[i]:.3f} | {f1[i]:.3f} | {sup[i]} |")

lines += [
    "",
    "## Confusion matrix",
    "",
    "| Actual \\ Predicted | A | D | H |",
    "| --- | ---: | ---: | ---: |",
    f"| A | {cm[0,0]} | {cm[0,1]} | {cm[0,2]} |",
    f"| D | {cm[1,0]} | {cm[1,1]} | {cm[1,2]} |",
    f"| H | {cm[2,0]} | {cm[2,1]} | {cm[2,2]} |",
    "",
    "## Comparison to baselines (test set)",
    "",
    "| Model | Log loss | Accuracy |",
    "| --- | ---: | ---: |",
    f"| MostFrequent | {log_loss(y_test_actual, p_mf, labels=[0,1,2]):.4f} | {accuracy_score(y_test_actual, p_mf.argmax(axis=1)):.4f} |",
    f"| Prior | {log_loss(y_test_actual, p_pr, labels=[0,1,2]):.4f} | {accuracy_score(y_test_actual, p_pr.argmax(axis=1)):.4f} |",
    f"| EloBaseline | {ll_elo_test:.4f} | {acc_elo_test:.4f} |",
    f"| **CHAMPION** | **{ll:.4f}** | **{acc:.4f}** |",
    "",
    "## Comparison: CV vs Test",
    "",
    "| Split | Log loss | Accuracy |",
    "| --- | ---: | ---: |",
    f"| 7-fold CV (train) | 0.9699 | 0.5466 |",
    f"| Test (2025/26) | {ll:.4f} | {acc:.4f} |",
    f"| Δ (test − CV) | {ll - 0.9699:+.4f} | {acc - 0.5466:+.4f} |",
    "",
    "## Verdict",
    "",
    "See Step 18 for full interpretation.",
    "",
]

with open(os.path.join(DOCS_DIR, 'step17_final_test.md'), 'w') as f:
    f.write("\n".join(lines))
print("Wrote:", os.path.join(DOCS_DIR, 'step17_final_test.md'))

# Freeze the results as CSV
results_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred': pred_test,
    'p_A': proba_test[:, 0],
    'p_D': proba_test[:, 1],
    'p_H': proba_test[:, 2],
})
results_path = os.path.join(RESULTS_DIR, 'step17_test_predictions.csv')
results_df.to_csv(results_path, index=False)
print("Saved:", results_path)

# SHA-256 hash
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

print("\n=== Result file SHA-256 ===")
print(f"{sha256(results_path)}  step17_test_predictions.csv")

print("\n" + "=" * 60)
print("STEP 17 COMPLETE. TEST RESULT FROZEN. 🔒")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 18 — FINAL PERFORMANCE REPORT
# Compile results from Steps 15 and 17 into one honest document.
# No model changes. Analysis only.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, log_loss, confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.calibration import calibration_curve
import joblib

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# ---------- Load test predictions (the frozen result) ----------
test_pred = pd.read_csv(os.path.join(RESULTS_DIR, 'step17_test_predictions.csv'))
test_feats = pd.read_csv(os.path.join(PROC_DIR, 'test_form.csv'), parse_dates=['Date'])
champion_meta = json.load(open(os.path.join(MODELS_DIR, 'champion_metadata.json')))

y_true = test_pred['y_true'].values
y_pred = test_pred['y_pred'].values
proba  = test_pred[['p_A', 'p_D', 'p_H']].values

print("Test set:", len(test_pred), "matches")
print(f"Champion test log loss: {log_loss(y_true, proba, labels=[0,1,2]):.4f}")
print(f"Champion test accuracy: {accuracy_score(y_true, y_pred):.4f}")

In [ ]:
# ---------- Confusion matrix ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['A','D','H'], yticklabels=['A','D','H'],
            ax=axes[0], cbar=False)
axes[0].set_title("Champion — confusion matrix (test set)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# ---------- Calibration ----------
for cls_idx, cls_name in enumerate(['A', 'D', 'H']):
    y_binary = (y_true == cls_idx).astype(int)
    if y_binary.sum() > 0 and y_binary.sum() < len(y_binary):
        prob_true, prob_pred = calibration_curve(y_binary, proba[:, cls_idx],
                                                  n_bins=8, strategy='uniform')
        axes[1].plot(prob_pred, prob_true, marker='o', label=f'Class {cls_name}')
axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect')
axes[1].set_title("Champion — calibration (test set)")
axes[1].set_xlabel("Predicted probability")
axes[1].set_ylabel("Empirical frequency")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step18_confusion_calibration.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)

In [ ]:
# ---------- Base rate shift: training vs test ----------
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
val_form   = pd.read_csv(os.path.join(PROC_DIR, 'val_form.csv'),   parse_dates=['Date'])
tv_form = pd.concat([train_form, val_form], ignore_index=True)

def class_dist(series):
    return series.value_counts(normalize=True).reindex([0, 1, 2], fill_value=0)

train_dist = class_dist(tv_form['FTR_int'])
test_dist  = class_dist(test_feats['FTR_int'])

dist_df = pd.DataFrame({
    'Train+Val': train_dist,
    'Test': test_dist,
    'Δ': test_dist - train_dist,
}, index=['A', 'D', 'H'])
print("=== Class distribution shift ===")
print((dist_df * 100).round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(3)
w = 0.35
ax.bar(x - w/2, train_dist.values * 100, w, label='Train+Val', color='#1f77b4')
ax.bar(x + w/2, test_dist.values  * 100, w, label='Test 2025/26', color='#d62728')
ax.set_xticks(x); ax.set_xticklabels(['Away', 'Draw', 'Home'])
ax.set_ylabel("% of matches")
ax.set_title("Class distribution shift: train+val vs test")
ax.legend()
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step18_base_rate_shift.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)


# ---------- Per-month log loss on test ----------
test_pred_full = test_pred.copy()
test_pred_full['Date'] = test_feats['Date'].values
test_pred_full['month'] = test_pred_full['Date'].dt.to_period('M').astype(str)

monthly = []
for m, grp in test_pred_full.groupby('month'):
    if len(grp) < 5:
        continue
    monthly.append({
        'month': m,
        'n': len(grp),
        'log_loss': log_loss(grp['y_true'].values,
                             grp[['p_A','p_D','p_H']].values,
                             labels=[0, 1, 2]),
    })
monthly_df = pd.DataFrame(monthly)
print("\n=== Per-month test log loss ===")
print(monthly_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_df['month'], monthly_df['log_loss'], marker='o', color='#1f77b4')
ax.axhline(log_loss(y_true, proba, labels=[0,1,2]), color='red', linestyle='--',
           label=f"Overall: {log_loss(y_true, proba, labels=[0,1,2]):.4f}")
ax.set_ylabel("Log loss")
ax.set_title("Test set — log loss by month")
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step18_monthly_logloss.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)

In [ ]:
# ---------- Compute everything we need for the report ----------
ll_test = log_loss(y_true, proba, labels=[0, 1, 2])
acc_test = accuracy_score(y_true, y_pred)
prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred,
                                                       labels=[0,1,2],
                                                       zero_division=0)

# Baselines from Step 17
baseline_rows = [
    {'model': 'MostFrequent', 'log_loss': 20.6777, 'accuracy': 0.4263},
    {'model': 'Prior',         'log_loss': 1.0838,  'accuracy': 0.4263},
    {'model': 'EloBaseline',   'log_loss': 1.0263,  'accuracy': 0.4868},
    {'model': 'CHAMPION',      'log_loss': ll_test, 'accuracy': acc_test},
]
baseline_df = pd.DataFrame(baseline_rows)

# CV vs test
cv_ll = 0.9699
cv_acc = 0.5466

# Per-class CSV
per_class_df = pd.DataFrame({
    'class': ['A', 'D', 'H'],
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'support': sup,
})
per_class_df.to_csv(os.path.join(RESULTS_DIR, 'step18_per_class_metrics.csv'), index=False)


# ---------- Write the report ----------
report = f"""# Step 18 — Final Performance Report

**Project:** English Premier League match outcome prediction
**Model:** {champion_meta['model_name']}
**Test set:** 2025/26 EPL season (380 matches), opened once at Step 17
**Report date:** {pd.Timestamp.utcnow().date()}

---

## 1. Executive summary

We built a full machine-learning pipeline to predict English Premier League
match outcomes (Home win / Draw / Away win) from pre-match information only.
After feature engineering, cross-validation, hyperparameter tuning, and a
single evaluation on a locked test season, the champion model achieved:

- **Test log loss: {ll_test:.4f}** (95% CI [0.9771, 1.0687])
- **Test accuracy: {acc_test:.4f}**

This is a real, honest test result from a leakage-safe pipeline. It is
essentially tied with a single-feature Elo baseline (log loss 1.0263) and
clearly beats the class-prior baseline (log loss 1.0838). No model in the
project meaningfully predicts draws.

The result reproduces a known phenomenon in football analytics: EPL match
outcomes are intrinsically difficult to predict, and simple strength-based
models perform nearly as well as complex ones.

---

## 2. Methodology recap

- **Data:** 11 seasons of EPL results (2015/16 – 2025/26), 4180 matches total.
- **Splits:** Train (2015/16 – 2022/23), Val (2023/24 – 2024/25), Test
  (2025/26, locked).
- **Features:** 47 engineered features including rolling form, venue form,
  rolling match stats, season-to-date, rest days, and Elo ratings.
- **Champion features:** 9 delta features (home minus away).
- **Champion model:** Multinomial Logistic Regression with ElasticNet
  regularization (C=0.1, l1_ratio=1.0 = pure Lasso).
- **Validation:** 7-fold expanding-window time-series CV, one season per fold.
- **Test:** Single evaluation on 2025/26, model frozen before opening.

---

## 3. The test result

| Metric | Value |
| --- | --- |
| **Log loss** | **{ll_test:.4f}** |
| Accuracy | {acc_test:.4f} |
| 95% bootstrap CI (log loss) | [0.9771, 1.0687] |
| Bootstrap std | 0.0231 |

### Per-class metrics

| Class | Precision | Recall | F1 | Support |
| --- | ---: | ---: | ---: | ---: |
| Away (A) | {prec[0]:.3f} | {rec[0]:.3f} | {f1[0]:.3f} | {sup[0]} |
| Draw (D) | {prec[1]:.3f} | {rec[1]:.3f} | {f1[1]:.3f} | {sup[1]} |
| Home (H) | {prec[2]:.3f} | {rec[2]:.3f} | {f1[2]:.3f} | {sup[2]} |

### Confusion matrix (test set)

| Actual \\ Predicted | A | D | H |
| --- | ---: | ---: | ---: |
| A | {cm[0,0]} | {cm[0,1]} | {cm[0,2]} |
| D | {cm[1,0]} | {cm[1,1]} | {cm[1,2]} |
| H | {cm[2,0]} | {cm[2,1]} | {cm[2,2]} |

---

## 4. Comparison to baselines

| Model | Test log loss | Test accuracy |
| --- | ---: | ---: |
| MostFrequent | 20.6777 | 0.4263 |
| Prior | 1.0838 | 0.4263 |
| EloBaseline | 1.0263 | 0.4868 |
| **CHAMPION** | **{ll_test:.4f}** | **{acc_test:.4f}** |

The champion:
- Beats the class-prior baseline by **{1.0838 - ll_test:.4f}** log loss ({(1.0838 - ll_test)/1.0838*100:.1f}%).
- Beats the Elo baseline by **{1.0263 - ll_test:.4f}** log loss.
- Has slightly lower accuracy than EloBaseline ({acc_test:.3f} vs 0.487).

---

## 5. CV vs test gap

| Split | Log loss | Accuracy |
| --- | ---: | ---: |
| 7-fold CV (train) | {cv_ll:.4f} | {cv_acc:.4f} |
| Test (2025/26) | {ll_test:.4f} | {acc_test:.4f} |
| Δ (test − CV) | **{ll_test - cv_ll:+.4f}** | {acc_test - cv_acc:+.4f} |

The test log loss is **{ll_test - cv_ll:+.4f}** worse than CV. This is
consistent with the well-documented phenomenon that CV tends to be
slightly optimistic, especially with time-series data and distribution
shift between training-era seasons and the test season.

---

## 6. Error analysis

### 6.1 Draw prediction failure

**The model never predicts a draw on the test set.** This is the single
most important error observation.

- Draw recall: **{rec[1]:.3f}**
- Draw F1: **{f1[1]:.3f}**
- Predicted draws: **{cm[0,1] + cm[1,1] + cm[2,1]}** out of 380

Draws are the minority class (23-27% depending on season) and are
genuinely difficult to distinguish from close wins. The log-loss
objective naturally penalizes predicting a rare class that's wrong
more often than right.

### 6.2 Base-rate shift

2025/26 had a **higher draw rate** than the training period:

| Class | Train+Val | Test | Δ |
| --- | ---: | ---: | ---: |
| Away | {train_dist.iloc[0]*100:.2f}% | {test_dist.iloc[0]*100:.2f}% | {(test_dist.iloc[0]-train_dist.iloc[0])*100:+.2f}% |
| Draw | {train_dist.iloc[1]*100:.2f}% | {test_dist.iloc[1]*100:.2f}% | {(test_dist.iloc[1]-train_dist.iloc[1])*100:+.2f}% |
| Home | {train_dist.iloc[2]*100:.2f}% | {test_dist.iloc[2]*100:.2f}% | {(test_dist.iloc[2]-train_dist.iloc[2])*100:+.2f}% |

The model was trained on seasons with ~23% draws and evaluated on a
season with ~27% draws. Since the model already under-predicts draws,
this mismatch hurts performance further.

### 6.3 Monthly pattern

See `results/step18_monthly_logloss.png`. Log loss varies across months
of the season; this is normal for football and reflects changes in
match importance (late-season dead rubbers vs title races).

---

## 7. Strengths

- **Leakage-safe methodology.** Every feature uses only earlier matches.
  The test set was locked and opened once.
- **Reproducible pipeline.** Seeds fixed, feature builders versioned,
  SHA-256 hashes recorded for the champion model and test predictions.
- **Small, interpretable champion.** 9 features, Lasso-regularized
  logistic regression. Coefficients are directly interpretable.
- **Beats all non-Elo baselines** on the test set.
- **Bootstrap CIs reported** — we acknowledge the uncertainty honestly.

---

## 8. Weaknesses

- **Cannot predict draws.** Draw recall = 0. This is the biggest
  limitation and the biggest source of log loss.
- **Marginal improvement over a 1-feature Elo baseline.** The champion
  beats Elo by only 0.0026 log loss on test — within the bootstrap CI.
- **CV-to-test gap.** The 0.054 log loss degradation from CV to test
  suggests our CV was optimistic.
- **Accuracy dropped more than log loss** (0.547 → 0.471), reflecting
  the base-rate shift in 2025/26.

---

## 9. Limitations

- No odds data (removed by design — this is a football-only project).
- No injuries, lineups, xG, managerial changes, or transfers.
- Only 11 seasons of history — small by ML standards.
- Single test season (380 matches) limits statistical precision.
- Elo hyperparameters not exhaustively tuned.
- Promoted teams handled with a simple promoted-baseline fill.

---

## 10. Possible improvements

1. **Add odds data as features.** Strongly predictive, but excluded by
   design in this project.
2. **Draw-specific modeling.** A separate binary classifier for "draw
   vs not-draw" combined with a 2-way classifier for the winner.
3. **Richer Elo variants.** Home/away split Elo, team-specific Elo
   decay rates, surprise-weighted updates.
4. **Team identity features.** Target-encoded team strength (with
   proper cross-fold leakage guards).
5. **More seasons.** Expanding to 20+ seasons would give the model
   more context, especially for rare events like title races.
6. **Hierarchical models.** Partial pooling for teams with limited
   history (e.g., promoted clubs).

---

## 11. Final verdict

The champion model performs **{ll_test:.4f}** log loss on the locked
2025/26 test season. It is a real, defensible result from a clean
pipeline, and it reproduces the well-known phenomenon that EPL match
outcomes are hard to predict and simple Elo-style models are extremely
competitive.

The project's value is not in beating the market. It is in:

- Building a **complete, leakage-safe ML pipeline** end to end.
- Making **honest, evidence-based decisions** at every step.
- Reporting the **truth**, including the parts that are unflattering.
- Producing an **interpretable, deployable** model with documented limits.

This is what a professional football-analytics project looks like.
"""

report_path = os.path.join(DOCS_DIR, 'step18_final_report.md')
with open(report_path, 'w') as f:
    f.write(report)
print("Wrote:", report_path)
print("Size:", os.path.getsize(report_path), "bytes")

In [ ]:
# ---------- Base rate shift: training vs test ----------
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
val_form   = pd.read_csv(os.path.join(PROC_DIR, 'val_form.csv'),   parse_dates=['Date'])
tv_form = pd.concat([train_form, val_form], ignore_index=True)

def class_dist(series):
    """Return normalized distribution over classes {0:A, 1:D, 2:H}, index as int."""
    s = pd.Series(series).dropna().astype(int)
    dist = s.value_counts(normalize=True).sort_index()
    return dist.reindex([0, 1, 2], fill_value=0)

train_dist = class_dist(tv_form['FTR_int'])
test_dist  = class_dist(test_feats['FTR_int'])

dist_df = pd.DataFrame({
    'Train+Val': train_dist,
    'Test': test_dist,
    'Delta': test_dist - train_dist,
}, index=['A', 'D', 'H'])
print("=== Class distribution shift ===")
print((dist_df * 100).round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(3)
w = 0.35
ax.bar(x - w/2, train_dist.values * 100, w, label='Train+Val', color='#1f77b4')
ax.bar(x + w/2, test_dist.values  * 100, w, label='Test 2025/26', color='#d62728')
ax.set_xticks(x); ax.set_xticklabels(['Away', 'Draw', 'Home'])
ax.set_ylabel("% of matches")
ax.set_title("Class distribution shift: train+val vs test")
ax.legend()
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step18_base_rate_shift.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)


# ---------- Per-month log loss on test ----------
test_pred_full = test_pred.copy()
test_pred_full['Date'] = test_feats['Date'].values
test_pred_full['month'] = test_pred_full['Date'].dt.to_period('M').astype(str)

monthly = []
for m, grp in test_pred_full.groupby('month'):
    if len(grp) < 5:
        continue
    monthly.append({
        'month': m,
        'n': len(grp),
        'log_loss': log_loss(grp['y_true'].values,
                             grp[['p_A','p_D','p_H']].values,
                             labels=[0, 1, 2]),
    })
monthly_df = pd.DataFrame(monthly)
print("\n=== Per-month test log loss ===")
print(monthly_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_df['month'], monthly_df['log_loss'], marker='o', color='#1f77b4')
ax.axhline(log_loss(y_true, proba, labels=[0,1,2]), color='red', linestyle='--',
           label=f"Overall: {log_loss(y_true, proba, labels=[0,1,2]):.4f}")
ax.set_ylabel("Log loss")
ax.set_title("Test set — log loss by month")
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step18_monthly_logloss.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)

In [ ]:
# ============================================================
# STEP 18 — FINAL PERFORMANCE REPORT
# Compile results from Steps 15 and 17 into one honest document.
# No model changes. Analysis only.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, log_loss, confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.calibration import calibration_curve
import joblib

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# ---------- Load test predictions (the frozen result) ----------
test_pred = pd.read_csv(os.path.join(RESULTS_DIR, 'step17_test_predictions.csv'))
test_feats = pd.read_csv(os.path.join(PROC_DIR, 'test_form.csv'), parse_dates=['Date'])
champion_meta = json.load(open(os.path.join(MODELS_DIR, 'champion_metadata.json')))

y_true = test_pred['y_true'].values
y_pred = test_pred['y_pred'].values
proba  = test_pred[['p_A', 'p_D', 'p_H']].values

print("Test set:", len(test_pred), "matches")
print(f"Champion test log loss: {log_loss(y_true, proba, labels=[0,1,2]):.4f}")
print(f"Champion test accuracy: {accuracy_score(y_true, y_pred):.4f}")

In [ ]:
# ---------- Confusion matrix + calibration ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['A','D','H'], yticklabels=['A','D','H'],
            ax=axes[0], cbar=False)
axes[0].set_title("Champion — confusion matrix (test set)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

for cls_idx, cls_name in enumerate(['A', 'D', 'H']):
    y_binary = (y_true == cls_idx).astype(int)
    if 0 < y_binary.sum() < len(y_binary):
        prob_true, prob_pred = calibration_curve(y_binary, proba[:, cls_idx],
                                                  n_bins=8, strategy='uniform')
        axes[1].plot(prob_pred, prob_true, marker='o', label=f'Class {cls_name}')
axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect')
axes[1].set_title("Champion — calibration (test set)")
axes[1].set_xlabel("Predicted probability")
axes[1].set_ylabel("Empirical frequency")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step18_confusion_calibration.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)

In [ ]:
# ---------- Base rate shift: training vs test ----------
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
val_form   = pd.read_csv(os.path.join(PROC_DIR, 'val_form.csv'),   parse_dates=['Date'])
tv_form = pd.concat([train_form, val_form], ignore_index=True)

def class_dist(series):
    """Return normalized distribution over classes {0:A, 1:D, 2:H}, index as int."""
    s = pd.Series(series).dropna().astype(int)
    dist = s.value_counts(normalize=True).sort_index()
    return dist.reindex([0, 1, 2], fill_value=0)

train_dist = class_dist(tv_form['FTR_int'])
test_dist  = class_dist(test_feats['FTR_int'])

dist_df = pd.DataFrame({
    'Train+Val': train_dist,
    'Test': test_dist,
    'Delta': test_dist - train_dist,
}, index=['A', 'D', 'H'])
print("=== Class distribution shift ===")
print((dist_df * 100).round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(3)
w = 0.35
ax.bar(x - w/2, train_dist.values * 100, w, label='Train+Val', color='#1f77b4')
ax.bar(x + w/2, test_dist.values  * 100, w, label='Test 2025/26', color='#d62728')
ax.set_xticks(x); ax.set_xticklabels(['Away', 'Draw', 'Home'])
ax.set_ylabel("% of matches")
ax.set_title("Class distribution shift: train+val vs test")
ax.legend()
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step18_base_rate_shift.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)


# ---------- Per-month log loss on test ----------
test_pred_full = test_pred.copy()
test_pred_full['Date'] = test_feats['Date'].values
test_pred_full['month'] = test_pred_full['Date'].dt.to_period('M').astype(str)

monthly = []
for m, grp in test_pred_full.groupby('month'):
    if len(grp) < 5:
        continue
    monthly.append({
        'month': m,
        'n': len(grp),
        'log_loss': log_loss(grp['y_true'].values,
                             grp[['p_A','p_D','p_H']].values,
                             labels=[0, 1, 2]),
    })
monthly_df = pd.DataFrame(monthly)
print("\n=== Per-month test log loss ===")
print(monthly_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_df['month'], monthly_df['log_loss'], marker='o', color='#1f77b4')
ax.axhline(log_loss(y_true, proba, labels=[0,1,2]), color='red', linestyle='--',
           label=f"Overall: {log_loss(y_true, proba, labels=[0,1,2]):.4f}")
ax.set_ylabel("Log loss")
ax.set_title("Test set — log loss by month")
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step18_monthly_logloss.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)

In [ ]:
# ---------- Compute everything we need for the report ----------
ll_test = log_loss(y_true, proba, labels=[0, 1, 2])
acc_test = accuracy_score(y_true, y_pred)
prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred,
                                                       labels=[0,1,2],
                                                       zero_division=0)

cv_ll  = 0.9699
cv_acc = 0.5466
ll_elo_test  = 1.0263
acc_elo_test = 0.4868
ll_prior_test = 1.0838
acc_prior_test = 0.4263

# Per-class CSV
per_class_df = pd.DataFrame({
    'class': ['A', 'D', 'H'],
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'support': sup,
})
per_class_df.to_csv(os.path.join(RESULTS_DIR, 'step18_per_class_metrics.csv'), index=False)
print("Saved per-class CSV.")

# Build report
report = f"""# Step 18 — Final Performance Report

**Project:** English Premier League match outcome prediction
**Model:** {champion_meta['model_name']}
**Test set:** 2025/26 EPL season (380 matches), opened once at Step 17
**Report date:** {pd.Timestamp.utcnow().date()}

---

## 1. Executive summary

We built a full machine-learning pipeline to predict English Premier League
match outcomes (Home win / Draw / Away win) from pre-match information only.
After feature engineering, cross-validation, hyperparameter tuning, and a
single evaluation on a locked test season, the champion model achieved:

- **Test log loss: {ll_test:.4f}** (95% CI [0.9771, 1.0687])
- **Test accuracy: {acc_test:.4f}**

This is a real, honest test result from a leakage-safe pipeline. It is
essentially tied with a single-feature Elo baseline (log loss 1.0263) and
clearly beats the class-prior baseline (log loss 1.0838). No model in the
project meaningfully predicts draws.

The result reproduces a known phenomenon in football analytics: EPL match
outcomes are intrinsically difficult to predict, and simple strength-based
models perform nearly as well as complex ones.

---

## 2. Methodology recap

- **Data:** 11 seasons of EPL results (2015/16 - 2025/26), 4180 matches total.
- **Splits:** Train (2015/16 - 2022/23), Val (2023/24 - 2024/25), Test
  (2025/26, locked).
- **Features:** 47 engineered features including rolling form, venue form,
  rolling match stats, season-to-date, rest days, and Elo ratings.
- **Champion features:** 9 delta features (home minus away).
- **Champion model:** Multinomial Logistic Regression with ElasticNet
  regularization (C=0.1, l1_ratio=1.0 = pure Lasso).
- **Validation:** 7-fold expanding-window time-series CV, one season per fold.
- **Test:** Single evaluation on 2025/26, model frozen before opening.

---

## 3. The test result

| Metric | Value |
| --- | --- |
| **Log loss** | **{ll_test:.4f}** |
| Accuracy | {acc_test:.4f} |
| 95% bootstrap CI (log loss) | [0.9771, 1.0687] |
| Bootstrap std | 0.0231 |

### Per-class metrics

| Class | Precision | Recall | F1 | Support |
| --- | ---: | ---: | ---: | ---: |
| Away (A) | {prec[0]:.3f} | {rec[0]:.3f} | {f1[0]:.3f} | {sup[0]} |
| Draw (D) | {prec[1]:.3f} | {rec[1]:.3f} | {f1[1]:.3f} | {sup[1]} |
| Home (H) | {prec[2]:.3f} | {rec[2]:.3f} | {f1[2]:.3f} | {sup[2]} |

### Confusion matrix (test set)

| Actual \\ Predicted | A | D | H |
| --- | ---: | ---: | ---: |
| A | {cm[0,0]} | {cm[0,1]} | {cm[0,2]} |
| D | {cm[1,0]} | {cm[1,1]} | {cm[1,2]} |
| H | {cm[2,0]} | {cm[2,1]} | {cm[2,2]} |

---

## 4. Comparison to baselines

| Model | Test log loss | Test accuracy |
| --- | ---: | ---: |
| MostFrequent | 20.6777 | 0.4263 |
| Prior | {ll_prior_test:.4f} | {acc_prior_test:.4f} |
| EloBaseline | {ll_elo_test:.4f} | {acc_elo_test:.4f} |
| **CHAMPION** | **{ll_test:.4f}** | **{acc_test:.4f}** |

The champion:
- Beats the class-prior baseline by **{ll_prior_test - ll_test:.4f}** log loss
  ({(ll_prior_test - ll_test)/ll_prior_test*100:.1f}%).
- Beats the Elo baseline by **{ll_elo_test - ll_test:.4f}** log loss.
- Has slightly lower accuracy than EloBaseline ({acc_test:.3f} vs {acc_elo_test:.3f}).

---

## 5. CV vs test gap

| Split | Log loss | Accuracy |
| --- | ---: | ---: |
| 7-fold CV (train) | {cv_ll:.4f} | {cv_acc:.4f} |
| Test (2025/26) | {ll_test:.4f} | {acc_test:.4f} |
| Delta (test - CV) | **{ll_test - cv_ll:+.4f}** | {acc_test - cv_acc:+.4f} |

The test log loss is **{ll_test - cv_ll:+.4f}** worse than CV. This is
consistent with the well-documented phenomenon that CV tends to be
slightly optimistic, especially with time-series data and distribution
shift between training-era seasons and the test season.

---

## 6. Error analysis

### 6.1 Draw prediction failure

**The model never predicts a draw on the test set.** This is the single
most important error observation.

- Draw recall: **{rec[1]:.3f}**
- Draw F1: **{f1[1]:.3f}**
- Predicted draws: **{cm[0,1] + cm[1,1] + cm[2,1]}** out of 380

Draws are the minority class (23-27% depending on season) and are
genuinely difficult to distinguish from close wins. The log-loss
objective naturally penalizes predicting a rare class that is wrong
more often than right.

### 6.2 Base-rate shift

2025/26 had a higher draw rate than the training period:

| Class | Train+Val | Test | Delta |
| --- | ---: | ---: | ---: |
| Away | {train_dist.iloc[0]*100:.2f}% | {test_dist.iloc[0]*100:.2f}% | {(test_dist.iloc[0]-train_dist.iloc[0])*100:+.2f}% |
| Draw | {train_dist.iloc[1]*100:.2f}% | {test_dist.iloc[1]*100:.2f}% | {(test_dist.iloc[1]-train_dist.iloc[1])*100:+.2f}% |
| Home | {train_dist.iloc[2]*100:.2f}% | {test_dist.iloc[2]*100:.2f}% | {(test_dist.iloc[2]-train_dist.iloc[2])*100:+.2f}% |

The model was trained on seasons with ~23% draws and evaluated on a
season with ~27% draws. Since the model already under-predicts draws,
this mismatch hurts performance further.

### 6.3 Monthly pattern

See `results/step18_monthly_logloss.png`. Log loss varies across months
of the season; this is normal for football and reflects changes in
match importance (late-season dead rubbers vs title races).

---

## 7. Strengths

- **Leakage-safe methodology.** Every feature uses only earlier matches.
  The test set was locked and opened once.
- **Reproducible pipeline.** Seeds fixed, feature builders versioned,
  SHA-256 hashes recorded for the champion model and test predictions.
- **Small, interpretable champion.** 9 features, Lasso-regularized
  logistic regression. Coefficients are directly interpretable.
- **Beats all non-Elo baselines** on the test set.
- **Bootstrap CIs reported** - we acknowledge the uncertainty honestly.

---

## 8. Weaknesses

- **Cannot predict draws.** Draw recall = 0. This is the biggest
  limitation and the biggest source of log loss.
- **Marginal improvement over a 1-feature Elo baseline.** The champion
  beats Elo by only {ll_elo_test - ll_test:.4f} log loss on test -
  within the bootstrap CI.
- **CV-to-test gap.** The {ll_test - cv_ll:.4f} log loss degradation from
  CV to test suggests our CV was optimistic.
- **Accuracy dropped more than log loss** ({cv_acc:.3f} -> {acc_test:.3f}),
  reflecting the base-rate shift in 2025/26.

---

## 9. Limitations

- No odds data (removed by design - this is a football-only project).
- No injuries, lineups, xG, managerial changes, or transfers.
- Only 11 seasons of history - small by ML standards.
- Single test season (380 matches) limits statistical precision.
- Elo hyperparameters not exhaustively tuned.
- Promoted teams handled with a simple promoted-baseline fill.

---

## 10. Possible improvements

1. **Add odds data as features.** Strongly predictive, but excluded by
   design in this project.
2. **Draw-specific modeling.** A separate binary classifier for
   "draw vs not-draw" combined with a 2-way classifier for the winner.
3. **Richer Elo variants.** Home/away split Elo, team-specific Elo
   decay rates, surprise-weighted updates.
4. **Team identity features.** Target-encoded team strength (with
   proper cross-fold leakage guards).
5. **More seasons.** Expanding to 20+ seasons would give the model
   more context, especially for rare events like title races.
6. **Hierarchical models.** Partial pooling for teams with limited
   history (e.g., promoted clubs).

---

## 11. Final verdict

The champion model performs **{ll_test:.4f}** log loss on the locked
2025/26 test season. It is a real, defensible result from a clean
pipeline, and it reproduces the well-known phenomenon that EPL match
outcomes are hard to predict and simple Elo-style models are extremely
competitive.

The project's value is not in beating the market. It is in:

- Building a complete, leakage-safe ML pipeline end to end.
- Making honest, evidence-based decisions at every step.
- Reporting the truth, including the parts that are unflattering.
- Producing an interpretable, deployable model with documented limits.

This is what a professional football-analytics project looks like.
"""

report_path = os.path.join(DOCS_DIR, 'step18_final_report.md')
with open(report_path, 'w') as f:
    f.write(report)
print("Wrote:", report_path)
print("Size:", os.path.getsize(report_path), "bytes")

In [ ]:
# ---------- DIAGNOSTIC: what's actually in FTR_int? ----------
print("=== tv_form['FTR_int'] ===")
print("dtype:", tv_form['FTR_int'].dtype)
print("unique:", sorted(tv_form['FTR_int'].dropna().unique().tolist()))
print("value_counts:")
print(tv_form['FTR_int'].value_counts(dropna=False).to_string())
print()

print("=== test_feats['FTR_int'] ===")
print("dtype:", test_feats['FTR_int'].dtype)
print("unique:", sorted(test_feats['FTR_int'].dropna().unique().tolist()))
print("value_counts:")
print(test_feats['FTR_int'].value_counts(dropna=False).to_string())

In [ ]:
# ---------- Base rate shift: training vs test ----------
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
val_form   = pd.read_csv(os.path.join(PROC_DIR, 'val_form.csv'),   parse_dates=['Date'])
tv_form = pd.concat([train_form, val_form], ignore_index=True)

# Explicit and simple — no custom function
tv_counts = tv_form['FTR_int'].value_counts(normalize=True).sort_index()
te_counts = test_feats['FTR_int'].value_counts(normalize=True).sort_index()

# Force the index to be exactly [0, 1, 2] in that order
tv_counts = tv_counts.reindex([0, 1, 2]).fillna(0.0)
te_counts = te_counts.reindex([0, 1, 2]).fillna(0.0)

# Debug print first
print("tv_counts index:", tv_counts.index.tolist(), "values:", tv_counts.tolist())
print("te_counts index:", te_counts.index.tolist(), "values:", te_counts.tolist())
print()

# Build the table explicitly
dist_df = pd.DataFrame({
    'Train+Val': [tv_counts[0], tv_counts[1], tv_counts[2]],
    'Test':      [te_counts[0], te_counts[1], te_counts[2]],
}, index=['A', 'D', 'H'])
dist_df['Delta'] = dist_df['Test'] - dist_df['Train+Val']

print("=== Class distribution shift ===")
print((dist_df * 100).round(2).to_string())

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(3)
w = 0.35
ax.bar(x - w/2, dist_df['Train+Val'].values * 100, w, label='Train+Val', color='#1f77b4')
ax.bar(x + w/2, dist_df['Test'].values  * 100, w, label='Test 2025/26', color='#d62728')
ax.set_xticks(x); ax.set_xticklabels(['Away', 'Draw', 'Home'])
ax.set_ylabel("% of matches")
ax.set_title("Class distribution shift: train+val vs test")
ax.legend()
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step18_base_rate_shift.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)


# ---------- Per-month log loss on test (unchanged) ----------
test_pred_full = test_pred.copy()
test_pred_full['Date'] = test_feats['Date'].values
test_pred_full['month'] = test_pred_full['Date'].dt.to_period('M').astype(str)

monthly = []
for m, grp in test_pred_full.groupby('month'):
    if len(grp) < 5:
        continue
    monthly.append({
        'month': m,
        'n': len(grp),
        'log_loss': log_loss(grp['y_true'].values,
                             grp[['p_A','p_D','p_H']].values,
                             labels=[0, 1, 2]),
    })
monthly_df = pd.DataFrame(monthly)
print("\n=== Per-month test log loss ===")
print(monthly_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_df['month'], monthly_df['log_loss'], marker='o', color='#1f77b4')
ax.axhline(log_loss(y_true, proba, labels=[0,1,2]), color='red', linestyle='--',
           label=f"Overall: {log_loss(y_true, proba, labels=[0,1,2]):.4f}")
ax.set_ylabel("Log loss")
ax.set_title("Test set — log loss by month")
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step18_monthly_logloss.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)

In [ ]:
# ---------- Compute everything we need for the report ----------
ll_test = log_loss(y_true, proba, labels=[0, 1, 2])
acc_test = accuracy_score(y_true, y_pred)
prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred,
                                                       labels=[0,1,2],
                                                       zero_division=0)

cv_ll  = 0.9699
cv_acc = 0.5466
ll_elo_test  = 1.0263
acc_elo_test = 0.4868
ll_prior_test = 1.0838
acc_prior_test = 0.4263

# Base rates from dist_df
tv_A, tv_D, tv_H = dist_df.loc['A', 'Train+Val'], dist_df.loc['D', 'Train+Val'], dist_df.loc['H', 'Train+Val']
te_A, te_D, te_H = dist_df.loc['A', 'Test'],      dist_df.loc['D', 'Test'],      dist_df.loc['H', 'Test']

# Per-class CSV
per_class_df = pd.DataFrame({
    'class': ['A', 'D', 'H'],
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'support': sup,
})
per_class_df.to_csv(os.path.join(RESULTS_DIR, 'step18_per_class_metrics.csv'), index=False)
print("Saved per-class CSV.")

# Build report
report = f"""# Step 18 — Final Performance Report

**Project:** English Premier League match outcome prediction
**Model:** {champion_meta['model_name']}
**Test set:** 2025/26 EPL season (380 matches), opened once at Step 17
**Report date:** {pd.Timestamp.utcnow().date()}

---

## 1. Executive summary

We built a full machine-learning pipeline to predict English Premier League
match outcomes (Home win / Draw / Away win) from pre-match information only.
After feature engineering, cross-validation, hyperparameter tuning, and a
single evaluation on a locked test season, the champion model achieved:

- **Test log loss: {ll_test:.4f}** (95% CI [0.9771, 1.0687])
- **Test accuracy: {acc_test:.4f}**

This is a real, honest test result from a leakage-safe pipeline. It is
essentially tied with a single-feature Elo baseline (log loss 1.0263) and
clearly beats the class-prior baseline (log loss 1.0838). No model in the
project meaningfully predicts draws.

The result reproduces a known phenomenon in football analytics: EPL match
outcomes are intrinsically difficult to predict, and simple strength-based
models perform nearly as well as complex ones.

---

## 2. Methodology recap

- **Data:** 11 seasons of EPL results (2015/16 - 2025/26), 4180 matches total.
- **Splits:** Train (2015/16 - 2022/23), Val (2023/24 - 2024/25), Test
  (2025/26, locked).
- **Features:** 47 engineered features including rolling form, venue form,
  rolling match stats, season-to-date, rest days, and Elo ratings.
- **Champion features:** 9 delta features (home minus away).
- **Champion model:** Multinomial Logistic Regression with ElasticNet
  regularization (C=0.1, l1_ratio=1.0 = pure Lasso).
- **Validation:** 7-fold expanding-window time-series CV, one season per fold.
- **Test:** Single evaluation on 2025/26, model frozen before opening.

---

## 3. The test result

| Metric | Value |
| --- | --- |
| **Log loss** | **{ll_test:.4f}** |
| Accuracy | {acc_test:.4f} |
| 95% bootstrap CI (log loss) | [0.9771, 1.0687] |
| Bootstrap std | 0.0231 |

### Per-class metrics

| Class | Precision | Recall | F1 | Support |
| --- | ---: | ---: | ---: | ---: |
| Away (A) | {prec[0]:.3f} | {rec[0]:.3f} | {f1[0]:.3f} | {sup[0]} |
| Draw (D) | {prec[1]:.3f} | {rec[1]:.3f} | {f1[1]:.3f} | {sup[1]} |
| Home (H) | {prec[2]:.3f} | {rec[2]:.3f} | {f1[2]:.3f} | {sup[2]} |

### Confusion matrix (test set)

| Actual \\ Predicted | A | D | H |
| --- | ---: | ---: | ---: |
| A | {cm[0,0]} | {cm[0,1]} | {cm[0,2]} |
| D | {cm[1,0]} | {cm[1,1]} | {cm[1,2]} |
| H | {cm[2,0]} | {cm[2,1]} | {cm[2,2]} |

---

## 4. Comparison to baselines

| Model | Test log loss | Test accuracy |
| --- | ---: | ---: |
| MostFrequent | 20.6777 | 0.4263 |
| Prior | {ll_prior_test:.4f} | {acc_prior_test:.4f} |
| EloBaseline | {ll_elo_test:.4f} | {acc_elo_test:.4f} |
| **CHAMPION** | **{ll_test:.4f}** | **{acc_test:.4f}** |

The champion:
- Beats the class-prior baseline by **{ll_prior_test - ll_test:.4f}** log loss
  ({(ll_prior_test - ll_test)/ll_prior_test*100:.1f}%).
- Beats the Elo baseline by **{ll_elo_test - ll_test:.4f}** log loss.
- Has slightly lower accuracy than EloBaseline ({acc_test:.3f} vs {acc_elo_test:.3f}).

---

## 5. CV vs test gap

| Split | Log loss | Accuracy |
| --- | ---: | ---: |
| 7-fold CV (train) | {cv_ll:.4f} | {cv_acc:.4f} |
| Test (2025/26) | {ll_test:.4f} | {acc_test:.4f} |
| Delta (test - CV) | **{ll_test - cv_ll:+.4f}** | {acc_test - cv_acc:+.4f} |

The test log loss is **{ll_test - cv_ll:+.4f}** worse than CV. This is
consistent with the well-documented phenomenon that CV tends to be
slightly optimistic, especially with time-series data and distribution
shift between training-era seasons and the test season.

---

## 6. Error analysis

### 6.1 Draw prediction failure

**The model never predicts a draw on the test set.** This is the single
most important error observation.

- Draw recall: **{rec[1]:.3f}**
- Draw F1: **{f1[1]:.3f}**
- Predicted draws: **{cm[0,1] + cm[1,1] + cm[2,1]}** out of 380

Draws are the minority class (23-27% depending on season) and are
genuinely difficult to distinguish from close wins. The log-loss
objective naturally penalizes predicting a rare class that is wrong
more often than right.

### 6.2 Base-rate shift

2025/26 had a higher draw rate than the training period:

| Class | Train+Val | Test | Delta |
| --- | ---: | ---: | ---: |
| Away | {tv_A*100:.2f}% | {te_A*100:.2f}% | {(te_A-tv_A)*100:+.2f}% |
| Draw | {tv_D*100:.2f}% | {te_D*100:.2f}% | {(te_D-tv_D)*100:+.2f}% |
| Home | {tv_H*100:.2f}% | {te_H*100:.2f}% | {(te_H-tv_H)*100:+.2f}% |

The model was trained on seasons with ~23% draws and evaluated on a
season with ~27% draws. Since the model already under-predicts draws,
this mismatch hurts performance further.

### 6.3 Monthly pattern

See `results/step18_monthly_logloss.png`. Log loss varies across months
of the season; this is normal for football and reflects changes in
match importance (late-season dead rubbers vs title races).

---

## 7. Strengths

- **Leakage-safe methodology.** Every feature uses only earlier matches.
  The test set was locked and opened once.
- **Reproducible pipeline.** Seeds fixed, feature builders versioned,
  SHA-256 hashes recorded for the champion model and test predictions.
- **Small, interpretable champion.** 9 features, Lasso-regularized
  logistic regression. Coefficients are directly interpretable.
- **Beats all non-Elo baselines** on the test set.
- **Bootstrap CIs reported** - we acknowledge the uncertainty honestly.

---

## 8. Weaknesses

- **Cannot predict draws.** Draw recall = 0. This is the biggest
  limitation and the biggest source of log loss.
- **Marginal improvement over a 1-feature Elo baseline.** The champion
  beats Elo by only {ll_elo_test - ll_test:.4f} log loss on test -
  within the bootstrap CI.
- **CV-to-test gap.** The {ll_test - cv_ll:.4f} log loss degradation from
  CV to test suggests our CV was optimistic.
- **Accuracy dropped more than log loss** ({cv_acc:.3f} -> {acc_test:.3f}),
  reflecting the base-rate shift in 2025/26.

---

## 9. Limitations

- No odds data (removed by design - this is a football-only project).
- No injuries, lineups, xG, managerial changes, or transfers.
- Only 11 seasons of history - small by ML standards.
- Single test season (380 matches) limits statistical precision.
- Elo hyperparameters not exhaustively tuned.
- Promoted teams handled with a simple promoted-baseline fill.

---

## 10. Possible improvements

1. **Add odds data as features.** Strongly predictive, but excluded by
   design in this project.
2. **Draw-specific modeling.** A separate binary classifier for
   "draw vs not-draw" combined with a 2-way classifier for the winner.
3. **Richer Elo variants.** Home/away split Elo, team-specific Elo
   decay rates, surprise-weighted updates.
4. **Team identity features.** Target-encoded team strength (with
   proper cross-fold leakage guards).
5. **More seasons.** Expanding to 20+ seasons would give the model
   more context, especially for rare events like title races.
6. **Hierarchical models.** Partial pooling for teams with limited
   history (e.g., promoted clubs).

---

## 11. Final verdict

The champion model performs **{ll_test:.4f}** log loss on the locked
2025/26 test season. It is a real, defensible result from a clean
pipeline, and it reproduces the well-known phenomenon that EPL match
outcomes are hard to predict and simple Elo-style models are extremely
competitive.

The project's value is not in beating the market. It is in:

- Building a complete, leakage-safe ML pipeline end to end.
- Making honest, evidence-based decisions at every step.
- Reporting the truth, including the parts that are unflattering.
- Producing an interpretable, deployable model with documented limits.

This is what a professional football-analytics project looks like.
"""

report_path = os.path.join(DOCS_DIR, 'step18_final_report.md')
with open(report_path, 'w') as f:
    f.write(report)
print("Wrote:", report_path)
print("Size:", os.path.getsize(report_path), "bytes")

In [ ]:
# ============================================================
# STEP 19 — REGULARIZATION ANALYSIS (analysis only, no re-tuning)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style='whitegrid')

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import validation
importlib.reload(validation)

# ---------- Load champion ----------
champion = joblib.load(os.path.join(MODELS_DIR, 'champion_final.joblib'))
features_meta = json.load(open(os.path.join(MODELS_DIR, 'champion_features.json')))
CHAMPION_FEATURES = features_meta['features']

# ---------- Extract coefficients ----------
coefs = champion.named_steps['model'].coef_    # shape (3, 9)
intercepts = champion.named_steps['model'].intercept_

coef_df = pd.DataFrame(coefs.T, index=CHAMPION_FEATURES, columns=['A', 'D', 'H'])
coef_df['max_abs'] = coef_df.abs().max(axis=1)
coef_df['n_nonzero'] = (coef_df[['A','D','H']].abs() > 1e-6).sum(axis=1)

print("=== Champion coefficients ===")
print(coef_df.round(4).to_string())
print()
print(f"Intercepts: A={intercepts[0]:.4f}, D={intercepts[1]:.4f}, H={intercepts[2]:.4f}")
print(f"\nZeroed features: {coef_df[coef_df['max_abs'] < 1e-6].index.tolist()}")

coef_df.to_csv(os.path.join(RESULTS_DIR, 'step19_champion_coefs.csv'))
print("\nSaved coefficients CSV.")

In [ ]:
# ---------- Regularization path ----------
train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
val_form   = pd.read_csv(os.path.join(PROC_DIR, 'val_form.csv'),   parse_dates=['Date'])
tv = pd.concat([train_form, val_form], ignore_index=True).reset_index(drop=True)

X_tv = tv[CHAMPION_FEATURES].values
y_tv = tv['FTR_int'].values

# Fit scaler once — so coefficients are on the same scale across all C
scaler = StandardScaler().fit(X_tv)
X_tv_scaled = scaler.transform(X_tv)

C_range = np.logspace(-3, 1, 25)   # 0.001 to 10, 25 points

# Store paths per class
coef_paths = {cls: [] for cls in ['A', 'D', 'H']}
for C in C_range:
    lr = LogisticRegression(penalty='l1', solver='saga',
                            C=C, max_iter=3000, random_state=42)
    lr.fit(X_tv_scaled, y_tv)
    for ci, cls in enumerate(['A', 'D', 'H']):
        coef_paths[cls].append(lr.coef_[ci])

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
class_color_map = plt.cm.tab10(np.linspace(0, 1, len(CHAMPION_FEATURES)))

for ax, cls in zip(axes, ['A', 'D', 'H']):
    path = np.array(coef_paths[cls])   # shape (n_C, n_features)
    for fi, feat in enumerate(CHAMPION_FEATURES):
        ax.plot(C_range, path[:, fi], linewidth=1.8,
                color=class_color_map[fi], label=feat)
    ax.set_xscale('log')
    ax.axvline(0.1, color='red', linestyle='--', alpha=0.6,
               label='Champion C=0.1')
    ax.set_xlabel("C (inverse regularization)")
    ax.set_ylabel("Coefficient (standardized)")
    ax.set_title(f"Class {cls}")
    ax.grid(alpha=0.3)
    ax.axhline(0, color='gray', linewidth=0.5)

# Legend only once, off to the side
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

plt.tight_layout()
path_plot = os.path.join(RESULTS_DIR, 'step19_regularization_path.png')
plt.savefig(path_plot, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", path_plot)

# Print sparsity at a few C values
print("\n=== Sparsity across C (number of nonzero coefficients out of 9 per class) ===")
print(f"{'C':>8s}  {'A':>5s}  {'D':>5s}  {'H':>5s}")
for i, C in enumerate(C_range):
    line = f"{C:>8.4f}"
    for cls in ['A', 'D', 'H']:
        nz = int((np.abs(np.array(coef_paths[cls])[i]) > 1e-6).sum())
        line += f"  {nz:>5d}"
    # Mark champion C
    marker = "  ← champion" if abs(C - 0.1) < 0.05 else ""
    print(line + marker)

In [ ]:
# ============================================================
# STEP 19 — REGULARIZATION ANALYSIS (analysis only, no re-tuning)
# All-in-one cell. Champion frozen; nothing here changes it.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss

sns.set_theme(style='whitegrid')

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
import validation
importlib.reload(validation)

# ============================================================
# 1. CHAMPION COEFFICIENTS
# ============================================================
print("=" * 60)
print("STEP 19.1 — Champion coefficients")
print("=" * 60)

champion = joblib.load(os.path.join(MODELS_DIR, 'champion_final.joblib'))
features_meta = json.load(open(os.path.join(MODELS_DIR, 'champion_features.json')))
CHAMPION_FEATURES = features_meta['features']

coefs       = champion.named_steps['model'].coef_   # shape (3, 9) — champion pipeline uses "model"
intercepts  = champion.named_steps['model'].intercept_

coef_df = pd.DataFrame(coefs.T, index=CHAMPION_FEATURES, columns=['A', 'D', 'H'])
coef_df['max_abs']   = coef_df.abs().max(axis=1)
coef_df['n_nonzero'] = (coef_df[['A','D','H']].abs() > 1e-6).sum(axis=1)

print(coef_df.round(4).to_string())
print(f"\nIntercepts: A={intercepts[0]:.4f}, D={intercepts[1]:.4f}, H={intercepts[2]:.4f}")
zeroed = coef_df[coef_df['max_abs'] < 1e-6].index.tolist()
print(f"Zeroed features (across all classes): {zeroed if zeroed else 'none'}")

coef_df.to_csv(os.path.join(RESULTS_DIR, 'step19_champion_coefs.csv'))
print("Saved: step19_champion_coefs.csv")


# ============================================================
# 2. REGULARIZATION PATH
# ============================================================
print("\n" + "=" * 60)
print("STEP 19.2 — Regularization path")
print("=" * 60)

train_form = pd.read_csv(os.path.join(PROC_DIR, 'train_form.csv'), parse_dates=['Date'])
val_form   = pd.read_csv(os.path.join(PROC_DIR, 'val_form.csv'),   parse_dates=['Date'])
tv = pd.concat([train_form, val_form], ignore_index=True).reset_index(drop=True)

X_tv = tv[CHAMPION_FEATURES].values
y_tv = tv['FTR_int'].values

# Fit scaler ONCE so coefficients are comparable across the entire path
scaler = StandardScaler().fit(X_tv)
X_tv_scaled = scaler.transform(X_tv)

C_range = np.logspace(-3, 1, 25)

coef_paths = {cls: [] for cls in ['A', 'D', 'H']}
for C in C_range:
    lr = LogisticRegression(penalty='l1', solver='saga',
                            C=C, max_iter=3000, random_state=42)
    lr.fit(X_tv_scaled, y_tv)
    for ci, cls in enumerate(['A', 'D', 'H']):
        coef_paths[cls].append(lr.coef_[ci])

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
class_color_map = plt.cm.tab10(np.linspace(0, 1, len(CHAMPION_FEATURES)))

for ax, cls in zip(axes, ['A', 'D', 'H']):
    path = np.array(coef_paths[cls])
    for fi, feat in enumerate(CHAMPION_FEATURES):
        ax.plot(C_range, path[:, fi], linewidth=1.8,
                color=class_color_map[fi], label=feat)
    ax.set_xscale('log')
    ax.axvline(0.1, color='red', linestyle='--', alpha=0.6, label='Champion C=0.1')
    ax.set_xlabel("C (inverse regularization)")
    ax.set_ylabel("Coefficient (standardized)")
    ax.set_title(f"Class {cls}")
    ax.grid(alpha=0.3)
    ax.axhline(0, color='gray', linewidth=0.5)

axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
path_plot = os.path.join(RESULTS_DIR, 'step19_regularization_path.png')
plt.savefig(path_plot, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", path_plot)

print("\nSparsity across C (number of nonzero coefficients out of 9, per class):")
print(f"{'C':>9s}  {'A':>3s}  {'D':>3s}  {'H':>3s}")
for i, C in enumerate(C_range):
    line = f"{C:>9.4f}"
    for cls in ['A', 'D', 'H']:
        nz = int((np.abs(np.array(coef_paths[cls])[i]) > 1e-6).sum())
        line += f"  {nz:>3d}"
    if abs(C - 0.1) < 0.05:
        line += "  ← champion"
    print(line)


# ============================================================
# 3. L1_RATIO COMPARISON (Ridge vs ElasticNet vs Lasso)
# ============================================================
print("\n" + "=" * 60)
print("STEP 19.3 — l1_ratio comparison (7-fold CV)")
print("=" * 60)

splitter = validation.SeasonSplit(train_form['season_label'], min_train_seasons=1)

l1_ratios = [0.0, 0.5, 1.0]
labels_map = {0.0: 'Ridge (L2)', 0.5: 'ElasticNet', 1.0: 'Lasso (L1)'}

rows = []
for l1 in l1_ratios:
    fold_lls = []
    nonzero_counts = []
    for tr_idx, va_idx in splitter:
        tr = train_form.iloc[tr_idx].reset_index(drop=True)
        va = train_form.iloc[va_idx].reset_index(drop=True)

        pipe = Pipeline([
            ("s", StandardScaler()),
            ("m", LogisticRegression(penalty='elasticnet', solver='saga',
                                     l1_ratio=l1, C=0.1, max_iter=3000,
                                     class_weight=None, random_state=42))
        ])
        pipe.fit(tr[CHAMPION_FEATURES].values, tr['FTR_int'].values)
        p = pipe.predict_proba(va[CHAMPION_FEATURES].values)
        fold_lls.append(log_loss(va['FTR_int'].values, p, labels=[0, 1, 2]))

        # NOTE: this pipeline names the estimator "m", not "model"
        coefs_fold = pipe.named_steps['m'].coef_
        nz = int((np.abs(coefs_fold) > 1e-6).sum())
        nonzero_counts.append(nz)

    rows.append({
        'l1_ratio': l1,
        'label': labels_map[l1],
        'll_mean': float(np.mean(fold_lls)),
        'll_std': float(np.std(fold_lls)),
        'nonzero_coefs_mean': float(np.mean(nonzero_counts)),
    })

l1_df = pd.DataFrame(rows)
print(l1_df.round(4).to_string(index=False))
l1_df.to_csv(os.path.join(RESULTS_DIR, 'step19_l1ratio_comparison.csv'), index=False)
print("Saved: step19_l1ratio_comparison.csv")


# ============================================================
# 4. XGBOOST REGULARIZATION TEST (brief)
# ============================================================
print("\n" + "=" * 60)
print("STEP 19.4 — XGBoost reg_lambda (7-fold CV)")
print("=" * 60)

try:
    from xgboost import XGBClassifier
    XGB_OK = True
except ImportError:
    XGB_OK = False
    print("XGBoost not available, skipping.")

if XGB_OK:
    import models as ml
    importlib.reload(ml)
    all_feat = ml.select_features(train_form)
    print(f"XGBoost using {len(all_feat)} features.")

    reg_lambdas = [0.0, 1.0, 5.0, 20.0]
    xgb_rows = []
    for rl in reg_lambdas:
        fold_lls = []
        for tr_idx, va_idx in splitter:
            tr = train_form.iloc[tr_idx].reset_index(drop=True)
            va = train_form.iloc[va_idx].reset_index(drop=True)
            xgb = XGBClassifier(
                n_estimators=200, max_depth=4, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                reg_lambda=rl, reg_alpha=0.0,
                objective='multi:softprob', num_class=3, eval_metric='mlogloss',
                random_state=42, n_jobs=-1, tree_method='hist', verbosity=0,
            )
            xgb.fit(tr[all_feat].values, tr['FTR_int'].values)
            p = xgb.predict_proba(va[all_feat].values)
            fold_lls.append(log_loss(va['FTR_int'].values, p, labels=[0, 1, 2]))

        xgb_rows.append({
            'reg_lambda': rl,
            'll_mean': float(np.mean(fold_lls)),
            'll_std': float(np.std(fold_lls)),
        })

    xgb_df = pd.DataFrame(xgb_rows)
    print(xgb_df.round(4).to_string(index=False))
    xgb_df.to_csv(os.path.join(RESULTS_DIR, 'step19_xgb_reg.csv'), index=False)
    print("Saved: step19_xgb_reg.csv")
else:
    xgb_df = pd.DataFrame()


# ============================================================
# 5. WRITE REPORT + DECISION LOG
# ============================================================
print("\n" + "=" * 60)
print("STEP 19.5 — Write report + decision log")
print("=" * 60)

lines = [
    "# Step 19 — Regularization Analysis",
    "",
    "**Analysis only.** The champion model is frozen; nothing here changes it.",
    "",
    "## 1. Champion coefficients",
    "",
    "| Feature | Class A | Class D | Class H | Max abs | Nonzero |",
    "| --- | ---: | ---: | ---: | ---: | ---: |",
]
for feat, row in coef_df.iterrows():
    lines.append(f"| `{feat}` | {row['A']:+.4f} | {row['D']:+.4f} | {row['H']:+.4f} | "
                 f"{row['max_abs']:.4f} | {int(row['n_nonzero'])} |")

lines += [
    "",
    f"**Intercepts:** A={intercepts[0]:.4f}, D={intercepts[1]:.4f}, H={intercepts[2]:.4f}",
    "",
    f"**Features zeroed by Lasso (across all classes):** {zeroed if zeroed else 'none'}",
    "",
    "## 2. Regularization path",
    "",
    "See `results/step19_regularization_path.png`.",
    "",
    "The path shows how each coefficient shrinks toward zero as regularization",
    "increases (C decreases). Features that enter the model at high C are the",
    "most robustly predictive; features that only appear at low regularization",
    "contribute weaker or noisier signal.",
    "",
    "## 3. l1_ratio comparison",
    "",
    "| l1_ratio | Label | CV Log loss (mean) | CV Log loss (std) | Mean nonzero coefs |",
    "| ---: | --- | ---: | ---: | ---: |",
]
for _, r in l1_df.iterrows():
    lines.append(f"| {r['l1_ratio']} | {r['label']} | {r['ll_mean']:.4f} | "
                 f"{r['ll_std']:.4f} | {r['nonzero_coefs_mean']:.1f} |")

lines += ["", "## 4. XGBoost regularization", ""]
if XGB_OK and not xgb_df.empty:
    lines += [
        "| reg_lambda | CV Log loss (mean) | CV Log loss (std) |",
        "| ---: | ---: | ---: |",
    ]
    for _, r in xgb_df.iterrows():
        lines.append(f"| {r['reg_lambda']} | {r['ll_mean']:.4f} | {r['ll_std']:.4f} |")
    lines += [
        "",
        "None of the XGBoost configurations approached the champion's performance.",
        "Confirms the earlier finding: tree models are the wrong choice here.",
    ]
else:
    lines.append("(XGBoost not tested.)")

lines += [
    "",
    "## 5. Conclusion",
    "",
    "Lasso regularization (l1_ratio=1.0, C=0.1) selected a sparse subset of",
    "the 9 delta features. Sparse solutions fit well and generalize; the",
    "champion's CV log loss (0.9699) is competitive with denser models",
    "despite fewer effective parameters.",
    "",
    "## Artifacts",
    "",
    "- results/step19_champion_coefs.csv",
    "- results/step19_regularization_path.png",
    "- results/step19_l1ratio_comparison.csv",
    "- results/step19_xgb_reg.csv (if XGBoost available)",
]
with open(os.path.join(DOCS_DIR, 'step19_regularization.md'), 'w') as f:
    f.write("\n".join(lines))
print("Wrote:", os.path.join(DOCS_DIR, 'step19_regularization.md'))

# Decision log
ridge_ll = l1_df.loc[l1_df['l1_ratio']==0.0, 'll_mean'].iloc[0]
enet_ll  = l1_df.loc[l1_df['l1_ratio']==0.5, 'll_mean'].iloc[0]
lasso_ll = l1_df.loc[l1_df['l1_ratio']==1.0, 'll_mean'].iloc[0]

decision_log = os.path.join(DOCS_DIR, 'decision_log.md')
append_block = (
    "\n\n## Step 19 — Regularization analysis (no model changes)\n"
    "**Champion:** Lasso (l1_ratio=1.0, C=0.1) on 9 delta features.\n"
    f"**Features zeroed by Lasso:** {zeroed if zeroed else 'none'}\n"
    "**l1_ratio comparison (CV log loss at C=0.1):**\n"
    f"- Ridge (L2, l1_ratio=0.0):  {ridge_ll:.4f}\n"
    f"- ElasticNet (l1_ratio=0.5): {enet_ll:.4f}\n"
    f"- Lasso (L1, l1_ratio=1.0):  {lasso_ll:.4f}\n"
    "**Conclusion:** Lasso gives a sparse, interpretable model without losing performance.\n"
    "**Test result unchanged.** Champion frozen at Step 17 (log loss 1.0238).\n"
)
with open(decision_log, 'a') as f:
    f.write(append_block)
print("Appended to decision log.")
print("\n" + "=" * 60)
print("STEP 19 COMPLETE.")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 20 — MODEL INTERPRETATION
# Read coefficients, produce plots, explain in football terms.
# Champion frozen; no model changes.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 6)

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
os.makedirs(RESULTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# ---------- Load champion ----------
champion = joblib.load(os.path.join(MODELS_DIR, 'champion_final.joblib'))
features_meta = json.load(open(os.path.join(MODELS_DIR, 'champion_features.json')))
CHAMPION_FEATURES = features_meta['features']

coefs = champion.named_steps['model'].coef_
intercepts = champion.named_steps['model'].intercept_

coef_df = pd.DataFrame(coefs.T, index=CHAMPION_FEATURES,
                        columns=['A', 'D', 'H'])
coef_df['max_abs'] = coef_df.abs().max(axis=1)
coef_df = coef_df.sort_values('max_abs', ascending=True)

# ============================================================
# 1. FEATURE IMPORTANCE PLOTS
# ============================================================
print("=" * 60)
print("STEP 20.1 — Feature importance")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel 1: max abs per feature (headline)
axes[0].barh(coef_df.index, coef_df['max_abs'], color='#1f77b4')
axes[0].set_xlabel("Max |coefficient| across classes")
axes[0].set_title("Feature importance (max abs across A/D/H)")
axes[0].grid(alpha=0.3, axis='x')

# Panel 2: grouped bars per class
y = np.arange(len(coef_df))
w = 0.28
axes[1].barh(y - w, coef_df['A'], w, label='A', color='#d62728')
axes[1].barh(y,     coef_df['D'], w, label='D', color='#7f7f7f')
axes[1].barh(y + w, coef_df['H'], w, label='H', color='#2ca02c')
axes[1].set_yticks(y)
axes[1].set_yticklabels(coef_df.index)
axes[1].axvline(0, color='black', linewidth=0.5)
axes[1].set_xlabel("Coefficient")
axes[1].set_title("Coefficients by class")
axes[1].legend()
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'step20_feature_importance.png')
plt.savefig(plot_path, dpi=140, bbox_inches='tight')
plt.show()
print("Saved:", plot_path)

# ============================================================
# 2. COEFFICIENT TABLE (top-to-bottom by importance)
# ============================================================
print("\n" + "=" * 60)
print("STEP 20.2 — Coefficients (sorted by importance)")
print("=" * 60)
print(coef_df[['A','D','H','max_abs']].round(4).to_string())

# ============================================================
# 3. FIVE EXAMPLE TEST MATCHES
# ============================================================
print("\n" + "=" * 60)
print("STEP 20.3 — Five example test matches")
print("=" * 60)

# Load test features + predictions
test_feats = pd.read_csv(os.path.join(PROC_DIR, 'test_form.csv'), parse_dates=['Date'])
test_pred  = pd.read_csv(os.path.join(RESULTS_DIR, 'step17_test_predictions.csv'))

test_combined = pd.concat([
    test_feats[['Date','HomeTeam','AwayTeam','FTR']].reset_index(drop=True),
    test_pred.reset_index(drop=True),
], axis=1)

# Assign a "prediction type" label
def classify_pred(row):
    p = [row['p_A'], row['p_D'], row['p_H']]
    winner_idx = int(np.argmax(p))
    confidence = max(p)
    pred_label = ['A', 'D', 'H'][winner_idx]
    actual = ['A', 'D', 'H'][int(row['y_true'])]
    correct = (pred_label == actual)
    if confidence > 0.6:
        return f"High-confidence {pred_label}" + (" (✓)" if correct else " (✗)")
    elif confidence > 0.45:
        return f"Medium-confidence {pred_label}" + (" (✓)" if correct else " (✗)")
    else:
        return f"Close-call {pred_label}" + (" (✓)" if correct else " (✗)")

test_combined['type'] = test_combined.apply(classify_pred, axis=1)

# Pick examples: highest-confidence correct H, highest-confidence correct A,
# closest-call, a draw that was missed, a surprise
examples = []

# High-confidence H (correct)
h_rows = test_combined[(test_combined['y_true']==2) &
                       (test_combined['p_H'] > 0.6)].sort_values('p_H', ascending=False)
if len(h_rows): examples.append(h_rows.iloc[0])

# High-confidence A (correct)
a_rows = test_combined[(test_combined['y_true']==0) &
                       (test_combined['p_A'] > 0.5)].sort_values('p_A', ascending=False)
if len(a_rows): examples.append(a_rows.iloc[0])

# Close call (max prob lowest)
close = test_combined.sort_values(['p_H','p_A'], ascending=True).copy()
close['max_prob'] = close[['p_A','p_D','p_H']].max(axis=1)
examples.append(close.sort_values('max_prob').iloc[0])

# A draw the model missed (predicted H or A, actual D)
draw_missed = test_combined[(test_combined['y_true']==1) &
                             (test_combined['y_pred']!=1)].sort_values('p_H', ascending=False)
if len(draw_missed): examples.append(draw_missed.iloc[0])

# Surprise: high-confidence wrong
high_wrong = test_combined[test_combined['y_true'] != test_combined['y_pred']].copy()
high_wrong['max_p'] = high_wrong[['p_A','p_D','p_H']].max(axis=1)
examples.append(high_wrong.sort_values('max_p', ascending=False).iloc[0])

# Deduplicate by index
examples_df = pd.DataFrame(examples).drop_duplicates(subset=['Date','HomeTeam','AwayTeam']).head(5)

print("\nSelected examples:")
for i, (_, row) in enumerate(examples_df.iterrows(), 1):
    actual = ['A', 'D', 'H'][int(row['y_true'])]
    pred   = ['A', 'D', 'H'][int(row['y_pred'])]
    print(f"\n--- Example {i}: {row['Date'].date()} | {row['HomeTeam']} vs {row['AwayTeam']} ---")
    print(f"  Actual: {actual}   Predicted: {pred}   Type: {row['type']}")
    print(f"  Probabilities: A={row['p_A']:.3f}  D={row['p_D']:.3f}  H={row['p_H']:.3f}")

# Feature values for the examples
print("\nFeature values for these matches:")
for i, (_, row) in enumerate(examples_df.iterrows(), 1):
    print(f"\nExample {i}:")
    match_row = test_feats[(test_feats['Date']==row['Date']) &
                           (test_feats['HomeTeam']==row['HomeTeam']) &
                           (test_feats['AwayTeam']==row['AwayTeam'])].iloc[0]
    for f in CHAMPION_FEATURES:
        print(f"  {f:28s} = {match_row[f]:+.3f}")

examples_df.to_csv(os.path.join(RESULTS_DIR, 'step20_examples.csv'), index=False)
print("\nSaved: step20_examples.csv")

# ============================================================
# 4. FOOTBALL SANITY CHECK
# ============================================================
print("\n" + "=" * 60)
print("STEP 20.4 — Football sanity check")
print("=" * 60)

sanity_checks = [
    ("delta_elo",
     "Team strength via Elo difference.",
     "Yes. This is the dominant signal — well-established in football analytics."),
    ("delta_form_shots_last5",
     "Recent shots-for differential.",
     "Yes. Underlying performance proxy — teams with more shots tend to score more."),
    ("delta_form_gd_last5",
     "Recent goal-difference differential.",
     "Yes. Captures both attack and defense over the last 5 matches."),
    ("delta_form_pts_last5",
     "Recent points-per-match differential.",
     "Yes. Direct short-term form signal."),
    ("delta_season_ppg",
     "Season-to-date points-per-game differential.",
     "Yes. Long-form season context."),
    ("delta_venue_pts_last5",
     "Venue-specific recent form differential.",
     "Yes. Captures home/away-specific patterns (coefficient only nonzero for H)."),
    ("delta_prev_season_ppg",
     "Last season's PPG differential.",
     "Yes. Baseline team quality (small coefficient — team turnover reduces signal)."),
    ("delta_form_sot_last5",
     "Shots-on-target differential.",
     "Yes. Shot quality proxy (small coefficient — shots already capture most of it)."),
    ("delta_rest_days",
     "Rest-days differential.",
     "Small but nonzero — rest matters, but weakly at 2-14 day scale."),
]

for feat, description, verdict in sanity_checks:
    coef = coef_df.loc[feat, 'max_abs']
    print(f"\n  {feat}  (max |coef| = {coef:.4f})")
    print(f"    Meaning: {description}")
    print(f"    Verdict: {verdict}")

# ============================================================
# 5. WRITE REPORT
# ============================================================
print("\n" + "=" * 60)
print("STEP 20.5 — Write report")
print("=" * 60)

lines = [
    "# Step 20 — Model Interpretation",
    "",
    "Analysis of what the champion model believes about football.",
    "Champion frozen; no model changes.",
    "",
    "## 1. Feature importance",
    "",
    "See `results/step20_feature_importance.png`.",
    "",
    "Coefficients (sorted by max absolute value across classes):",
    "",
    "| Feature | A | D | H | Max abs |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for feat, row in coef_df.sort_values('max_abs', ascending=False).iterrows():
    lines.append(f"| `{feat}` | {row['A']:+.4f} | {row['D']:+.4f} | {row['H']:+.4f} | {row['max_abs']:.4f} |")

lines += [
    "",
    f"**Intercepts:** A={intercepts[0]:.4f}, D={intercepts[1]:.4f}, H={intercepts[2]:.4f}",
    "",
    "**Home intercept dominates** — the model starts with a strong home-win prior,",
    "which features then adjust.",
    "",
    "## 2. What the coefficients say",
    "",
    "### Dominant feature: `delta_elo` (±0.44)",
    "",
    "The Elo difference is by far the strongest predictor. Teams with a higher",
    "Elo rating are more likely to win, all else equal.",
    "",
    "### Secondary features (±0.10 to ±0.23)",
    "",
    "- `delta_form_shots_last5` (−0.23 A, +0.10 H): Recent shot volume matters.",
    "  Teams generating more shots than opponents are more likely to win.",
    "- `delta_form_gd_last5` (+0.17 H): Recent goal difference gives further signal.",
    "- `delta_form_pts_last5` (−0.09 H): Recent form is captured but partly overlaps",
    "  with the above.",
    "",
    "### Small-but-nonzero features (< ±0.10)",
    "",
    "- `delta_season_ppg` (+0.08 H)",
    "- `delta_venue_pts_last5` (−0.07 H)",
    "- `delta_form_sot_last5` (+0.03 H, −0.04 D)",
    "- `delta_prev_season_ppg` (−0.02 A)",
    "- `delta_rest_days` (±0.01)",
    "",
    "### The Draw class collapse",
    "",
    "**6 of 9 features have exactly zero D coefficient.** Only `delta_form_sot_last5`",
    "(−0.044) is nonzero and it's tiny. This confirms the well-documented difficulty",
    "of predicting draws: there is essentially no reliable feature pattern that",
    "identifies draws beyond noise.",
    "",
    "## 3. Example matches",
    "",
    "See `results/step20_examples.csv`.",
    "",
    "The five selected examples cover:",
    "1. A high-confidence correct H prediction",
    "2. A high-confidence correct A prediction",
    "3. A close-call (model uncertain)",
    "4. A draw the model missed",
    "5. A high-confidence surprise (wrong)",
    "",
    "## 4. Football sanity check",
    "",
    "| Feature | Meaning | Verdict |",
    "| --- | --- | --- |",
]
for feat, description, verdict in sanity_checks:
    lines.append(f"| `{feat}` | {description} | {verdict} |")

lines += [
    "",
    "## 5. What the model teaches us about EPL prediction",
    "",
    "1. **Elo alone captures most of the predictable signal.** Our 9 features",
    "   give only a small edge over a 1-feature Elo baseline.",
    "2. **Draws are fundamentally hard.** The model essentially refuses to predict",
    "   them because the statistical structure of the data doesn't support it.",
    "3. **Recent form matters, but less than strength.** Rolling shots and goal",
    "   difference add modest signal.",
    "4. **Rest-days effects are weak** at the day scale we modeled (2-14 days).",
    "5. **Lasso-selected coefficients make football sense** — the model passes",
    "   the domain-expert smell test.",
    "",
    "## Artifacts",
    "",
    "- `results/step20_feature_importance.png`",
    "- `results/step20_examples.csv`",
]
with open(os.path.join(DOCS_DIR, 'step20_interpretation.md'), 'w') as f:
    f.write("\n".join(lines))
print("Wrote:", os.path.join(DOCS_DIR, 'step20_interpretation.md'))

# Decision log
decision_log = os.path.join(DOCS_DIR, 'decision_log.md')
append_block = (
    "\n\n## Step 20 — Model interpretation\n"
    "**Dominant feature:** delta_elo (coef ±0.44 across A/H).\n"
    "**Second:** delta_form_shots_last5 (−0.23 A).\n"
    "**Draw class:** 6/9 coefficients zeroed; the model effectively\n"
    "does not predict draws.\n"
    "**Home intercept dominates** (0.3156): strong prior for home wins.\n"
    "**Conclusion:** Lasso-selected coefficients make football sense.\n"
    "**Test result unchanged.** Champion frozen at Step 17 (log loss 1.0238).\n"
)
with open(decision_log, 'a') as f:
    f.write(append_block)
print("Appended to decision log.")
print("\n" + "=" * 60)
print("STEP 20 COMPLETE.")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 21 — PART A: verify, snapshot, requirements, canary
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, shutil, datetime
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
SNAPSHOT_DIR = os.path.join(MODELS_DIR, 'src_snapshot')
os.makedirs(SNAPSHOT_DIR, exist_ok=True)

# ---- 1. Verify existing artifacts ----
print("=" * 60)
print("STEP 21.1 — Verify existing artifacts")
print("=" * 60)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_HASHES = {
    'champion_final.joblib':   'b75342ce4f5b778465b3529e566ef2d5c9a78fa15f1cc5f7d40b7c7b033cca4c',
    'champion_features.json':  'b38cce5722bc911c749c50cf1da49bd7f66ce6fea0f42a194706f6dd36d7d56c',
    'champion_metadata.json':  '94a92417226f270422f5676ee189fdbae9c1c75c48625ba55f0458b46d403367',
}

all_ok = True
for fname, expected in EXPECTED_HASHES.items():
    path = os.path.join(MODELS_DIR, fname)
    if not os.path.exists(path):
        print(f"  MISSING: {fname}")
        all_ok = False
        continue
    actual = sha256(path)
    status = "OK" if actual == expected else "CHANGED"
    print(f"  {status} {fname}")
    if actual != expected:
        print(f"    expected: {expected}")
        print(f"    actual:   {actual}")
        all_ok = False

print("All artifacts unchanged." if all_ok else "Some artifacts changed.")

# ---- 2. Snapshot source code ----
print("\n" + "=" * 60)
print("STEP 21.2 — Snapshot source code")
print("=" * 60)

src_files = ['features.py', 'pipeline.py', 'validation.py', 'models.py', 'baselines.py']
snapshot_meta = {
    'created_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'files': {},
}
for f in src_files:
    src_path = os.path.join(SRC_DIR, f)
    if os.path.exists(src_path):
        dst = os.path.join(SNAPSHOT_DIR, f)
        shutil.copy2(src_path, dst)
        snapshot_meta['files'][f] = {
            'sha256': sha256(dst),
            'size':   os.path.getsize(dst),
        }
        print(f"  {f}  ({os.path.getsize(dst)} bytes)")
    else:
        print(f"  missing: {f}")

with open(os.path.join(MODELS_DIR, 'src_snapshot_manifest.json'), 'w') as fp:
    json.dump(snapshot_meta, fp, indent=2)
print("Saved src_snapshot_manifest.json")

# ---- 3. requirements.txt ----
print("\n" + "=" * 60)
print("STEP 21.3 — requirements.txt")
print("=" * 60)

import sklearn, matplotlib
try:
    import xgboost as xgb
    xgb_ver = xgb.__version__
except ImportError:
    xgb_ver = None

reqs = [
    f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"matplotlib=={matplotlib.__version__}",
    f"joblib=={joblib.__version__}",
]
if xgb_ver:
    reqs.append(f"xgboost=={xgb_ver}")
try:
    import seaborn as sns
    reqs.append(f"seaborn=={sns.__version__}")
except ImportError:
    pass

req_path = os.path.join(MODELS_DIR, 'requirements.txt')
with open(req_path, 'w') as f:
    f.write("\n".join(reqs) + "\n")
print("Wrote:", req_path)
for r in reqs:
    print("  " + r)

# ---- 4. Canary row ----
print("\n" + "=" * 60)
print("STEP 21.4 — Canary row")
print("=" * 60)

champion = joblib.load(os.path.join(MODELS_DIR, 'champion_final.joblib'))
features_meta = json.load(open(os.path.join(MODELS_DIR, 'champion_features.json')))
CHAMPION_FEATURES = features_meta['features']

test_feats = pd.read_csv(os.path.join(PROC_DIR, 'test_form.csv'), parse_dates=['Date'])
canary_row = test_feats.iloc[0]

canary_features = canary_row[CHAMPION_FEATURES].to_dict()
X_canary = np.array([[canary_features[f] for f in CHAMPION_FEATURES]])
proba_canary = champion.predict_proba(X_canary)[0]

canary_data = {
    'created_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'match': {
        'date':   str(canary_row['Date'].date()),
        'home':   canary_row['HomeTeam'],
        'away':   canary_row['AwayTeam'],
        'season': canary_row['season_label'],
    },
    'features': {k: float(v) for k, v in canary_features.items()},
    'expected_proba': {
        'A': float(proba_canary[0]),
        'D': float(proba_canary[1]),
        'H': float(proba_canary[2]),
    },
    'tolerance': 1e-9,
}
canary_path = os.path.join(MODELS_DIR, 'canary.json')
with open(canary_path, 'w') as f:
    json.dump(canary_data, f, indent=2)
print("Wrote:", canary_path)
print(f"Canary: {canary_data['match']['date']} | "
      f"{canary_data['match']['home']} vs {canary_data['match']['away']}")
print(f"Expected proba: A={proba_canary[0]:.6f}  D={proba_canary[1]:.6f}  H={proba_canary[2]:.6f}")

proba_reload = champion.predict_proba(X_canary)[0]
assert np.allclose(proba_reload, proba_canary, atol=1e-9), "Canary drift!"
print("Canary self-check passed.")

In [ ]:
# ============================================================
# STEP 21 — PART B: model card, README, final verification
# ============================================================

import os, json
from google.colab import drive

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')

# ---- 5. Model card ----
print("=" * 60)
print("STEP 21.5 — Model card")
print("=" * 60)

meta = json.load(open(os.path.join(MODELS_DIR, 'champion_metadata.json')))

model_card_lines = [
    "# Model Card — EPL Match Outcome Predictor",
    "",
    f"**Model version:** {meta['model_name']}",
    f"**Created:** {meta.get('created_utc', 'unknown')}",
    "**Task:** Multiclass classification (3 classes: A, D, H)",
    "",
    "---",
    "",
    "## What it does",
    "",
    "Predicts the outcome of English Premier League football matches from",
    "pre-match information only. Output is a probability distribution over",
    "{Away win, Draw, Home win} that sums to 1.",
    "",
    "## When to use it",
    "",
    "- Before kickoff, using only matches that have already been played.",
    "- For matches between teams whose recent history is available in the",
    "  feature builders. Teams must have appeared at least once in the",
    "  training window (2015/16 - 2024/25).",
    "",
    "## When NOT to use it",
    "",
    "- Not for in-play prediction (uses only pre-match data).",
    "- Not for leagues outside the English Premier League.",
    "- Not for seasons earlier than 2015/16 (no training context).",
    "- Not as a betting tool (learning project; no odds data).",
    "- Not for confident draw prediction - the model never predicts draws",
    "  as its top class. Treat draw probabilities with strong skepticism.",
    "",
    "## Model architecture",
    "",
    "- Type: Multinomial Logistic Regression",
    "- Regularization: ElasticNet (l1_ratio = 1.0, i.e. pure Lasso)",
    "- C: 0.1",
    "- Solver: saga",
    "- Input: 9 delta features (home minus away)",
    "- Output: 3 probabilities (A, D, H)",
    "- Preprocessing: StandardScaler (fit on training data)",
    "",
    "## Features (9)",
    "",
    "1. `delta_elo` - Elo rating difference (dominant feature)",
    "2. `delta_form_pts_last5` - recent points-per-match",
    "3. `delta_form_gd_last5` - recent goal difference",
    "4. `delta_venue_pts_last5` - venue-specific recent form",
    "5. `delta_season_ppg` - season-to-date PPG",
    "6. `delta_prev_season_ppg` - previous season's PPG",
    "7. `delta_form_shots_last5` - recent shots per match",
    "8. `delta_form_sot_last5` - recent shots on target per match",
    "9. `delta_rest_days` - days since last match",
    "",
    "## Training data",
    "",
    "- Seasons: 2015/16 - 2024/25 (train + val)",
    "- Rows: 3800 matches",
    "- League: English Premier League only",
    "- All features built using only earlier matches",
    "",
    "## Performance",
    "",
    "### 7-fold CV (on training data)",
    "",
    "- Log loss: 0.9699 (mean), 0.0342 (std)",
    "- Accuracy: 0.5466",
    "- Folds won vs EloBaseline: 5 of 7",
    "",
    "### Final test (2025/26, 380 matches)",
    "",
    "- Log loss: **1.0238** (95% CI [0.9771, 1.0687])",
    "- Accuracy: 0.4711",
    "",
    "### Baselines (test set)",
    "",
    "| Baseline | Log loss |",
    "| --- | ---: |",
    "| Class-prior | 1.0838 |",
    "| Elo only (1 feature) | 1.0263 |",
    "| **Champion** | **1.0238** |",
    "",
    "## Known limitations",
    "",
    "1. **Never predicts draws.** Draw recall = 0. This is not a bug;",
    "   the model has no reliable signal for distinguishing draws from",
    "   close wins.",
    "2. **Marginal edge over Elo alone.** The champion beats a single-feature",
    "   Elo baseline by only 0.0026 log loss on test - within the CI.",
    "3. **CV-to-test degradation.** Test log loss is 0.054 worse than CV.",
    "   Partly explained by 2025/26 having ~4 percentage points more draws",
    "   than the training period.",
    "4. **Football intrinsic unpredictability.** Even the best public football",
    "   models land in the 0.95-1.00 log loss range; the task has a hard ceiling.",
    "",
    "## Intended use",
    "",
    "Portfolio/learning project, football-analytics research, and probability",
    "calibration study. Not intended for wagering or any financial use.",
    "",
    "## Artifacts in this folder",
    "",
    "- `champion_final.joblib` - the fitted pipeline",
    "- `champion_features.json` - the exact feature names, in order",
    "- `champion_metadata.json` - config, training info, CV performance",
    "- `canary.json` - canonical input + expected output for stability",
    "- `src_snapshot/` - frozen copy of the feature engineering code",
    "- `src_snapshot_manifest.json` - SHA-256 of every snapshot file",
    "- `requirements.txt` - Python dependencies",
    "- `README.md` - this folder's navigation guide",
]

with open(os.path.join(MODELS_DIR, 'champion_model_card.md'), 'w') as f:
    f.write("\n".join(model_card_lines))
print("Wrote: champion_model_card.md")

# ---- 6. models README ----
print("\n" + "=" * 60)
print("STEP 21.6 — models/README.md")
print("=" * 60)

readme_lines = [
    "# models/ — Final Artifacts",
    "",
    "This folder contains the frozen champion model and everything needed",
    "to load, verify, and reproduce it.",
    "",
    "## Files",
    "",
    "| File | Purpose |",
    "| --- | --- |",
    "| `champion_final.joblib` | The fitted sklearn Pipeline (scaler + LR). |",
    "| `champion_features.json` | Ordered feature list. Pass in the same order the model was trained on. |",
    "| `champion_metadata.json` | Hyperparameters, training info, CV metrics. |",
    "| `champion_model_card.md` | Human-readable model card. **Read this first.** |",
    "| `canary.json` | Canonical test: features + expected probabilities. |",
    "| `requirements.txt` | Python dependencies. |",
    "| `src_snapshot/` | Frozen copy of `src/*.py` at freeze time. |",
    "| `src_snapshot_manifest.json` | SHA-256 of every file in `src_snapshot/`. |",
    "",
    "## Quick usage",
    "",
    "```python",
    "import json, joblib, numpy as np",
    "",
    "champion = joblib.load('champion_final.joblib')",
    "features = json.load(open('champion_features.json'))['features']",
    "",
    "# features must be in this exact order",
    "X = np.array([[match[f] for f in features]])",
    "",
    "proba = champion.predict_proba(X)[0]   # [P(A), P(D), P(H)]",
    "```",
    "",
    "## Verification",
    "",
    "To verify the model file is unchanged:",
    "",
    "```",
    "sha256sum champion_final.joblib",
    "# Compare against b75342ce4f5b778465b3529e566ef2d5c9a78fa15f1cc5f7d40b7c7b033cca4c",
    "```",
    "",
    "To verify predictions are stable, load `canary.json`, predict, and",
    "compare probabilities.",
    "",
    "## Version history",
    "",
    "- v1.0 — frozen at Step 17 of the project pipeline. Test log loss 1.0238.",
]

with open(os.path.join(MODELS_DIR, 'README.md'), 'w') as f:
    f.write("\n".join(readme_lines))
print("Wrote: README.md")

# ---- 7. Final verification ----
print("\n" + "=" * 60)
print("STEP 21.7 — Final models/ listing")
print("=" * 60)

print("\nmodels/ contents:")
for f in sorted(os.listdir(MODELS_DIR)):
    p = os.path.join(MODELS_DIR, f)
    if os.path.isdir(p):
        print(f"  {f}/")
        for sub in sorted(os.listdir(p)):
            print(f"    {sub}")
    else:
        print(f"  {f}  ({os.path.getsize(p)} bytes)")

print("\n" + "=" * 60)
print("STEP 21 COMPLETE. Model artifacts frozen and documented.")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 22 — DEPLOYMENT / PREDICTION (documentation-only)
# The champion is frozen. This step documents how someone would
# deploy and use it in production. No new data used.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, json
import datetime

PROJECT_ROOT = '/content/drive/MyDrive/epl_match_outcome'
DOCS_DIR     = os.path.join(PROJECT_ROOT, 'docs')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')

# ============================================================
# 1. Decision log entry
# ============================================================
decision_log = os.path.join(DOCS_DIR, 'decision_log.md')

append_block = (
    "\n\n## Step 22 — Deployment (documentation-only) + scope decision\n"
    "**2026/27 dataset excluded.** A new source (football-data.co.uk) with a\n"
    "different schema (100 columns vs 22) and different date format was reviewed\n"
    "and excluded from the project for consistency and scope control.\n"
    "**Reason:** The original 11 CSVs came from a single consistent source\n"
    "(datahub). Integrating a different provider would require normalization\n"
    "work, schema reconciliation, and re-validation — outside the project scope.\n"
    "**Impact on champion:** none. The frozen champion predicts any EPL match\n"
    "from any source as long as the 9 delta features can be computed from\n"
    "prior match history.\n"
    "**Step 22 output:** deployment methodology documentation only.\n"
    "**Test result unchanged.** Champion frozen at Step 17 (log loss 1.0238).\n"
)

with open(decision_log, 'a') as f:
    f.write(append_block)
print("Appended Step 22 decision to:", decision_log)

# ============================================================
# 2. Deployment documentation
# ============================================================
deployment_doc_lines = [
    "# Step 22 — Deployment / Prediction",
    "",
    "**This step is documentation-only.** The champion is frozen at v1.0",
    "(test log loss 1.0238 on 2025/26). No new data was used to complete",
    "this project. This document describes how someone would deploy and use",
    "the model in a real workflow.",
    "",
    "---",
    "",
    "## 1. What the model does",
    "",
    "Given an EPL fixture (`Date`, `HomeTeam`, `AwayTeam`) and a history of",
    "prior matches for both teams, the champion returns a probability",
    "distribution over {Away win, Draw, Home win} that sums to 1.",
    "",
    "**Requirement:** Both teams must have appeared at least once in the",
    "training data (2015/16 – 2024/25) OR have enough prior matches in the",
    "input history for the feature builders to compute rolling features.",
    "",
    "---",
    "",
    "## 2. Data requirements",
    "",
    "### Minimum input per match in history",
    "",
    "| Column | Purpose |",
    "| --- | --- |",
    "| `Date` | Sort matches chronologically |",
    "| `HomeTeam`, `AwayTeam` | Identify participants |",
    "| `FTHG`, `FTAG`, `FTR` | Goals and result |",
    "| `HTHG`, `HTAG`, `HTR` | Half-time (used in logic checks) |",
    "| `HS`, `AS`, `HST`, `AST` | Shots and shots on target |",
    "| `HC`, `AC` | Corners |",
    "| `HF`, `AF` | Fouls |",
    "| `HY`, `AY`, `HR`, `AR` | Cards |",
    "| `Referee` | Required by schema but not used by champion |",
    "",
    "**Data sources must match or be normalized to the training schema.**",
    "Adding a new data source with a different schema requires explicit",
    "validation before use.",
    "",
    "---",
    "",
    "## 3. Prediction workflow",
    "",
    "### Step-by-step, for one upcoming fixture",
    "",
    "```",
    "1. Prepare a history DataFrame containing all matches played before",
    "   the fixture date, for both teams (and any other teams needed to",
    "   compute Elo — the champion uses a global Elo state).",
    "",
    "2. Append the upcoming fixture as a row with NaN for result columns.",
    "   Set its Date, HomeTeam, AwayTeam.",
    "",
    "3. Run the feature builders (features.py) on the combined DataFrame.",
    "   The 9 delta features will be computed for the upcoming fixture.",
    "",
    "4. Extract the 9 delta features for the fixture row.",
    "",
    "5. Load the champion pipeline (champion_final.joblib).",
    "",
    "6. Call champion.predict_proba(X) where X is the 9 features",
    "   in the exact order from champion_features.json.",
    "",
    "7. Result: [P(A), P(D), P(H)].",
    "```",
    "",
    "### Reference Python",
    "",
    "```python",
    "import json, joblib, numpy as np, pandas as pd",
    "import sys",
    "sys.path.append('src')",
    "import features",
    "",
    "champion = joblib.load('models/champion_final.joblib')",
    "features_meta = json.load(open('models/champion_features.json'))",
    "FEATURES = features_meta['features']",
    "",
    "# history_df: past matches (training schema)",
    "# fixture_df: the upcoming match(es), schema-matching, results set to NaN",
    "combined = pd.concat([history_df, fixture_df], ignore_index=True)",
    "combined = combined.sort_values('Date').reset_index(drop=True)",
    "",
    "# Feature build (same as the pipeline)",
    "# ... uses all the feature builders: rolling form, venue form, stats,",
    "#     season-to-date, rest days, Elo, previous-season PPG, deltas",
    "# ...",
    "",
    "X = combined[FEATURES].values",
    "proba = champion.predict_proba(X)",
    "```",
    "",
    "---",
    "",
    "## 4. Prediction logging",
    "",
    "Predictions should be saved BEFORE kickoff so they cannot be",
    "retrospectively adjusted. Suggested schema:",
    "",
    "| Column | Meaning |",
    "| --- | --- |",
    "| `fixture_id` | Unique match identifier |",
    "| `date` | Match date (UTC) |",
    "| `home` | Home team name |",
    "| `away` | Away team name |",
    "| `p_A`, `p_D`, `p_H` | Predicted probabilities |",
    "| `predicted_at_utc` | Timestamp when prediction was made |",
    "| `model_version` | e.g. `v1.0` |",
    "",
    "Save to CSV at `predictions/`. Never overwrite an existing prediction.",
    "",
    "---",
    "",
    "## 5. Post-match scoring",
    "",
    "After matches are played, join the actual result to the prediction log",
    "and compute:",
    "",
    "- Log loss (primary metric)",
    "- Accuracy",
    "- Per-class metrics",
    "- Comparison to baselines (class-prior, Elo-only)",
    "",
    "Track these per gameweek and cumulatively across the season.",
    "",
    "---",
    "",
    "## 6. Retraining policy",
    "",
    "**The champion is frozen.** Retraining is permitted only under",
    "controlled conditions (pipeline Rule 19):",
    "",
    "1. A new evaluation set must be reserved in advance.",
    "2. The old champion must be frozen as v1.0 and archived.",
    "3. A new model (v2.0) is trained, tuned, and evaluated against the",
    "   new test set once.",
    "4. Both results are reported side-by-side.",
    "",
    "**Do not silently retrain the deployed model.** Do not retrain on data",
    "that was used to evaluate it. Any such change invalidates the evaluation.",
    "",
    "---",
    "",
    "## 7. What is NOT part of this deployment",
    "",
    "- **Live data ingestion.** Not implemented. The pipeline assumes a",
    "  fixed historical dataset plus upcoming fixtures.",
    "- **UI / Streamlit page.** Not implemented. The pipeline lists this as",
    "  optional and it is a straightforward extension of the predict + log",
    "  + score workflow described above.",
    "- **Automatic retraining.** Not implemented, per policy.",
    "",
    "---",
    "",
    "## 8. Deployment risks and mitigations",
    "",
    "| Risk | Mitigation |",
    "| --- | --- |",
    "| Source drift (new provider, schema change) | Normalize to training schema before prediction. Re-validate features. |",
    "| Team renames / abbreviations | Maintain a canonical team-name mapping. |",
    "| Promoted teams with no history | Use the promoted-baseline fill (Step 07B); verify before first prediction. |",
    "| Concept drift (game changes over years) | Track live log loss. If it degrades meaningfully, plan a v2.0 with a new test set. |",
    "| Overconfidence on draws | The champion never predicts draws as top class. Do not report draw predictions as reliable. |",
    "",
    "---",
    "",
    "## 9. Final status",
    "",
    "- **Champion:** Deltas_LR_tuned, frozen at Step 17.",
    "- **Test log loss (2025/26):** 1.0238 (95% CI [0.9771, 1.0687]).",
    "- **Test accuracy:** 0.4711.",
    "- **Model artifacts:** complete in `models/`.",
    "- **Documentation:** complete in `docs/`.",
    "- **Predicting capability:** any future EPL match, provided history",
    "  and schema are consistent.",
    "",
    "---",
    "",
    "*End of Step 22. The project is complete.*",
]

deployment_doc_path = os.path.join(DOCS_DIR, 'step22_deployment.md')
with open(deployment_doc_path, 'w') as f:
    f.write("\n".join(deployment_doc_lines))
print("Wrote:", deployment_doc_path)
print("Size:", os.path.getsize(deployment_doc_path), "bytes")

print("\n" + "=" * 60)
print("STEP 22 COMPLETE — PROJECT COMPLETE")
print("=" * 60)

In [ ]:
# Run this ONCE in Colab before deploying
import pandas as pd, os

PROC_DIR = '/content/drive/MyDrive/epl_match_outcome/data/processed'
OUT_DIR  = '/content/drive/MyDrive/epl_match_outcome/deployment_data'
os.makedirs(OUT_DIR, exist_ok=True)

frames = []
for split in ['train.csv', 'val.csv', 'test.csv']:
    df = pd.read_csv(os.path.join(PROC_DIR, split), parse_dates=['Date'])
    frames.append(df)

combined = pd.concat(frames, ignore_index=True).sort_values('Date').reset_index(drop=True)
combined = combined[['Date','HomeTeam','AwayTeam','FTHG','FTAG','FTR',
                     'HTHG','HTAG','HTR','Referee',
                     'HS','AS','HST','AST','HF','AF','HC','AC',
                     'HY','AY','HR','AR','season','season_label']]

out_path = os.path.join(OUT_DIR, 'matches_history.csv')
combined.to_csv(out_path, index=False)
print(f"Saved {out_path} — {len(combined)} rows, {os.path.getsize(out_path)} bytes")